# Clinical Trial Data Assembly & Provenance

> **NOTE - v4 dataset.** The model-performance cells in this notebook (section 38) run on the
> **v4 training dataset** (`training_dataset_v4_with_drumap.csv`, 211 features) and print **v4-era AUCs**.
> They do **not** reproduce the current v8 manuscript numbers. Cross-references have been renumbered to the
> v8 scheme: model performance -> **Supplementary Table S1** (Overall 0.770, Safety 0.784, Efficacy 0.765),
> per-module ablation -> **Supplementary Fig. S1**, repositioning -> **Fig. 4**. The v8 headline numbers are
> produced by `scripts/retrain_calibrated.py` (see `00_reproduce_full_pipeline.ipynb`), not by this notebook.

---

This notebook documents every step from raw external data to the final modeling dataset.
Each source is loaded independently, saved as a pristine file in `data/sources/`, and merged
with full audit trail. **No opaque scripts.** Every transformation has visible intermediate outputs.

---

## Pipeline Walkthrough

The notebook is organized into numbered sections. Here is what each does and why:

### Data Loading (Sections 1–6)
| Section | What | Why |
|---------|------|-----|
| **1. AACT** | Load 428K drug-intervention records from ClinicalTrials.gov | Raw trial data — the foundation |
| **2. ClinTox** | Load clinical toxicity labels (CT_TOX, FDA_APPROVED) | Ground-truth safety failures (NOT used for outcome labels — kept for reference only) |
| **3. ChEMBL** | Load drug development phases, withdrawn flags, black box warnings | Pre-trial regulatory features (has_black_box) |
| **4. DILIrank** | Load drug-induced liver injury rankings | Reference only — was previously injected as fake trials (FIXED) |
| **5. Withdrawn** | Load post-market drug withdrawals | Reference only — was previously injected as fake trials (FIXED) |
| **6. repoDB/SIDER** | Load drug approval/indication databases | Attempted integration — identifier mismatch, not used |

### Outcome Classification (Sections 7–8)
| Section | What | Why |
|---------|------|-----|
| **7. Keyword Classification** | Classify `why_stopped` text → FAIL_SAFETY / FAIL_EFFICACY / ENROLLMENT / BUSINESS | Auditable keyword rules with known false-positive patterns (DSMB) |
| **7b. Manual Review** | Human review of all 672 terminated trials, reclassify 34 | Catches keyword errors (DSMB≠safety, business decisions, enrollment issues) |
| **8. API Enrichment** | Enrich trial records with additional metadata | Supplementary context |

### SMILES Resolution (Section 9) ← REBUILT
| Section | What | Why |
|---------|------|-----|
| **9a. ChEMBL Index** | Build drug name → SMILES lookup from ChEMBL 36 SQLite (105K names) | Replaces broken hardcoded dictionary that had ~99 wrong SMILES |
| **9b. Multi-pass Match** | Exact → suffix strip → parenthetical → dosage → brand mapping | Handles "metformin hydrochloride", "Copanlisib (BAY80-6946)", etc. |
| **9c. Second Pass** | Extra mappings, word extraction, SQL prefix, manual metal complexes | Catches truncations, experimental codes, platinum drugs |
| **9d. Apply & Report** | Write corrected SMILES, list every unresolved drug | Zero silent data loss — every drug accounted for |

### Feature Assembly (Sections 10–15)
| Section | What | Why |
|---------|------|-----|
| **10. Disease Extraction** | Parse disease/indication from trial records | Needed for disease context features and disease encoding |
| **11. Biological Features** | Load Binding binding, network enrichment, tissue interaction features | Core STAR pipeline outputs |
| **12. Corrected Outcomes** | Apply Phase III+ pass rule, exclude ENROLLMENT/BUSINESS/non-drugs | Final outcome labels: PASS, FAIL_SAFETY, FAIL_EFFICACY, FAIL_BOTH |
| **13. Pipeline Gap** | Identify drugs needing bioinformatics pipeline runs | 550 drugs await processing |
| **13b. Feature Validity Audit** | Compare old vs new InChIKeys; exclude drugs with wrong-structure features | ~81 drugs had features computed on wrong molecule — excluded until reprocessed |
| **14. Safety Features** | Toxicity binding (cardiac/hepatic/renal/DNA damage targets) | Safety-specific molecular features |
| **15. Disease Context** | Oncology/infectious/CNS/cardiac/autoimmune flags, patient vulnerability | Disease-level features (biggest safety predictor) |

### Analysis & Validation (Sections 15–38)
| Section | What | Why |
|---------|------|-----|
| **15** | Disease context features analysis | Oncology/infectious/CNS disease flags |
| **16** | Label quality fixes | OOF error analysis |
| **18** | Drug context & leakage audit | Caught 4 leaking features (drug_is_code, log_n_trials, etc.) |
| **20** | Organ-specific toxicity classification | Hepato/hemato/neuro/cardio breakdown |
| **21–23** | Feature exploration, ADMET reversal, disease-mechanism alignment | Key biological findings |
| **27** | Top-20 feature leakage audit | Verified all features are clean |
| **28** | Disease target encoding | Bayesian-smoothed disease difficulty |
| **38** | Model validation (to be re-run after pipeline reprocessing) | **Canonical results for manuscript** — all numbers trace here |

---

### Key Corrections Made
1. **391 fake trials removed** — DILIrank/Withdrawn entries injected as EXT_* NCT IDs (Sections 4–5, 12)
2. **~99 wrong SMILES fixed** — Hardcoded dictionary replaced with ChEMBL bulk download (Section 9)
3. **~81 drugs excluded** — Pipeline features computed on wrong molecule, awaiting reprocessing (Section 13b)
4. **34 trials reclassified** — DSMB false matches, business decisions (Section 7b)
5. **4 leaking features excluded** — drug_is_code, log_drug_n_trials, log_drug_n_diseases, phase_numeric (Section 27)
6. **Combination therapy SMILES cleared** — Multi-drug interventions can't have single SMILES (Section 9d)

In [1]:
import pandas as pd
import numpy as np
import json
import os
import requests
import time
import gzip
from pathlib import Path
from io import StringIO
from rdkit import Chem

PROJECT_ROOT = Path('..').resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
SOURCES_DIR = PROJECT_ROOT / 'data' / 'sources'
CACHE_DIR = PROJECT_ROOT / 'data' / 'cache'
SOURCES_DIR.mkdir(exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Sources dir:  {SOURCES_DIR}')

Project root: <repo>
Sources dir:  <repo>/data/sources


In [2]:
def canonical_smiles(smi):
    """Canonicalize a SMILES string using RDKit."""
    if pd.isna(smi) or not smi:
        return None
    try:
        mol = Chem.MolFromSmiles(str(smi))
        if mol:
            return Chem.MolToSmiles(mol)
    except:
        pass
    return None

---
## 1. AACT — ClinicalTrials.gov

**Source**: AACT database (Aggregate Analysis of ClinicalTrials.gov)  
**File**: `data/raw/aact_drug_trials.csv`  
**Key fields**: `nct_id`, `phase`, `overall_status`, `why_stopped`, `intervention_name`

In [3]:
aact_raw = pd.read_csv(RAW_DIR / 'aact_drug_trials.csv')

print(f'AACT: {len(aact_raw):,} rows, {aact_raw["nct_id"].nunique():,} unique trials')
print(f'Columns: {list(aact_raw.columns)}')
print(f'\nStatus distribution:')
print(aact_raw['overall_status'].value_counts().head(10))
print(f'\nPhase distribution:')
print(aact_raw['phase'].value_counts().head(10))

aact_raw.to_csv(SOURCES_DIR / '01_aact_clinicaltrials_raw.csv', index=False)
print(f'\nSaved: 01_aact_clinicaltrials_raw.csv')

AACT: 428,377 rows, 225,003 unique trials
Columns: ['nct_id', 'intervention_name', 'intervention_type', 'phase', 'overall_status', 'why_stopped', 'brief_title', 'start_date', 'completion_date', 'enrollment']

Status distribution:
overall_status
COMPLETED                  245895
UNKNOWN                     47891
RECRUITING                  43585
TERMINATED                  37841
ACTIVE_NOT_RECRUITING       20072
WITHDRAWN                   14696
NOT_YET_RECRUITING          14645
ENROLLING_BY_INVITATION      1468
SUSPENDED                    1274
NO_LONGER_AVAILABLE           535
Name: count, dtype: int64

Phase distribution:
phase
PHASE2           113437
PHASE1            83591
PHASE3            76018
PHASE4            53862
PHASE1/PHASE2     28067
PHASE2/PHASE3     11511
EARLY_PHASE1       7323
Name: count, dtype: int64



Saved: 01_aact_clinicaltrials_raw.csv


In [4]:
# Terminated trials with why_stopped text
terminated = aact_raw[aact_raw['overall_status'].isin(
    ['TERMINATED', 'WITHDRAWN', 'SUSPENDED', 'Terminated', 'Withdrawn', 'Suspended'])].copy()
has_reason = terminated[terminated['why_stopped'].notna()]

print(f'Terminated/Withdrawn/Suspended: {len(terminated):,} rows, {terminated["nct_id"].nunique():,} trials')
print(f'With why_stopped text: {len(has_reason):,} ({100*len(has_reason)/max(len(terminated),1):.1f}%)')
print(f'\nSample why_stopped values:')
for reason in has_reason['why_stopped'].head(10):
    print(f'  - {reason[:120]}')

Terminated/Withdrawn/Suspended: 53,811 rows, 28,345 trials
With why_stopped text: 47,021 (87.4%)

Sample why_stopped values:
  - Replaced by another study.
  - Replaced by another study.
  - Replaced by another study.
  - Replaced by another study.
  - Study never opened; never enrolled participants
  - Original P.I. left the institution
  - Unable to recruit adequate number of subjects
  - Study was never funded.
  - Pairing D-Cycloserine with Clozapine was found to worsen negative side effects in patients with Schizophrenia, so the st
  - Pairing D-Cycloserine with Clozapine was found to worsen negative side effects in patients with Schizophrenia, so the st


---
## 2. ClinTox — Clinical Toxicity Database

**Source**: MoleculeNet / DeepChem  
**URL**: `https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/clintox.csv.gz`  
**Key fields**: `smiles`, `FDA_APPROVED` (0/1), `CT_TOX` (0/1 — ground truth for clinical toxicity failure)

In [5]:
CLINTOX_URL = 'https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/clintox.csv.gz'
clintox_path = SOURCES_DIR / '02_clintox_raw.csv'

if clintox_path.exists():
    clintox_raw = pd.read_csv(clintox_path)
else:
    resp = requests.get(CLINTOX_URL)
    resp.raise_for_status()
    csv_data = gzip.decompress(resp.content).decode('utf-8')
    clintox_raw = pd.read_csv(StringIO(csv_data))
    clintox_raw.to_csv(clintox_path, index=False)

print(f'ClinTox: {len(clintox_raw)} compounds')
print(f'FDA_APPROVED: {clintox_raw["FDA_APPROVED"].value_counts().to_dict()}')
print(f'CT_TOX (toxicity failures): {(clintox_raw["CT_TOX"]==1).sum()}')

ClinTox: 1484 compounds
FDA_APPROVED: {1: 1390, 0: 94}
CT_TOX (toxicity failures): 112


---
## 3. ChEMBL — Drug Development Phases

**Source**: EMBL-EBI ChEMBL API  
**Key fields**: `max_phase` (1-4), `withdrawn_flag`, `black_box_warning`, `first_approval`  
**Cached in**: `data/cache/chembl_cache.json`

In [6]:
chembl_path = PROJECT_ROOT / 'data' / 'processed' / 'chembl_compound_results.csv'
if chembl_path.exists():
    chembl_results = pd.read_csv(chembl_path)
    print(f'ChEMBL: {len(chembl_results)} compounds')
    print(f'\nmax_phase distribution:')
    print(chembl_results['max_phase'].value_counts().sort_index())
    print(f'\nwithdrawn: {(chembl_results.get("withdrawn_flag", pd.Series())==1).sum()}')
    print(f'black_box_warning: {(chembl_results.get("black_box_warning", pd.Series())==1).sum()}')
    chembl_results.to_csv(SOURCES_DIR / '03_chembl_compound_results.csv', index=False)
else:
    print('ChEMBL results not found. Run scripts/query_chembl_phases.py.')
    chembl_results = pd.DataFrame()

ChEMBL: 4863 compounds

max_phase distribution:
max_phase
-1.0      43
 0.5       3
 1.0      54
 2.0     490
 3.0     487
 4.0    1681
Name: count, dtype: int64

withdrawn: 82


---
## 4. DILIrank 2.0 — Drug-Induced Liver Injury Rankings

**Source**: FDA / Charité Bioinformatics  
**URL**: `https://bioinformatics.charite.de/dilirank/`  
**Note**: These are FDA-approved drugs ranked by liver injury severity — a separate safety signal source, not trial outcomes.

In [7]:
dilirank_path = RAW_DIR / 'dilirank_2.0.xlsx'
if dilirank_path.exists():
    # Skip title row, use row 1 as header
    dilirank_raw = pd.read_excel(dilirank_path, skiprows=1)
    dilirank_raw.columns = ['LTKBID', 'CompoundName', 'SeverityClass', 'LabelSection', 'DILIConcern', 'Status']
    print(f'DILIrank: {len(dilirank_raw)} drugs')
    print(f'\nDILI Concern distribution:')
    print(dilirank_raw['DILIConcern'].value_counts())
    print(f'\nLabel Section:')
    print(dilirank_raw['LabelSection'].value_counts())
    dilirank_raw.to_csv(SOURCES_DIR / '04_dilirank_raw.csv', index=False)
else:
    print('DILIrank not found.')
    dilirank_raw = pd.DataFrame()

DILIrank: 1336 drugs

DILI Concern distribution:
DILIConcern
vNo-DILI-concern          413
Ambiguous-DILI-concern    354
vLess-DILI-concern        351
vMost-DILI-concern        215
vMOST-DILI-concern          2
vNo-DILI-Concern            1
Name: count, dtype: int64

Label Section:
LabelSection
No match                  467
Adverse reactions         435
Warnings & precautions    333
Withdrawn                  54
Box warning                39
Discontinued                8
Name: count, dtype: int64


---
## 5. Withdrawn 2.0 — Post-Market Drug Withdrawals

**Source**: Charité Bioinformatics  
**Citation**: Gallo K et al., Nucleic Acids Res 2024;52(D1):D1503-D1507  
**Note**: Drugs withdrawn from market after approval — a separate safety signal source.

In [8]:
withdrawn_path = PROJECT_ROOT / 'data' / 'processed' / 'withdrawn_drugs.csv'
if withdrawn_path.exists():
    withdrawn = pd.read_csv(withdrawn_path)
    print(f'Withdrawn database: {len(withdrawn)} drugs')
    if 'toxtype' in withdrawn.columns:
        print(f'\nWithdrawal reasons:')
        print(withdrawn['toxtype'].value_counts().head(15))
    withdrawn.to_csv(SOURCES_DIR / '05_withdrawn_drugs.csv', index=False)
else:
    print('Withdrawn database not found.')
    withdrawn = pd.DataFrame()

Withdrawn database: 264 drugs


---
## 6. repoDB & SIDER

**repoDB**: Drug repurposing database (10,563 entries, DrugBank IDs — 0% matched to our compounds)  
**SIDER**: Drug adverse events coded in MedDRA terms

In [9]:
# repoDB
repodb_path = RAW_DIR / 'repoDB.csv'
if repodb_path.exists():
    repodb = pd.read_csv(repodb_path)
    print(f'repoDB: {len(repodb)} entries')
    print(f'Status distribution:')
    print(repodb['status'].value_counts())
    repodb.to_csv(SOURCES_DIR / '06_repodb_raw.csv', index=False)
else:
    repodb = pd.DataFrame()

# SIDER
print()
sider_dir = RAW_DIR / 'sider'
if sider_dir.exists():
    print('SIDER files:')
    for f in sorted(sider_dir.iterdir()):
        size = f.stat().st_size / 1024
        unit, val = ('MB', size/1024) if size >= 1024 else ('KB', size)
        print(f'  {f.name}: {val:.1f} {unit}')
else:
    print('SIDER directory not found.')

repoDB: 10562 entries
Status distribution:
status
Approved      6677
Terminated    2754
Withdrawn      648
Suspended      483
Name: count, dtype: int64



SIDER files:
  drug_names.tsv: 33.9 KB
  meddra_all_se.tsv: 18.6 MB
  meddra_all_se.tsv.gz: 2.3 MB
  meddra_freq.tsv: 22.4 MB
  meddra_freq.tsv.gz: 2.0 MB


---
## 7. Trial Outcome Classification: Keyword Rules

**How `why_stopped` free text is classified into FAIL_SAFETY / FAIL_EFFICACY.**  
Priority ordering: **safety > efficacy > enrollment > business > admin**.  
First keyword match wins.

In [10]:
import re

# --- Keyword lists ---
# These are checked in priority order: safety > efficacy > enrollment > business > admin.
# IMPORTANT: simple substring matching has known failure modes. The classify function
# below handles key edge cases with contextual rules BEFORE falling through to keywords.

SAFETY_KEYWORDS = [
    "toxicity", "toxic", "dose-limiting toxicit", "dlt",
    "adverse event", "adverse reaction", "side effect",
    "serious adverse", "sae",
    "safety concern", "safety signal", "safety issue",
    "death", "died", "fatality", "mortality",
    "hepatotoxicity", "cardiotoxicity", "neurotoxicity", "nephrotoxicity",
    "liver injury", "liver toxicity",
    "dangerous",
    "clinical hold", "fda hold",
]

EFFICACY_KEYWORDS = [
    "lack of efficacy", "insufficient efficacy", "no efficacy",
    "efficacy endpoint", "efficacy criteria",
    "futility analysis", "futility interim", "due to futility",
    "failed to demonstrate", "failed to meet", "failed to show",
    "did not meet", "did not demonstrate", "did not show",
    "endpoint not met", "primary endpoint",
    "no benefit", "lack of benefit", "insufficient benefit",
    "no activity", "lack of activity",
    "lack of clinical response", "no clinical response",
    "therapeutic effect",
]

ENROLLMENT_KEYWORDS = [
    "slow accrual", "poor accrual", "low accrual",
    "slow enrollment", "poor enrollment", "low enrollment",
    "insufficient enrollment", "difficulty in enrolling",
    "unable to recruit", "difficulty in recruiting",
    "recruitment challenges", "recruitment difficulties",
    "futility in recruitment", "futility in enrollment",
    "enrollment futility", "accrual futility",
]

BUSINESS_KEYWORDS = [
    "business decision", "business reason",
    "sponsor decision", "strategic decision", "commercial decision",
    "funding", "financial",
    "portfolio", "acquisition", "merger",
    "company decision",
]

ADMINISTRATIVE_KEYWORDS = [
    "administrative", "pi left", "investigator left",
    "site closed", "facility closed",
    "logistics", "feasibility",
]

# --- Negation / ambiguity phrases that OVERRIDE simple keyword matching ---
# These appear in text that mentions a keyword but means the opposite or is ambiguous.
SAFETY_NEGATIONS = [
    "no safety concern", "no safety issue", "no safety signal",
    "not due to safety", "no relevant safety",
    "no new safety", "without any safety concern",
    "without safety concern",
]

def classify_why_stopped(text):
    """Classify why_stopped text with contextual rules.
    
    Priority: safety > efficacy > enrollment > business > admin.
    But contextual rules override simple keyword matching:
    - "benefit-risk profile no longer supports" → ambiguous (both)
    - "DSMB ... futility" with no actual safety event → efficacy
    - "no safety concerns" negates safety keyword
    - "futility in recruitment" → enrollment, not efficacy
    """
    if not text or pd.isna(text):
        return 'unknown'
    text_lower = str(text).lower()
    
    # --- Contextual rules (checked FIRST, before keyword fallback) ---
    
    # Rule 1: "benefit-risk profile no longer supports" without explicit safety event
    # = ambiguous, classify as safety+efficacy (FAIL_BOTH)
    if re.search(r'benefit.{0,5}risk.{0,30}no longer supports', text_lower):
        # But if there's an explicit safety event mentioned too, keep as safety
        if any(kw in text_lower for kw in ['toxicity', 'adverse event', 'death', 'hepatotox',
                                             'cardiotox', 'neutropenia', 'safety concern']):
            return 'safety'
        return 'both'
    
    # Rule 2: DSMB/monitoring board + futility = efficacy, not safety
    if re.search(r'(dsmb|data safety monitoring|monitoring board|dsmc)', text_lower):
        if re.search(r'(futility|futile|did not meet|lack of efficacy|no benefit)', text_lower):
            # Check if there's ALSO a real safety event
            if any(kw in text_lower for kw in ['toxicity', 'adverse event', 'death', 'sae']):
                return 'both'
            return 'efficacy'
    
    # Rule 3: Negated safety — "no safety concerns" means it's NOT a safety failure
    if any(neg in text_lower for neg in SAFETY_NEGATIONS):
        # Safety is negated — skip safety keywords, check efficacy etc.
        if any(kw in text_lower for kw in EFFICACY_KEYWORDS): return 'efficacy'
        if any(kw in text_lower for kw in ENROLLMENT_KEYWORDS): return 'enrollment'
        if any(kw in text_lower for kw in BUSINESS_KEYWORDS): return 'business'
        return 'other'
    
    # Rule 4: Enrollment keywords get priority when combined with futility
    # "futility in recruitment" = enrollment problem, not efficacy
    if any(kw in text_lower for kw in ENROLLMENT_KEYWORDS):
        # But if there's ALSO a clear efficacy signal, classify as efficacy
        if any(kw in text_lower for kw in ['lack of efficacy', 'insufficient efficacy',
                                             'endpoint not met', 'failed to meet',
                                             'failed to demonstrate']):
            return 'efficacy'
        return 'enrollment'
    
    # --- Standard keyword matching (first match wins) ---
    if any(kw in text_lower for kw in SAFETY_KEYWORDS): return 'safety'
    if any(kw in text_lower for kw in EFFICACY_KEYWORDS): return 'efficacy'
    # Enrollment already handled above in Rule 4
    if any(kw in text_lower for kw in BUSINESS_KEYWORDS): return 'business'
    if any(kw in text_lower for kw in ADMINISTRATIVE_KEYWORDS): return 'administrative'
    return 'other'

print(f'Safety keywords:     {len(SAFETY_KEYWORDS)}')
print(f'Efficacy keywords:   {len(EFFICACY_KEYWORDS)}')
print(f'Enrollment keywords: {len(ENROLLMENT_KEYWORDS)}')
print(f'Business keywords:   {len(BUSINESS_KEYWORDS)}')
print(f'Admin keywords:      {len(ADMINISTRATIVE_KEYWORDS)}')
print(f'Safety negations:    {len(SAFETY_NEGATIONS)}')
print()
print('Contextual rules:')
print('  1. "benefit-risk no longer supports" → both (unless explicit safety event)')
print('  2. DSMB + futility → efficacy (unless also mentions toxicity/AE)')
print('  3. Negated safety ("no safety concerns") → skip safety keywords')
print('  4. Enrollment keywords get priority over efficacy "futility"')
print()
print('Removed from SAFETY: "risk" (too broad), "harm" (rare), "regulatory" (ambiguous),')
print('  "fda" (too broad), "hold" (needs "clinical hold"), "suspend" (too broad),')
print('  bare "safety" (catches "no safety concerns"), bare "adverse" (too broad)')
print('Removed from ENROLLMENT: "patient", "subject", "participation" (way too broad)')
print('Tightened EFFICACY: "futility" alone removed (catches enrollment futility),')
print('  replaced with "futility analysis", "due to futility" etc.')
print('Tightened BUSINESS: "sponsor" alone removed (catches "sponsor decision to stop')
print('  due to lack of efficacy"), replaced with "sponsor decision" as phrase')

Safety keywords:     25
Efficacy keywords:   24
Enrollment keywords: 16
Business keywords:   11
Admin keywords:      7
Safety negations:    8

Contextual rules:
  1. "benefit-risk no longer supports" → both (unless explicit safety event)
  2. DSMB + futility → efficacy (unless also mentions toxicity/AE)
  3. Negated safety ("no safety concerns") → skip safety keywords
  4. Enrollment keywords get priority over efficacy "futility"

Removed from SAFETY: "risk" (too broad), "harm" (rare), "regulatory" (ambiguous),
  "fda" (too broad), "hold" (needs "clinical hold"), "suspend" (too broad),
  bare "safety" (catches "no safety concerns"), bare "adverse" (too broad)
Removed from ENROLLMENT: "patient", "subject", "participation" (way too broad)
Tightened EFFICACY: "futility" alone removed (catches enrollment futility),
  replaced with "futility analysis", "due to futility" etc.
Tightened BUSINESS: "sponsor" alone removed (catches "sponsor decision to stop
  due to lack of efficacy"), replaced 

In [11]:
# Apply classification to terminated trials
terminated['failure_category'] = terminated['why_stopped'].apply(classify_why_stopped)

print(f'Classification of {len(terminated):,} terminated/withdrawn/suspended trials:')
print(terminated['failure_category'].value_counts())

# Show examples for each category
for cat in ['safety', 'efficacy', 'enrollment', 'business', 'other', 'unknown']:
    examples = terminated[terminated['failure_category'] == cat]['why_stopped'].dropna().head(3)
    if len(examples) > 0:
        print(f'\n--- {cat.upper()} ---')
        for ex in examples:
            print(f'  "{ex[:120]}"')

terminated.to_csv(SOURCES_DIR / '07_aact_terminated_classified.csv', index=False)
print(f'\nSaved: 07_aact_terminated_classified.csv')

Classification of 53,811 terminated/withdrawn/suspended trials:
failure_category
other             29405
business           7068
unknown            6790
enrollment         5278
safety             1950
efficacy           1924
administrative     1385
both                 11
Name: count, dtype: int64

--- SAFETY ---
  "Pairing D-Cycloserine with Clozapine was found to worsen negative side effects in patients with Schizophrenia, so the st"
  "Pairing D-Cycloserine with Clozapine was found to worsen negative side effects in patients with Schizophrenia, so the st"
  "Pairing D-Cycloserine with Clozapine was found to worsen negative side effects in patients with Schizophrenia, so the st"

--- EFFICACY ---
  "Study enrollment did not meet expected goals"
  "Study enrollment did not meet expected goals"
  "Study enrollment did not meet expected goals"

--- ENROLLMENT ---
  "Unable to recruit adequate number of subjects"
  "Due to poor accrual and lack of peptide vaccine"
  "Due to poor accrual 


Saved: 07_aact_terminated_classified.csv


In [12]:
# Known classification issues
print('=== KNOWN CLASSIFICATION ISSUES ===')

# Issue 1: "Data Safety Monitoring Board" contains "safety" → false safety match
safety_classified = terminated[terminated['failure_category'] == 'safety']
false_dsmb = safety_classified[
    safety_classified['why_stopped'].str.contains('safety monitoring|dsmb|data safety|dsmc',
                                                   case=False, na=False) &
    safety_classified['why_stopped'].str.contains('futility|futile|did not meet|lack of efficacy|no benefit',
                                                   case=False, na=False)
]
print(f'\n1. DSMB + futility (classified safety, likely efficacy): {len(false_dsmb)}')
if len(false_dsmb) > 0:
    for t in false_dsmb['why_stopped'].head(5):
        print(f'   "{t[:150]}"')

# Issue 2: "safety and efficacy" in why_stopped
false_se = safety_classified[
    safety_classified['why_stopped'].str.contains('safety and efficacy|safe and effective',
                                                   case=False, na=False)
]
print(f'\n2. "safety and efficacy" in text (may not be safety failure): {len(false_se)}')

# Issue 3: trials with BOTH safety and efficacy keywords
both = terminated[
    terminated['why_stopped'].apply(lambda x: isinstance(x, str) and
        any(kw in x.lower() for kw in SAFETY_KEYWORDS) and
        any(kw in x.lower() for kw in EFFICACY_KEYWORDS))
]
print(f'\n3. Trials with BOTH safety AND efficacy keywords: {len(both)}')
print(f'   All classified as: {both["failure_category"].value_counts().to_dict()}')

=== KNOWN CLASSIFICATION ISSUES ===

1. DSMB + futility (classified safety, likely efficacy): 0

2. "safety and efficacy" in text (may not be safety failure): 6

3. Trials with BOTH safety AND efficacy keywords: 196
   All classified as: {'safety': 99, 'efficacy': 97}


### 7b. Manual Review of All Classifications

All 672 terminated trials with `why_stopped` text were manually reviewed in two batches:

**Batch 1** (63 trials with both safety AND efficacy keywords — highest misclassification risk):  
File: `data/sources/REVIEW_ambiguous_classification.csv`

**Batch 2** (609 remaining trials — safety-only or efficacy-only keywords):  
File: `data/sources/REVIEW_remaining_classifications.csv`

Every trial was read by the authors. Reclassification categories:
- **FAIL_SAFETY** — genuine safety/toxicity failure
- **FAIL_EFFICACY** — genuine efficacy failure  
- **FAIL_BOTH** — both safety and efficacy issues
- **BUSINESS** — sponsor/strategic decision unrelated to drug performance
- **ENROLLMENT** — recruitment/feasibility failure

In [13]:
# Load both review batches
batch1 = pd.read_csv(SOURCES_DIR / 'REVIEW_ambiguous_classification.csv')
batch2 = pd.read_csv(SOURCES_DIR / 'REVIEW_remaining_classifications.csv')

print(f'=== MANUAL REVIEW RESULTS ===')
print(f'Batch 1 (ambiguous keywords): {len(batch1)} trials')
print(batch1['Manual_Classification'].value_counts().to_string())

print(f'\nBatch 2 (remaining): {len(batch2)} trials')
print(batch2['Manual_Classification'].value_counts().to_string())

# Combine all manual corrections
all_manual = pd.concat([
    batch1[['NCT_ID', 'Final_Outcome', 'Manual_Classification']],
    batch2[['NCT_ID', 'Final_Outcome', 'Manual_Classification']]
])

# Show all reclassifications
changed = all_manual[all_manual['Final_Outcome'] != all_manual['Manual_Classification']]
print(f'\nTotal reclassified: {len(changed)} trials')
print(f'\nReclassification summary:')
for _, row in changed.iterrows():
    print(f'  {row["NCT_ID"]}:  {row["Final_Outcome"]:>15} → {row["Manual_Classification"]}')

=== MANUAL REVIEW RESULTS ===
Batch 1 (ambiguous keywords): 63 trials
Manual_Classification
FAIL_EFFICACY    52
FAIL_BOTH        10
ENROLLMENT        1

Batch 2 (remaining): 609 trials
Manual_Classification
FAIL_EFFICACY    451
FAIL_SAFETY      144
BUSINESS           7
ENROLLMENT         4
FAIL_BOTH          3

Total reclassified: 34 trials

Reclassification summary:
  NCT05036317:      FAIL_SAFETY → FAIL_EFFICACY
  NCT04552899:      FAIL_SAFETY → FAIL_EFFICACY
  NCT04594707:      FAIL_SAFETY → FAIL_EFFICACY
  NCT03765541:      FAIL_SAFETY → ENROLLMENT
  NCT04214418:      FAIL_SAFETY → FAIL_BOTH
  NCT04478266:      FAIL_SAFETY → FAIL_EFFICACY
  NCT03192215:      FAIL_SAFETY → FAIL_EFFICACY
  NCT03504917:      FAIL_SAFETY → FAIL_EFFICACY
  NCT03436420:      FAIL_SAFETY → FAIL_BOTH
  NCT02722369:      FAIL_SAFETY → FAIL_BOTH
  NCT02618577:      FAIL_SAFETY → FAIL_BOTH
  NCT02956486:      FAIL_SAFETY → FAIL_BOTH
  NCT01414166:      FAIL_SAFETY → FAIL_BOTH
  NCT01274559:      FAIL_SAFETY →

---
## 8. API-Based Enrichment

**Script**: `scripts/enrich_failure_reasons_v2.py`  
For compounds without `why_stopped` in AACT, two APIs were queried:

1. **ClinicalTrials.gov API v2** — terminated trials by drug name (weight: 1.5x)  
2. **PubMed (NCBI eUtils)** — failure-related publications (weight: 0.5x)  

Extended keyword list with negation handling ("not due to safety" → reduces safety score).  
Confidence threshold: ≥ 0.3.

In [14]:
enriched_path = PROJECT_ROOT / 'data' / 'intermediate' / 'failure_reasons_enriched_v2.csv'
if enriched_path.exists():
    enriched = pd.read_csv(enriched_path)
    print(f'API-enriched failure reasons: {len(enriched)} compounds')
    if 'best_category' in enriched.columns:
        print(f'\nCategory distribution:')
        print(enriched['best_category'].value_counts())
    if 'confidence' in enriched.columns:
        print(f'\nConfidence: mean={enriched["confidence"].mean():.2f}, median={enriched["confidence"].median():.2f}')
    enriched.to_csv(SOURCES_DIR / '08_api_enriched_failure_reasons.csv', index=False)
else:
    print('No API-enriched results found.')

API-enriched failure reasons: 1000 compounds

Category distribution:
best_category
unknown     848
safety      102
efficacy     50
Name: count, dtype: int64

Confidence: mean=0.11, median=0.00


---
## 9. SMILES Resolution from ChEMBL Bulk Download

**Previous approach (BROKEN):** Hardcoded dictionary (`add_known_smiles.py`) + PubChem/ChEMBL
API free-text search. Produced ~99 wrong SMILES affecting ~847 trials (28.5% of training data).
Key failures: lenalidomide=pomalidomide (same wrong structure), simvastatin=pravastatin,
"ASA" (aspirin) matched to dasatinib via substring, ibrutinib MW 646 vs correct 441.

**New approach:** ChEMBL 36 SQLite bulk download (`data/cache/chembl_36/.../chembl_36.db`).
- Build index of all drug names (pref_name + synonyms) → canonical SMILES
- Match Drug_Clean exactly, then by suffix stripping, case normalization, parenthetical extraction
- Combination therapies (multi-drug interventions) get SMILES cleared — cannot be represented as single molecule
- Every drug is either resolved to a ChEMBL-validated SMILES or explicitly flagged

In [15]:
# 9a. Build ChEMBL name → SMILES index from bulk SQLite download
#
# ChEMBL 36 SQLite: ~28GB, contains all molecules with preferred names,
# synonyms (trade names, INN, USAN, research codes), and canonical SMILES.
# This is the ground truth for drug structures — no API, no guessing.

import sqlite3
import re

CHEMBL_DB = PROJECT_ROOT / 'data' / 'cache' / 'chembl_36' / 'chembl_36_sqlite' / 'chembl_36.db'
assert CHEMBL_DB.exists(), f'ChEMBL database not found at {CHEMBL_DB}'

conn = sqlite3.connect(str(CHEMBL_DB))
cur = conn.cursor()

# Index 1: preferred names (e.g., "LENALIDOMIDE" → SMILES)
cur.execute("""
    SELECT LOWER(md.pref_name), md.pref_name, cs.canonical_smiles, md.chembl_id
    FROM molecule_dictionary md
    JOIN compound_structures cs ON md.molregno = cs.molregno
    WHERE md.pref_name IS NOT NULL
""")
chembl_name_index = {}
for lower_name, pref, smi, cid in cur.fetchall():
    if lower_name not in chembl_name_index:
        chembl_name_index[lower_name] = (pref, smi, cid)

# Index 2: all synonyms (trade names, research codes, INN names)
cur.execute("""
    SELECT LOWER(ms.synonyms), md.pref_name, cs.canonical_smiles, md.chembl_id
    FROM molecule_synonyms ms
    JOIN molecule_dictionary md ON ms.molregno = md.molregno
    JOIN compound_structures cs ON ms.molregno = cs.molregno
""")
for lower_syn, pref, smi, cid in cur.fetchall():
    if lower_syn not in chembl_name_index:
        chembl_name_index[lower_syn] = (pref, smi, cid)

conn.close()

print(f'ChEMBL name index: {len(chembl_name_index):,} entries')
print(f'Database: {CHEMBL_DB}')

# Quick sanity check on drugs we know were wrong before
for test in ['lenalidomide', 'pomalidomide', 'simvastatin', 'pravastatin', 'ibrutinib', 'aspirin']:
    if test in chembl_name_index:
        pref, smi, cid = chembl_name_index[test]
        print(f'  {test} → {cid} ({pref}), SMILES={smi[:50]}{"..." if len(smi)>50 else ""}')

ChEMBL name index: 105,689 entries
Database: <repo>/data/cache/chembl_36/chembl_36_sqlite/chembl_36.db
  lenalidomide → CHEMBL848 (LENALIDOMIDE), SMILES=Nc1cccc2c1CN(C1CCC(=O)NC1=O)C2=O
  pomalidomide → CHEMBL43452 (POMALIDOMIDE), SMILES=Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O
  simvastatin → CHEMBL1064 (SIMVASTATIN), SMILES=CCC(C)(C)C(=O)O[C@H]1C[C@@H](C)C=C2C=C[C@H](C)[C@H...
  pravastatin → CHEMBL1144 (PRAVASTATIN), SMILES=CC[C@H](C)C(=O)O[C@H]1C[C@H](O)C=C2C=C[C@H](C)[C@H...
  ibrutinib → CHEMBL1873475 (IBRUTINIB), SMILES=C=CC(=O)N1CCC[C@@H](n2nc(-c3ccc(Oc4ccccc4)cc3)c3c(...
  aspirin → CHEMBL25 (ASPIRIN), SMILES=CC(=O)Oc1ccccc1C(=O)O


In [16]:
# 9b. Match Drug_Clean names to ChEMBL SMILES — multi-pass resolution
#
# Pass 1: Exact match (lowercase)
# Pass 2: Strip salt/ester suffixes ("hydrochloride", "sodium", etc.)
# Pass 3: Extract drug name from parentheticals, brand names, dosage forms
# Pass 4: Flag combination therapies (cannot have single SMILES)
# Pass 5: Report unresolved drugs for manual review

import re

# All unique Drug_Clean from corrected outcomes
all_drug_names = review_real['Drug'].dropna().unique() if 'review_real' in dir() else \
    pd.read_csv(SOURCES_DIR / '12_trials_corrected_outcomes.csv')['Drug_Clean'].dropna().unique()
print(f'Drug names to resolve: {len(all_drug_names)}')

# --- Salt/ester suffixes to strip ---
SALT_SUFFIXES = [
    ' hydrochloride', ' hcl', ' dihydrochloride', ' trihydrochloride',
    ' sodium', ' potassium', ' calcium', ' magnesium', ' lithium',
    ' sulfate', ' hemisulfate', ' phosphate', ' nitrate',
    ' acetate', ' propionate', ' valerate', ' enanthate', ' decanoate',
    ' maleate', ' mesylate', ' besylate', ' tosylate', ' esylate',
    ' fumarate', ' tartrate', ' citrate', ' succinate', ' oxalate',
    ' chloride', ' bromide', ' iodide',
    ' dihydrate', ' monohydrate', ' trihydrate', ' sesquihydrate',
    ' dimesylate', ' ditosylate', ' diacetate',
]

# --- Patterns that indicate combination therapies (no single SMILES possible) ---
COMBO_PATTERNS = [
    r'\band\b', r'\bplus\b', r'\bwith\b', r'\bfollowed by\b',
    r'\bcombination\b', r'\bregimen\b', r'\btherapy group\b',
    r'/', r'\+',  # "Drug A / Drug B" or "Drug A + Drug B"
]

# --- Brand name → generic mappings for common unresolvable names ---
BRAND_TO_GENERIC = {
    'erbitux': 'cetuximab',           # But cetuximab is a biologic — no SMILES
    'sutent': 'sunitinib',
    'xarelto': 'rivaroxaban',
    'jardiance': 'empagliflozin',
    'dacogen': 'decitabine',
    'exparel': 'bupivacaine',
    'synacthen': 'cosyntropin',
    'cytoxan': 'cyclophosphamide',
    'vidaza': 'azacitidine',
    'inqovi': 'decitabine',
    'buprenex': 'buprenorphine',
    'vivitrol': 'naltrexone',
    'nab-paclitaxel': 'paclitaxel',
    '5fu': 'fluorouracil',
    '5fluorouracil': 'fluorouracil',
    'asa': 'acetylsalicylic acid',
    'sel': 'oseltamivir',            # Verify — "SEL" is ambiguous
    'mor': 'morphine',
    'folfiri': 'irinotecan',         # Combo regimen, will flag as combo
    'gemox': 'gemcitabine',          # Combo regimen
}

def resolve_drug(drug_name, index):
    """Try to resolve a drug name to ChEMBL SMILES through multiple strategies."""
    key = drug_name.lower().strip()
    
    # Pass 1: Exact match
    if key in index:
        pref, smi, cid = index[key]
        return pref, smi, cid, 'exact'
    
    # Pass 2: Strip salt suffixes (try longest first to avoid partial strips)
    for suffix in sorted(SALT_SUFFIXES, key=len, reverse=True):
        stripped = key.replace(suffix, '').strip()
        if stripped != key and stripped in index:
            pref, smi, cid = index[stripped]
            return pref, smi, cid, f'suffix_strip ({suffix.strip()})'
    
    # Pass 3a: Extract from parenthetical — "Copanlisib (BAY80-6946)" → try both parts
    paren_match = re.match(r'^([^(]+?)\s*\(([^)]+)\)', key)
    if paren_match:
        base = paren_match.group(1).strip()
        inner = paren_match.group(2).strip()
        # Try base name first
        if base in index:
            pref, smi, cid = index[base]
            return pref, smi, cid, 'paren_base'
        # Try inner (might be a code like BAY80-6946)
        if inner in index:
            pref, smi, cid = index[inner]
            return pref, smi, cid, 'paren_inner'
        # Try base with suffix stripping
        for suffix in sorted(SALT_SUFFIXES, key=len, reverse=True):
            stripped = base.replace(suffix, '').strip()
            if stripped != base and stripped in index:
                pref, smi, cid = index[stripped]
                return pref, smi, cid, f'paren_base_suffix ({suffix.strip()})'
    
    # Pass 3b: Strip trailing dosage/formulation info
    # "Drug 100mg" → "drug", "Drug 0.01% Ophthalmic" → "drug"
    cleaned = re.sub(r'\s+\d+[\.\d]*\s*(mg|mcg|ml|%|miligram).*', '', key, flags=re.IGNORECASE).strip()
    if cleaned != key and cleaned in index:
        pref, smi, cid = index[cleaned]
        return pref, smi, cid, 'dosage_strip'
    
    # Pass 3c: Remove trailing qualifiers
    for qual in [' only', ' monotherapy', ' inhalation', ' loading dose',
                 ' film-coated', ' usp micronized', ' or equivalent',
                 ' ophthalmic', ' enemas', ' period']:
        stripped = key.replace(qual, '').strip()
        if stripped != key and stripped in index:
            pref, smi, cid = index[stripped]
            return pref, smi, cid, f'qualifier_strip ({qual.strip()})'
    
    # Pass 3d: Brand name → generic mapping
    # First try the whole name, then try extracting known brands
    for brand, generic in BRAND_TO_GENERIC.items():
        if brand == key or brand in key:
            if generic in index:
                pref, smi, cid = index[generic]
                return pref, smi, cid, f'brand_map ({brand}→{generic})'
    
    # Pass 3e: Try stripping "^2" or trailing special chars
    stripped = re.sub(r'[\^:®]+\d*$', '', key).strip()
    if stripped != key and stripped in index:
        pref, smi, cid = index[stripped]
        return pref, smi, cid, 'special_char_strip'
    
    # Pass 3f: Handle "Daunorubicin (nonliposomal)" → "daunorubicin"
    base_only = re.sub(r'\s*\(.*\)\s*', '', key).strip()
    if base_only != key and base_only in index:
        pref, smi, cid = index[base_only]
        return pref, smi, cid, 'paren_remove'
    
    return None, None, None, 'unresolved'

# --- Run resolution ---
resolved = {}    # Drug_Clean → (pref_name, smiles, chembl_id, match_type)
unresolved = []  # Drug_Clean names that could not be matched
combos = []      # Drug_Clean names identified as combination therapies

for drug in all_drug_names:
    key = drug.lower().strip()
    
    # Check for combination therapy BEFORE trying to resolve
    # (but only if it has multiple drugs — single drugs with "and" in name are OK)
    is_combo = False
    for pattern in COMBO_PATTERNS:
        if re.search(pattern, key):
            # Heuristic: if it has "and/+//" AND multiple known drug names, it's a combo
            # Simple check: does it contain common combo indicators?
            if any(x in key for x in [' and ', ' + ', '/', ' followed by',
                                        ' combination ', ' plus ']):
                is_combo = True
                break
    
    # Some "combo-looking" names are actually single drugs with brand qualifiers
    # e.g., "Rivaroxaban (Xarelto, BAY59-7939)" has commas but is one drug
    # So try resolving first, only flag as combo if resolution fails
    pref, smi, cid, match_type = resolve_drug(drug, chembl_name_index)
    
    if smi:
        resolved[drug] = (pref, smi, cid, match_type)
    elif is_combo:
        combos.append(drug)
    else:
        unresolved.append(drug)

print(f'\n=== SMILES RESOLUTION RESULTS ===')
print(f'Resolved:   {len(resolved)}/{len(all_drug_names)} ({100*len(resolved)/len(all_drug_names):.1f}%)')
print(f'Combos:     {len(combos)} (multi-drug interventions, no single SMILES)')
print(f'Unresolved: {len(unresolved)}')

# Match type breakdown
from collections import Counter
type_counts = Counter(v[3] for v in resolved.values())
print(f'\nMatch types:')
for mt, count in type_counts.most_common():
    print(f'  {mt}: {count}')

# Trial coverage
co_temp = pd.read_csv(SOURCES_DIR / '12_trials_corrected_outcomes.csv')
resolved_trials = co_temp[co_temp['Drug_Clean'].isin(resolved.keys())]
combo_trials = co_temp[co_temp['Drug_Clean'].isin(combos)]
unresolved_trials = co_temp[co_temp['Drug_Clean'].isin(unresolved)]
print(f'\nTrial coverage:')
print(f'  Resolved:   {len(resolved_trials)} trials')
print(f'  Combos:     {len(combo_trials)} trials (will lose SMILES)')
print(f'  Unresolved: {len(unresolved_trials)} trials')

Drug names to resolve: 1158

=== SMILES RESOLUTION RESULTS ===
Resolved:   878/1158 (75.8%)
Combos:     46 (multi-drug interventions, no single SMILES)
Unresolved: 234

Match types:
  exact: 862
  paren_base: 8
  paren_inner: 3
  special_char_strip: 2
  dosage_strip: 2
  qualifier_strip (only): 1

Trial coverage:
  Resolved:   4604 trials
  Combos:     46 trials (will lose SMILES)
  Unresolved: 282 trials


In [17]:
# 9c. Second-pass resolution for remaining drugs
#
# The index-based lookup (cell 9b) misses drugs where:
# - Name is a truncation ("Cyclophosphamid" → "cyclophosphamide")
# - Name has trailing junk ("Apixaban 2." → "apixaban")  
# - Dosage variant ("0.06% Resiquimod - A" → "resiquimod")
# - ChEMBL has no SMILES for metal complexes (Pt, Fe, Mn, Gd)
#
# This cell does a direct SQLite query for each unresolved drug,
# then applies manual SMILES for metal complexes from PubChem.

conn2 = sqlite3.connect(str(CHEMBL_DB))
cur2 = conn2.cursor()

# Additional brand/abbreviation mappings discovered during validation
EXTRA_MAPPINGS = {
    'cyclophosphamid': 'cyclophosphamide',
    'apixaban 2.': 'apixaban',
    'hydroxychloroquine sulfate once a day': 'hydroxychloroquine',
    'hydroxychloroquine sulfate twice a day': 'hydroxychloroquine',
    'calcium leucovorin': 'leucovorin',  # leucovorin calcium = folinic acid
    'calcium edta': 'edetate calcium disodium',
    'folfiri': 'irinotecan',       # FOLFIRI is a combo regimen (5-FU/leucovorin/irinotecan)
    'haic (gemox)': 'gemcitabine', # GEMOX = gemcitabine + oxaliplatin
    'ferroquine ssr97193': 'ferroquine',
    'venglustat gz402671': 'venglustat',
    'gsk2586184': 'solcitinib',
    'zpl389': 'adriforant',
    'bkm120': 'buparlisib',
    'lcq908': 'pradigastat',
    'gc4711': 'rucosopasem',       # No SMILES in ChEMBL (Mn complex)
    'gc4711 +sbrt': 'rucosopasem',
    'gd-dota': 'gadoterate',       # No SMILES in ChEMBL (Gd complex)
    'erbitux': 'cetuximab',         # Biologic — no SMILES
    'sel': 'oseltamivir',
    'taxotere': 'docetaxel',
    'auy922': 'luminespib',  # Novartis HSP90 inhibitor (NVP-AUY922)
}

# Manual SMILES for metal coordination complexes that ChEMBL cannot represent.
# These are canonical SMILES from PubChem (verified by CID).
# ChEMBL has these drugs but with structure_type=NONE because standard SMILES
# can't properly encode metal-ligand bonds — PubChem uses extended SMILES.
MANUAL_METAL_SMILES = {
    # Platinum complexes — PubChem CID verified
    'carboplatin':          ('CARBOPLATIN',          'O=C1O[Pt](N)(N)OC(=O)C11CCC1',                    'CHEMBL1351',  'PubChem CID 498142'),
    'cisplatin':            ('CISPLATIN',            '[NH3][Pt]([NH3])(Cl)Cl',                           'CHEMBL11359', 'PubChem CID 441203'),
    'oxaliplatin':          ('OXALIPLATIN',          'O=C1O[Pt]2(OC(=O)C1)N[C@@H]1CCCC[C@@H]1N2',      'CHEMBL414804','PubChem CID 9887054'),
    # Iron complex
    'sodium nitroprusside': ('SODIUM NITROPRUSSIDE', '[Na+].[Na+].N#C[Fe-2](C#N)(C#N)(C#N)(C#N)[N+]#O', 'CHEMBL136478','PubChem CID 11953895'),
}

second_pass_resolved = 0
still_unresolved = []

for drug in unresolved[:]:  # iterate over copy
    key = drug.lower().strip()
    found = False
    
    # Try extra mappings first (name → ChEMBL generic name)
    mapped = EXTRA_MAPPINGS.get(key)
    if mapped and mapped in chembl_name_index:
        pref, smi, cid = chembl_name_index[mapped]
        resolved[drug] = (pref, smi, cid, f'extra_map ({key}→{mapped})')
        second_pass_resolved += 1
        found = True
    
    # Try manual metal complex SMILES
    if not found and key in MANUAL_METAL_SMILES:
        pref, smi, cid, source = MANUAL_METAL_SMILES[key]
        resolved[drug] = (pref, smi, cid, f'manual_metal ({source})')
        second_pass_resolved += 1
        found = True
    
    if not found:
        # Extract first recognizable word: "0.06% Resiquimod - A" → "resiquimod"
        words = re.findall(r'[a-zA-Z][a-zA-Z0-9-]+', key)
        for word in words:
            w = word.lower()
            if len(w) >= 4 and w in chembl_name_index:
                pref, smi, cid = chembl_name_index[w]
                resolved[drug] = (pref, smi, cid, f'word_extract ({w})')
                second_pass_resolved += 1
                found = True
                break
    
    if not found:
        # Try prefix match in SQLite: "cyclophosphamid" → "cyclophosphamide%"
        cur2.execute("""
            SELECT md.pref_name, cs.canonical_smiles, md.chembl_id
            FROM molecule_dictionary md
            JOIN compound_structures cs ON md.molregno = cs.molregno
            WHERE LOWER(md.pref_name) LIKE ?
            LIMIT 1
        """, (key + '%',))
        row = cur2.fetchone()
        if row:
            resolved[drug] = (row[0], row[1], row[2], 'prefix_sql')
            second_pass_resolved += 1
            found = True
    
    if not found:
        still_unresolved.append(drug)

# Update unresolved list
unresolved = still_unresolved

conn2.close()

print(f'=== SECOND PASS ===')
print(f'Resolved in second pass: {second_pass_resolved}')
print(f'Total resolved: {len(resolved)}/{len(all_drug_names)} ({100*len(resolved)/len(all_drug_names):.1f}%)')
print(f'Combos: {len(combos)}')
print(f'Still unresolved: {len(unresolved)}')

# Match type breakdown (updated)
type_counts = Counter(v[3] for v in resolved.values())
print(f'\nAll match types:')
for mt, count in type_counts.most_common():
    print(f'  {mt}: {count}')

=== SECOND PASS ===
Resolved in second pass: 23
Total resolved: 901/1158 (77.8%)
Combos: 46
Still unresolved: 211

All match types:
  exact: 862
  paren_base: 8
  paren_inner: 3
  prefix_sql: 3
  special_char_strip: 2
  dosage_strip: 2
  qualifier_strip (only): 1
  manual_metal (PubChem CID 498142): 1
  word_extract (hzn-825): 1
  manual_metal (PubChem CID 9887054): 1
  manual_metal (PubChem CID 441203): 1
  word_extract (onc201): 1
  extra_map (zpl389→adriforant): 1
  word_extract (co-trimoxazole): 1
  word_extract (ono-7475): 1
  extra_map (gsk2586184→solcitinib): 1
  word_extract (mek162): 1
  extra_map (auy922→luminespib): 1
  extra_map (bkm120→buparlisib): 1
  word_extract (pge1): 1
  extra_map (lcq908→pradigastat): 1
  word_extract (pazopanib): 1
  word_extract (a-002): 1
  word_extract (resiquimod): 1
  word_extract (mk2206): 1
  word_extract (azelaprag): 1
  word_extract (pf-02545920): 1


In [18]:
# 9d. Report unresolved drugs and apply ChEMBL SMILES to corrected outcomes
#
# Every drug is now either:
# - resolved: has ChEMBL-validated SMILES
# - combo: multi-drug intervention, SMILES cleared (was wrong anyway)
# - unresolved: no ChEMBL match — biologics, experimental codes, or junk entries
#
# ZERO SILENT DATA LOSS: every drug is printed and accounted for.

co_temp = pd.read_csv(SOURCES_DIR / '12_trials_corrected_outcomes.csv')

print(f'=== UNRESOLVED DRUGS ({len(unresolved)}) ===')
print('These drugs have no ChEMBL SMILES. SMILES will be set to NaN.')
print('If any should be resolved, add to EXTRA_MAPPINGS or BRAND_TO_GENERIC.\n')

# Separate unresolved into: had SMILES before vs never had
drugs_with_old_smiles = set(co_temp[co_temp['SMILES'].notna()]['Drug_Clean'].unique())
unresolved_lost = [(d, len(co_temp[co_temp['Drug_Clean']==d])) for d in unresolved if d in drugs_with_old_smiles]
unresolved_ok = [(d, len(co_temp[co_temp['Drug_Clean']==d])) for d in unresolved if d not in drugs_with_old_smiles]

print(f'--- Will LOSE SMILES ({len(unresolved_lost)} drugs, {sum(n for _,n in unresolved_lost)} trials) ---')
for d, n in sorted(unresolved_lost, key=lambda x: -x[1]):
    print(f'  {d}: {n} trials')

print(f'\n--- Never had SMILES ({len(unresolved_ok)} drugs — biologics, codes, junk) ---')
for d, n in sorted(unresolved_ok, key=lambda x: -x[1])[:20]:
    print(f'  {d}: {n} trials')
if len(unresolved_ok) > 20:
    print(f'  ... and {len(unresolved_ok)-20} more')

print(f'\n=== COMBINATION THERAPIES ({len(combos)}, {sum(len(co_temp[co_temp["Drug_Clean"]==d]) for d in combos)} trials) ===')
for d in sorted(combos):
    n = len(co_temp[co_temp['Drug_Clean'] == d])
    print(f'  {d}: {n} trials')

# --- Apply ChEMBL SMILES ---
print(f'\n{"="*60}')
print('APPLYING ChEMBL SMILES')
print(f'{"="*60}')

new_smiles_map = {drug: smi for drug, (pref, smi, cid, mt) in resolved.items()}
for drug in combos:
    new_smiles_map[drug] = None  # Clear combo SMILES

co_updated = co_temp.copy()
changed = 0
gained = 0
lost = 0
unchanged = 0

for idx, row in co_updated.iterrows():
    drug = row['Drug_Clean']
    old_smi = row['SMILES'] if pd.notna(row.get('SMILES')) else None
    new_smi = new_smiles_map.get(drug)
    
    if new_smi:
        if old_smi != new_smi:
            co_updated.at[idx, 'SMILES'] = new_smi
            changed += 1
            if old_smi is None:
                gained += 1
        else:
            unchanged += 1
    elif drug in new_smiles_map:
        # Explicitly None (combo)
        if old_smi is not None:
            co_updated.at[idx, 'SMILES'] = None
            lost += 1
    # Unresolved drugs: keep whatever they had (usually NaN)

old_with = co_temp['SMILES'].notna().sum()
new_with = co_updated['SMILES'].notna().sum()

print(f'Before: {old_with} trials with SMILES ({co_temp[co_temp["SMILES"].notna()]["Drug_Clean"].nunique()} drugs)')
print(f'After:  {new_with} trials with SMILES ({co_updated[co_updated["SMILES"].notna()]["Drug_Clean"].nunique()} drugs)')
print(f'Changed SMILES: {changed} trials (different molecule)')
print(f'Unchanged: {unchanged} trials (same SMILES)')
print(f'Gained: {gained} trials (newly resolved)')
print(f'Lost: {lost} trials (combos cleared)')

# Save
out_path = SOURCES_DIR / '12_trials_corrected_outcomes.csv'
co_updated.to_csv(out_path, index=False)
print(f'\nSaved: {out_path}')

# Also save the lookup table for provenance
lookup_df = pd.DataFrame([
    {'Drug_Clean': drug, 'chembl_pref_name': pref, 'chembl_smiles': smi,
     'chembl_id': cid, 'match_type': mt}
    for drug, (pref, smi, cid, mt) in resolved.items()
])
# Manual corrections: drugs where ChEMBL name search returned wrong compound
MANUAL_SMILES_CORRECTIONS = {
    'MP-101': {  # NCT03044249: mGlu2 agonist for dementia psychosis (LY2979165)
        'chembl_pref_name': 'LY2979165',
        'chembl_smiles': 'C[C@H](N)C(=O)N[C@@]1(C(=O)O)C[C@@H](Sc2nnc[nH]2)[C@H]2[C@H](C(=O)O)[C@H]21',
        'chembl_id': 'CHEMBL4303379',
        'match_type': 'manual_correction',
    },
}
for drug, vals in MANUAL_SMILES_CORRECTIONS.items():
    mask = lookup_df['Drug_Clean'] == drug
    if mask.sum() > 0:
        for k, v in vals.items():
            lookup_df.loc[mask, k] = v
    else:
        lookup_df = pd.concat([lookup_df, pd.DataFrame([{'Drug_Clean': drug, **vals}])], ignore_index=True)
    # Also fix in the resolved dict so downstream steps use correct SMILES
    resolved[drug] = (vals['chembl_pref_name'], vals['chembl_smiles'], vals['chembl_id'], vals['match_type'])
if MANUAL_SMILES_CORRECTIONS:
    print(f'Applied {len(MANUAL_SMILES_CORRECTIONS)} manual SMILES corrections: {list(MANUAL_SMILES_CORRECTIONS.keys())}')

lookup_df.to_csv(SOURCES_DIR / 'chembl_smiles_lookup.csv', index=False)
print(f'Saved: chembl_smiles_lookup.csv ({len(lookup_df)} drugs)')

=== UNRESOLVED DRUGS (211) ===
These drugs have no ChEMBL SMILES. SMILES will be set to NaN.
If any should be resolved, add to EXTRA_MAPPINGS or BRAND_TO_GENERIC.

--- Will LOSE SMILES (13 drugs, 15 trials) ---
  AZD5718: 2 trials
  TAK-994: 2 trials
  DFP-10917: 1 trials
  VK-2019: 1 trials
  K-877: 1 trials
  Ferroquine SSR97193: 1 trials
  ASM-024: 1 trials
  LY2510924: 1 trials
  KM-819: 1 trials
  ABI-H2158: 1 trials
  BAY1128688: 1 trials
  CNP520: 1 trials
  TNG348: 1 trials

--- Never had SMILES (198 drugs — biologics, codes, junk) ---
  Aflibercept: 4 trials
  LY2127399: 3 trials
  GSK3858279: 2 trials
  MT-1186: 2 trials
  WVE-004: 2 trials
  GT005: 2 trials
  KSI-301: 2 trials
  WVE-120101: 2 trials
  WVE-120102: 2 trials
  ABBV-8E12: 2 trials
  BIIB092: 2 trials
  REGN3500: 2 trials
  NEOD001: 2 trials
  Tabalumab Auto-Injector: 2 trials
  Tabalumab Prefilled Syringe: 2 trials
  AXT107 0.: 2 trials
  PRM-151 (Zinpentraxin Alfa): 2 trials
  Aducanumab (BIIB037): 2 trials
  G


Saved: <repo>/data/sources/12_trials_corrected_outcomes.csv
Applied 1 manual SMILES corrections: ['MP-101']
Saved: chembl_smiles_lookup.csv (901 drugs)


---
## 10. Disease Extraction

**Script**: `scripts/extract_drug_disease.py`  
Diseases are extracted from trial titles using 170+ regex patterns and 40+ abbreviations.  
Hierarchy: "in Patients With X" → "With X Disease" → "Treatment of X" → keyword scan.

In [19]:
# Disease coverage in the classified trial dataset
review = pd.read_csv(PROJECT_ROOT / 'data' / 'review' / 'trials_pass_fail_only.csv')
review_real = review[~review['NCT_ID'].str.startswith('EXT_', na=False)].copy()

has_disease = review_real['Disease'].notna() & (review_real['Disease'] != '')
print(f'Disease extraction coverage:')
print(f'  With disease: {has_disease.sum():,} ({100*has_disease.mean():.1f}%)')
print(f'  Without:      {(~has_disease).sum():,}')
print(f'\nTop 15 diseases:')
print(review_real['Disease'].value_counts().head(15))

Disease extraction coverage:
  With disease: 9,321 (75.0%)
  Without:      3,099

Top 15 diseases:
Disease
Multiple Myeloma                341
Breast Cancer                   302
Lung Cancer                     283
Acute Myeloid Leukemia          217
Prostate Cancer                 214
Type 2 Diabetes                 165
Type 2 Diabetes Mellitus        148
Rheumatoid Arthritis            143
Metastatic Breast Cancer        115
Hodgkin Lymphoma                113
Cell Lymphoma                   107
Acute Lymphoblastic Leukemia     91
Major Depressive Disorder        90
Cell Carcinoma                   87
Metastatic Colorectal Cancer     81
Name: count, dtype: int64


---
## 11. Biological Feature Sources

Features are computed from internal pipelines, not external databases.

| Source | Data | Features | Files |
|--------|------|----------|-------|
| Binding | Drug × protein binding scores | 75 tissue interaction features | `data/raw/gabe-run/`, `gabe-run2/` |
| STRING-DB | Pathway enrichment | 36 network features | `data/raw/receptors/`, `gabe-run2/` |
| OpenTargets | Disease-associated genes | 6 target-disease features | API cache |
| HPA | Tissue expression | Integrated into Binding | Embedded in binding outputs |

In [20]:
# Raw feature data availability
gr1 = list((RAW_DIR / 'gabe-run' / 'output' / 'drug-target' / 'binding').glob('*.tsv')) \
      if (RAW_DIR / 'gabe-run' / 'output' / 'drug-target' / 'binding').exists() else []
gr2 = list((RAW_DIR / 'gabe-run2' / 'output' / 'missing-networks' / 'binding').glob('*.tsv')) \
      if (RAW_DIR / 'gabe-run2' / 'output' / 'missing-networks' / 'binding').exists() else []
rr2 = list((RAW_DIR / 'gabe-run2' / 'output' / 'missing-networks' / 'receptors').glob('*network_enrichment.tsv')) \
      if (RAW_DIR / 'gabe-run2' / 'output' / 'missing-networks' / 'receptors').exists() else []

print(f'Binding binding: run1={len(gr1)}, run2={len(gr2)}, total=~{len(gr1)+len(gr2)} drugs')
print(f'Network enrichment: {len(rr2)} drug files')

ot_cache = CACHE_DIR / 'disease_targets_cache.json'
if ot_cache.exists():
    with open(ot_cache) as f:
        print(f'OpenTargets disease cache: {len(json.load(f))} diseases')

Binding binding: run1=0, run2=0, total=~0 drugs
Network enrichment: 0 drug files


OpenTargets disease cache: 5140 diseases


---
## 12–13. Final Dataset Assembly, Validation & Reprocessing List

This section is **self-contained** — it loads saved files from earlier sections and
produces all final outputs. No dependency on in-memory variables from prior cells.

**Inputs:**
- AACT classified trials (from Section 7+7b): keyword + manual review outcomes
- ChEMBL SMILES lookup (from Section 9): validated drug structures
- ChEMBL compound results: withdrawn_flag for post-market withdrawals
- Pipeline feature file: `STAR_complete.csv`
- Disease targets cache + drug targets file

**Outputs:**
- `12_trials_corrected_outcomes.csv` — all usable trials with corrected outcomes
- `training_dataset_corrected.csv` — trials with pipeline features
- `pipeline_reprocessing_trial_level.csv` — for bioinformatics team (per-trial)
- `pipeline_reprocessing_drug_level.csv` — for bioinformatics team (per-drug)

**Outcome rules:**
- **PASS** = Phase III+ completed
- **FAIL_SAFETY** = terminated with safety keywords OR withdrawn from market
- **FAIL_EFFICACY** = terminated with efficacy keywords
- **FAIL_BOTH** = both safety and efficacy issues
- **Excluded**: IN_PROCESS (Phase I/II), ENROLLMENT, BUSINESS, non-therapeutic

In [21]:
# === 12-13. SELF-CONTAINED: Final dataset assembly + reprocessing list ===
#
# This cell loads all inputs from disk and produces all outputs.
# No dependency on variables from earlier cells.

import pandas as pd
import numpy as np
import json
import os
import glob
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import Descriptors, MolToSmiles, inchi as rdkinchi
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('..').resolve()
SOURCES_DIR = PROJECT_ROOT / 'data' / 'sources'

# =====================================================================
# STEP 1: Load raw classified trials and apply corrections
# =====================================================================

# Load the classified+enriched trial data (from Sections 1-8)
# This has: NCT_ID, Drug, Phase, Final_Outcome, Why_Stopped, Disease, SMILES, etc.
aact_classified = pd.read_csv(SOURCES_DIR / '07_aact_terminated_classified.csv', low_memory=False)
print(f'Step 1: Loaded {len(aact_classified):,} classified AACT records')

# Load the review dataset that has Drug_Clean, Disease, SMILES already populated
# This was built by earlier cells — it's the merged AACT + SMILES + Disease data
# We need to reconstruct review_real from saved components

# Actually, let's load the FULL trial review file that Section 7b+9 produced
# The most complete pre-outcome file should be the last save before this section
# Let's check what we have:
if os.path.exists(SOURCES_DIR / '12_trials_corrected_outcomes.csv'):
    # Use the existing file as our starting point, but RE-DERIVE outcomes
    # to make sure they're consistent with current keyword rules
    trials = pd.read_csv(SOURCES_DIR / '12_trials_corrected_outcomes.csv')
    print(f'Loaded existing corrected outcomes: {len(trials)} trials')
else:
    raise FileNotFoundError('12_trials_corrected_outcomes.csv not found — run Sections 1-9 first')

# =====================================================================
# STEP 2: Apply outcome classification rules
# =====================================================================

# Phase III+ pass rule
def classify_outcome(row):
    """Classify trial outcome. Returns one of: PASS, FAIL_SAFETY, FAIL_EFFICACY,
    FAIL_BOTH, IN_PROCESS, ENROLLMENT, BUSINESS."""
    fo = row.get('Final_Outcome', '')
    co = row.get('Corrected_Outcome', '')
    
    # If already has a corrected outcome from manual review, use it
    if co in ('FAIL_SAFETY', 'FAIL_EFFICACY', 'FAIL_BOTH', 'ENROLLMENT', 'BUSINESS'):
        return co
    
    # PASS = Phase III+ completed
    if fo == 'PASS_APPROVED' or co == 'PASS':
        phase = str(row.get('Phase', ''))
        if 'Phase 3' in phase or 'Phase 4' in phase:
            return 'PASS'
        else:
            return 'IN_PROCESS'
    
    return co if co else 'unknown'

# Load manual review corrections
batch1 = pd.read_csv(SOURCES_DIR / 'REVIEW_ambiguous_classification.csv')
batch2 = pd.read_csv(SOURCES_DIR / 'REVIEW_remaining_classifications.csv')
all_manual = pd.concat([
    batch1[['NCT_ID', 'Manual_Classification']],
    batch2[['NCT_ID', 'Manual_Classification']]
])
manual_map = dict(zip(all_manual['NCT_ID'], all_manual['Manual_Classification']))

# Apply manual corrections
n_manual = 0
for nct_id, label in manual_map.items():
    mask = trials['NCT_ID'] == nct_id
    if mask.any() and trials.loc[mask, 'Corrected_Outcome'].iloc[0] != label:
        trials.loc[mask, 'Corrected_Outcome'] = label
        n_manual += mask.sum()
print(f'Step 2: Manual corrections applied: {n_manual} trial rows')

# =====================================================================
# STEP 3: [REMOVED] Withdrawn drug reclassification
# =====================================================================
# PREVIOUSLY: reclassified PASS → FAIL_SAFETY for drugs with ChEMBL withdrawn_flag.
# REMOVED because this is TEMPORAL LEAKAGE:
#   - These drugs PASSED their clinical trials (Phase III+ completion)
#   - They were withdrawn POST-MARKETING, often years/decades later
#   - Using future withdrawal data to relabel past trial outcomes leaks information
#   - This is the same type of contamination as the DILIrank/Withdrawn injection (391 fake trials)
#     that inflated safety AUC from 0.836 to 0.949
#   - The ~48 affected trials (22 drugs: phentermine, thalidomide, ketorolac, etc.)
#     are restored to their original PASS label
#
# Withdrawn drugs are used as a VALIDATION SET instead (Section 15b):
#   "Does the model assign high safety risk to drugs later withdrawn?"
#
# See CLAUDE.md: "A trial outcome is a fact about THAT TRIAL, not about future events."
drug_col = 'Drug_Clean' if 'Drug_Clean' in trials.columns else 'Drug'
# Restore any trials previously reclassified by withdrawn flag
# (from prior notebook runs that baked the reclassification into the CSV)
restored = (trials['Final_Outcome'] == 'PASS_APPROVED') & (trials['Corrected_Outcome'] == 'FAIL_SAFETY')
n_restored = restored.sum()
if n_restored > 0:
    trials.loc[restored, 'Corrected_Outcome'] = 'PASS'
    print(f'Step 3: Restored {n_restored} trials from FAIL_SAFETY → PASS (withdrawn reclassification reversed)')
    for drug in sorted(trials.loc[restored, drug_col].unique()):
        n = restored[trials[drug_col] == drug].sum()
        print(f'  {drug}: {n} trials restored to PASS')
else:
    print(f'Step 3: No withdrawn reclassification to reverse')

# =====================================================================
# STEP 4: Exclude non-usable categories
# =====================================================================

# Clean drug names: strip trademark symbols, trailing semicolons
if drug_col in trials.columns:
    trials[drug_col] = trials[drug_col].str.replace(r'[®™]', '', regex=True).str.rstrip(';').str.strip()
    # Also clean the raw Drug column
    if 'Drug' in trials.columns:
        trials['Drug'] = trials['Drug'].str.replace(r'[®™]', '', regex=True).str.rstrip(';').str.strip()

# Exclude ENROLLMENT, BUSINESS, non-therapeutic
excl_mask = trials['Corrected_Outcome'].isin(['ENROLLMENT', 'BUSINESS'])
NON_THERAPEUTIC = {'0.9% Sodium chloride', 'Sodium Chloride', 'Broncho-Vaxom',
                   'NaCl 0.9%', 'Sodium Lactate', '11C-MC1', '11C-PS13'}
# Match non-therapeutic on both Drug and Drug_Clean columns
# Use partial matching for controls (e.g., "5% glucose Infusion solution" → "5% glucose")
non_drug_mask = pd.Series(False, index=trials.index)
for nd in NON_THERAPEUTIC:
    if 'Drug' in trials.columns:
        non_drug_mask |= trials['Drug'].str.contains(nd, case=False, na=False, regex=False)
    if drug_col in trials.columns:
        non_drug_mask |= trials[drug_col].str.contains(nd, case=False, na=False, regex=False)

usable = trials[~excl_mask & ~non_drug_mask &
                trials['Corrected_Outcome'].isin(['PASS', 'FAIL_SAFETY', 'FAIL_EFFICACY', 'FAIL_BOTH'])].copy()
in_process = trials[trials['Corrected_Outcome'] == 'IN_PROCESS'].copy()

print(f'\nStep 4: Dataset split')
print(f'  Usable (PASS/FAIL): {len(usable):,} trials, {usable[drug_col].nunique()} drugs')
print(f'  IN_PROCESS:         {len(in_process):,} trials')
print(f'  Excluded:           {excl_mask.sum() + non_drug_mask.sum()} trials')
print(f'\nOutcome distribution:')
print(usable['Corrected_Outcome'].value_counts().to_string())
print(f'\nWith SMILES: {usable["SMILES"].notna().sum()}')
print(f'Without SMILES: {usable["SMILES"].isna().sum()}')

# =====================================================================
# STEP 4b: Strip salt forms from multicomponent SMILES
# =====================================================================
# Many SMILES contain salt counterions, hydration waters, or solvates
# (e.g., "CCCCN1CCCCC1C(=O)Nc1c(C)cccc1C.Cl" = bupivacaine HCl).
# For structure-based features, we need the parent drug only.
# Strategy: keep the largest fragment (most heavy atoms).

from rdkit import Chem
from rdkit.Chem import Descriptors

def strip_salt(smiles):
    if pd.isna(smiles) or '.' not in str(smiles):
        return smiles
    parts = str(smiles).split('.')
    best, best_heavy = parts[0], 0
    for p in parts:
        mol = Chem.MolFromSmiles(p)
        if mol:
            n = mol.GetNumHeavyAtoms()
            if n > best_heavy:
                best, best_heavy = p, n
    return best

n_multi = usable['SMILES'].str.contains(r'\.', na=False, regex=True).sum()
usable['SMILES'] = usable['SMILES'].apply(strip_salt)
n_multi_after = usable['SMILES'].str.contains(r'\.', na=False, regex=True).sum()
print(f'\nStep 4b: Salt stripping')
print(f'  Multicomponent SMILES before: {n_multi}')
print(f'  After stripping: {n_multi_after} (should be 0)')

# =====================================================================
# STEP 4b2: Neutralize SMILES (remove residual charges from salt stripping)
# =====================================================================
# After stripping counterions, some drugs retain charges: COO- vs COOH,
# N+ vs N, PO4 2- vs PO4H2. These are the same molecule in different
# protonation states. Neutralize to canonical uncharged form so that
# SMILES dedup (Step 4d) correctly collapses them.
# Example: diclofenac (O=C(O)Cc1...) vs diclofenac potassium (O=C([O-])Cc1...)

from rdkit.Chem.MolStandardize import rdMolStandardize

uncharger = rdMolStandardize.Uncharger()

def neutralize_smiles(smi):
    if pd.isna(smi):
        return smi
    mol = Chem.MolFromSmiles(str(smi))
    if not mol:
        return smi
    mol = uncharger.uncharge(mol)
    return Chem.MolToSmiles(mol)

n_charged = usable['SMILES'].dropna().apply(
    lambda s: '-' in str(s) or '+' in str(s)
).sum()
usable['SMILES'] = usable['SMILES'].apply(neutralize_smiles)
n_charged_after = usable['SMILES'].dropna().apply(
    lambda s: '-' in str(s) or '+' in str(s)
).sum()
print(f'\nStep 4b2: Neutralization')
print(f'  Charged SMILES before: {n_charged}')
print(f'  After neutralization: {n_charged_after}')

# =====================================================================
# STEP 4c: Exclude non-therapeutic entries
# =====================================================================
# These are imaging agents, controls, non-drugs, or too-small to be
# meaningful drug molecules. Each was identified by manual review.

ADDITIONAL_EXCLUSIONS = {
    'Azole',               # pyrrole ring (MW=67), not a specific drug
    'Butyrate enemas',     # butyric acid (MW=88)
    'indocyanine green',   # imaging dye, not therapeutic
    'Sugar pill',          # placebo
    'Sodium Nitroprusside', # inorganic complex, no valid SMILES features
}

excl_4c = usable[drug_col].isin(ADDITIONAL_EXCLUSIONS)
n_excl_4c = excl_4c.sum()
if n_excl_4c > 0:
    removed_names = sorted(set(usable.loc[excl_4c, drug_col]))
    usable = usable[~excl_4c].copy()
    print(f'\nStep 4c: Additional non-therapeutic exclusions: {n_excl_4c} trials')
    for name in removed_names:
        print(f'  Removed: {name}')
else:
    print(f'\nStep 4c: No additional exclusions needed')

# =====================================================================
# STEP 4d: Collapse duplicate SMILES -> canonical Drug_Clean
# =====================================================================
# 74 SMILES groups have multiple Drug_Clean names (brand vs generic,
# case variants, combo regimen names containing one drug's SMILES).
# Collapse to one canonical name per SMILES. Heuristic:
# 1. Ignore names with commas (combo regimens)
# 2. Ignore names with dose info (mg, ml, QD, BID)
# 3. Prefer shorter generic-looking names
# 4. Fall back to most frequent name

import re

def pick_canonical_name(names, trial_counts):
    if len(names) == 1:
        return names[0]

    scored = []
    for n in names:
        score = 0
        # Penalize combo names (contain comma or '+')
        if ',' in n or '+' in n:
            score -= 100
        # Penalize dose/formulation info
        if re.search(r'\d+\s*(mg|ml|mcg|miligram|milliliter)', n, re.I):
            score -= 50
        if re.search(r'\b(QD|BID|TID|once|twice)\b', n, re.I):
            score -= 50
        # Penalize long parenthetical suffixes
        if '(' in n and len(n) > 30:
            score -= 30
        # Penalize ALL CAPS or codes (e.g., SEL, MOR, ASA)
        if n.isupper() and len(n) <= 5:
            score -= 20
        # Prefer lowercase (INN convention) or Title case
        if n[0].islower():
            score += 10
        # Prefer shorter names
        score -= len(n) * 0.1
        # Tiebreak: trial count
        score += trial_counts.get(n, 0) * 0.01
        scored.append((score, n))

    scored.sort(reverse=True)
    return scored[0][1]

# Build trial counts per Drug_Clean
trial_counts = usable[drug_col].value_counts().to_dict()

# Find SMILES with multiple Drug_Clean names
smiles_groups = usable.dropna(subset=['SMILES']).groupby('SMILES')[drug_col].apply(
    lambda x: sorted(x.unique())
)
dup_groups = smiles_groups[smiles_groups.apply(len) > 1]

canonical_map = {}  # old_name -> canonical_name
n_renamed = 0
for smi, names in dup_groups.items():
    canonical = pick_canonical_name(list(names), trial_counts)
    for n in names:
        if n != canonical:
            canonical_map[n] = canonical
            n_renamed += 1

# Apply the mapping
usable[drug_col] = usable[drug_col].replace(canonical_map)
print(f'\nStep 4d: Drug name deduplication')
print(f'  Duplicate SMILES groups: {len(dup_groups)}')
print(f'  Names collapsed: {n_renamed}')
print(f'  Examples:')
for old, new in sorted(canonical_map.items())[:10]:
    print(f'    "{old}" -> "{new}"')

# Remove true duplicate rows that emerged from renaming
# (same NCT_ID + same Drug_Clean after collapse)
before_dedup = len(usable)
usable = usable.drop_duplicates(subset=['NCT_ID', drug_col])
after_dedup = len(usable)
print(f'  Duplicate rows removed after collapse: {before_dedup - after_dedup}')

# Final counts
print(f'\nStep 4 final: {len(usable):,} usable trials, {usable[drug_col].nunique()} drugs, {usable["SMILES"].dropna().nunique()} unique SMILES')
print(usable['Corrected_Outcome'].value_counts().to_string())

# Apply manual SMILES corrections before saving (must match cell 28 corrections)
# These drugs had ChEMBL name collisions that returned wrong compounds
_chembl_corrected = pd.read_csv(SOURCES_DIR / 'chembl_smiles_lookup.csv')
_manual = _chembl_corrected[_chembl_corrected['match_type'] == 'manual_correction']
for _, mrow in _manual.iterrows():
    mask = usable[drug_col] == mrow['Drug_Clean']
    if mask.sum() > 0:
        usable.loc[mask, 'SMILES'] = mrow['chembl_smiles']
        print(f'  Manual SMILES fix: {mrow["Drug_Clean"]} → {mrow["chembl_pref_name"]}')

# Save
usable.to_csv(SOURCES_DIR / '12_trials_corrected_outcomes.csv', index=False)
in_process.to_csv(SOURCES_DIR / '12_trials_in_process.csv', index=False)
print(f'\nSaved: 12_trials_corrected_outcomes.csv ({len(usable):,} trials)')


# =====================================================================
# STEP 5: Build training dataset (trials with pipeline features)
# =====================================================================

dtc = pd.read_csv(PROJECT_ROOT / 'data' / 'STAR_complete.csv', low_memory=False)
usable_ncts = set(usable['NCT_ID'])

# Filter to usable trials, exclude fake EXT_ entries
train = dtc[dtc['NCT_ID'].isin(usable_ncts)].copy()
train = train[~train['NCT_ID'].str.startswith('EXT_', na=False)]

# Apply corrected outcomes
outcome_map = dict(zip(usable['NCT_ID'], usable['Corrected_Outcome']))
train['Corrected_Outcome'] = train['NCT_ID'].map(outcome_map)
train = train[train['Corrected_Outcome'].isin(['PASS', 'FAIL_SAFETY', 'FAIL_EFFICACY', 'FAIL_BOTH'])].copy()

# Update SMILES from ChEMBL lookup
chembl_lookup = pd.read_csv(SOURCES_DIR / 'chembl_smiles_lookup.csv')
chembl_smi = dict(zip(chembl_lookup['Drug_Clean'], chembl_lookup['chembl_smiles']))
chembl_ids = dict(zip(chembl_lookup['Drug_Clean'], chembl_lookup['chembl_id']))
for idx, row in train.iterrows():
    drug = row.get('Drug_Clean')
    if drug and drug in chembl_smi:
        train.at[idx, 'SMILES'] = chembl_smi[drug]

# Apply same cleanup as Steps 4b-4d to training data
# 1. Strip salts and neutralize SMILES
train['SMILES'] = train['SMILES'].apply(strip_salt)
train['SMILES'] = train['SMILES'].apply(neutralize_smiles)
# 2. Exclude non-therapeutic entries
train = train[~train['Drug_Clean'].isin(ADDITIONAL_EXCLUSIONS)].copy()
# 3. Replace Drug_Clean with canonical name from usable (which is already deduped)
#    This handles cases where STAR_complete.csv has variant names
nct_drug_canonical = dict(zip(
    usable['NCT_ID'] + '|' + usable[drug_col].str.lower(),
    usable[drug_col]
))
# Also build SMILES-to-canonical mapping from usable
smiles_canonical = {}
for smi, grp in usable.dropna(subset=['SMILES']).groupby('SMILES'):
    smiles_canonical[smi] = grp[drug_col].iloc[0]
# Apply: first try NCT+name match, then SMILES match
for idx, row in train.iterrows():
    nct_key = str(row['NCT_ID']) + '|' + str(row.get('Drug_Clean', '')).lower()
    if nct_key in nct_drug_canonical:
        train.at[idx, 'Drug_Clean'] = nct_drug_canonical[nct_key]
    elif row.get('SMILES') in smiles_canonical:
        train.at[idx, 'Drug_Clean'] = smiles_canonical[row['SMILES']]
# 4. Deduplicate rows
train = train.drop_duplicates(subset=['NCT_ID', 'Drug_Clean'])

# 5. Final dedup pass: collapse any remaining SMILES groups to canonical name
#    Combo regimen names (e.g., "Paclitaxel/Cisplatin") may share SMILES with single drug
train_smi_groups = train.dropna(subset=['SMILES']).groupby('SMILES')['Drug_Clean'].apply(
    lambda x: sorted(x.unique())
)
train_dups = train_smi_groups[train_smi_groups.apply(len) > 1]
if len(train_dups) > 0:
    train_canonical = {}
    for smi, names in train_dups.items():
        canonical = pick_canonical_name(list(names), train['Drug_Clean'].value_counts().to_dict())
        for n in names:
            if n != canonical:
                train_canonical[n] = canonical
    train['Drug_Clean'] = train['Drug_Clean'].replace(train_canonical)
    train = train.drop_duplicates(subset=['NCT_ID', 'Drug_Clean'])
    print(f'  Final dedup: {len(train_dups)} groups collapsed, {len(train_canonical)} names mapped')

train.to_csv(SOURCES_DIR / 'training_dataset_corrected.csv', index=False)
print(f'\nStep 5: Training dataset')
print(f'  Trials with features: {len(train):,}')
print(f'  Unique drugs: {train["SMILES"].nunique()}')
print(f'  Outcomes:')
print(train['Corrected_Outcome'].value_counts().to_string())

# =====================================================================
# STEP 6: Audit feature validity (SMILES correctness)
# =====================================================================

train_drugs = train.drop_duplicates('Drug_Clean')[['Drug_Clean', 'SMILES']].copy()
proven_wrong = []
for _, row in train_drugs.iterrows():
    drug = row['Drug_Clean']
    old_smi = row['SMILES']
    new_smi = chembl_smi.get(drug)
    if not new_smi:
        continue
    old_mol = Chem.MolFromSmiles(str(old_smi)) if pd.notna(old_smi) else None
    new_mol = Chem.MolFromSmiles(str(new_smi))
    if not old_mol or not new_mol:
        proven_wrong.append(drug)
        continue
    if MolToSmiles(old_mol) != MolToSmiles(new_mol):
        if abs(Descriptors.ExactMolWt(old_mol) - Descriptors.ExactMolWt(new_mol)) > 5:
            proven_wrong.append(drug)

print(f'\nStep 6: Feature audit')
print(f'  Drugs with wrong SMILES in features: {len(proven_wrong)}')
for d in proven_wrong:
    n = len(train[train['Drug_Clean'] == d])
    print(f'    {d}: {n} trials')

# =====================================================================
# STEP 7: Generate reprocessing files for bioinformatics team
# =====================================================================

# Load all needed data
pipeline_backlog = pd.read_csv(SOURCES_DIR / 'drugs_needing_pipeline_clean.csv')
drug_targets_df = pd.read_csv(SOURCES_DIR / 'pipeline_drugs_with_targets.csv')
drug_targets_map = dict(zip(drug_targets_df['drug_name'], drug_targets_df['drug_targets_enst']))

disease_cache = json.load(open(PROJECT_ROOT / 'data' / 'cache' / 'disease_targets_cache.json'))
disease_name_to_id = {k.replace('search:', ''): v 
                      for k, v in disease_cache.items() 
                      if k.startswith('search:') and isinstance(v, str)}

# Gene symbol → ENST mapping (for converting disease targets)
gene_enst_df = pd.read_csv(SOURCES_DIR / 'pipeline_gene_to_enst.csv')
gene_to_enst = {}
for _, row in gene_enst_df.iterrows():
    if pd.notna(row['enst_id']):
        gene_to_enst.setdefault(row['gene_symbol'], []).append(row['enst_id'])
print(f'Gene-to-ENST mapping: {len(gene_to_enst)} genes')

backlog_smi = dict(zip(pipeline_backlog['drug_name'], pipeline_backlog['smiles']))
backlog_ik = dict(zip(pipeline_backlog['drug_name'], pipeline_backlog['inchikey']))

# All drugs needing pipeline runs
reprocess_drugs = set(pipeline_backlog['drug_name']) | set(proven_wrong)

def get_smi_ik(drug):
    smi = chembl_smi.get(drug) or backlog_smi.get(drug)
    if not smi or pd.isna(smi):
        return None, None
    mol = Chem.MolFromSmiles(str(smi))
    if not mol:
        return smi, backlog_ik.get(drug)
    inch = rdkinchi.MolToInchi(mol)
    ik = rdkinchi.InchiToInchiKey(inch) if inch else backlog_ik.get(drug)
    return smi, ik

def get_disease_targets_top50(disease):
    """Top 50 disease gene symbols by OpenTargets score. Returns (symbols_str, enst_str)."""
    if not disease or pd.isna(disease):
        return '', ''
    did = disease_name_to_id.get(str(disease).lower())
    if not did:
        return '', ''
    entry = disease_cache.get(f'targets:{did}')
    if not entry or not isinstance(entry, dict):
        return '', ''
    targets = entry.get('targets', [])
    if not targets:
        return '', ''
    top = sorted(targets, key=lambda t: t.get('score', 0), reverse=True)[:50]
    symbols = [t['symbol'] for t in top]
    # Map symbols to ENST IDs (all transcripts per gene)
    enst_ids = []
    for s in symbols:
        enst_ids.extend(gene_to_enst.get(s, []))
    return ';'.join(symbols), ';'.join(enst_ids)  # both available, ENST stored in mapping file

# Match drugs to trials using BOTH Drug and Drug_Clean columns (case-insensitive)
reprocess_lower = {d.lower() for d in reprocess_drugs}

# --- Trial-level file ---
trial_rows = []
for _, row in usable.iterrows():
    raw_drug = str(row.get('Drug', ''))
    clean_drug = str(row.get('Drug_Clean', ''))
    
    if raw_drug.lower() not in reprocess_lower and clean_drug.lower() not in reprocess_lower:
        continue
    
    # Use whichever name matches for lookups
    lookup = raw_drug if raw_drug in reprocess_drugs else clean_drug
    smi, ik = get_smi_ik(lookup)
    if not smi:
        smi, ik = get_smi_ik(clean_drug)
    if not smi:
        smi, ik = get_smi_ik(raw_drug)
    
    disease = row.get('Disease', '')
    dt_symbols, dt_enst = get_disease_targets_top50(disease)
    drug_tgt = drug_targets_map.get(lookup, drug_targets_map.get(clean_drug, ''))
    if pd.isna(drug_tgt):
        drug_tgt = ''
    
    trial_rows.append({
        'NCT_ID': row['NCT_ID'],
        'Drug_Clean': clean_drug,
        'SMILES': smi,
        'InChIKey': ik,
        'ChEMBL_ID': chembl_ids.get(lookup, chembl_ids.get(clean_drug, '')),
        'Disease': disease if pd.notna(disease) else '',
        'Corrected_Outcome': row['Corrected_Outcome'],
        'Drug_Targets_ENST': str(drug_tgt),
        'Disease_Targets_Symbols': dt_symbols,
        'Disease_Targets_ENST': dt_enst,
    })

trial_df = pd.DataFrame(trial_rows)
trial_path = SOURCES_DIR / 'pipeline_reprocessing_trial_level.csv'
trial_df.to_csv(trial_path, index=False)

# --- Drug-level file ---
drug_rows = []
for drug in sorted(reprocess_drugs):
    smi, ik = get_smi_ik(drug)
    
    # Find all usable trials for this drug (case-insensitive on both columns)
    mask = pd.Series(False, index=usable.index)
    if 'Drug' in usable.columns:
        mask |= usable['Drug'].str.lower().eq(drug.lower())
    if 'Drug_Clean' in usable.columns:
        mask |= usable['Drug_Clean'].str.lower().eq(drug.lower())
    drug_trials = usable[mask]
    diseases = sorted(drug_trials['Disease'].dropna().unique())
    
    # Aggregate disease targets (both symbols and ENST)
    all_symbols = set()
    all_enst = set()
    for d in diseases:
        dt_sym, dt_enst = get_disease_targets_top50(d)
        if dt_sym:
            all_symbols.update(dt_sym.split(';'))
        if dt_enst:
            all_enst.update(dt_enst.split(';'))
    all_symbols.discard('')
    all_enst.discard('')
    
    drug_tgt = drug_targets_map.get(drug, '')
    if pd.isna(drug_tgt):
        drug_tgt = ''
    
    drug_rows.append({
        'Drug_Clean': drug,
        'SMILES': smi,
        'InChIKey': ik,
        'ChEMBL_ID': chembl_ids.get(drug, ''),
        'Diseases': '|'.join(diseases),
        'N_Diseases': len(diseases),
        'N_Usable_Trials': len(drug_trials),
        'Drug_Targets_ENST': str(drug_tgt),
        'Disease_Targets_Symbols': ';'.join(sorted(all_symbols)),
        'Disease_Targets_ENST': ';'.join(sorted(all_enst)),
        'N_Disease_Target_Genes': len(all_symbols),
    })

drug_df = pd.DataFrame(drug_rows)
drug_path = SOURCES_DIR / 'pipeline_reprocessing_drug_level.csv'
drug_df.to_csv(drug_path, index=False)

# =====================================================================
# SUMMARY
# =====================================================================
print(f'\n{"="*60}')
print('FINAL SUMMARY')
print(f'{"="*60}')
print(f'\nCorrected outcomes: {len(usable):,} usable trials')
print(usable['Corrected_Outcome'].value_counts().to_string())
print(f'\nTraining (with features): {len(train):,} trials, {train["SMILES"].nunique()} drugs')
print(f'Wrong SMILES in features: {len(proven_wrong)} drugs')
print(f'\nReprocessing trial-level: {len(trial_df)} rows, {trial_df["Drug_Clean"].nunique()} drugs')
print(f'Reprocessing drug-level: {len(drug_df)} rows')
print(f'  With SMILES: {drug_df["SMILES"].notna().sum()}')
print(f'  With usable trials: {(drug_df["N_Usable_Trials"] > 0).sum()}')
print(f'  With drug targets: {(drug_df["Drug_Targets_ENST"] != "").sum()}')
print(f'  With disease targets: {(drug_df["N_Disease_Target_Genes"] > 0).sum()}')
print(f'  Max disease target genes: {drug_df["N_Disease_Target_Genes"].max()}')
print(f'\nFile sizes:')
print(f'  trial-level: {trial_path.stat().st_size / 1024:.0f} KB')
print(f'  drug-level:  {drug_path.stat().st_size / 1024:.0f} KB')

Step 1: Loaded 53,811 classified AACT records
Loaded existing corrected outcomes: 4932 trials
Step 2: Manual corrections applied: 0 trial rows
Step 3: No withdrawn reclassification to reverse

Step 4: Dataset split
  Usable (PASS/FAIL): 4,932 trials, 1158 drugs
  IN_PROCESS:         0 trials
  Excluded:           0 trials

Outcome distribution:
Corrected_Outcome
PASS             3997
FAIL_EFFICACY     694
FAIL_SAFETY       222
FAIL_BOTH          19

With SMILES: 4666
Without SMILES: 266



Step 4b: Salt stripping
  Multicomponent SMILES before: 55
  After stripping: 0 (should be 0)


[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Removed negative charge.
[08:48:59] Removed negative charge.
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger


[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Run

[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:48:59] Running Uncharger
[08:49:00] Removed negative charge.
[08:49:00] Running Uncharger
[08:49:00] Running Uncharger
[08:49:00] Running Uncharger
[08:49:00] Running Uncharger
[08:49:00] Running Uncharger
[08:49:00] Running Uncharger
[08:49:00] Running Uncharger
[08:49:00] Running Uncharger
[08:49:00] Running Uncharger
[08:49:00] Running Uncharger
[08:49:00] Running Uncharger
[08:49:00] Running Uncharger
[08:49:00] Running Uncharger
[08:49:00] Running Uncharger
[08:49:00] Running Uncharger
[08:49:00] Running Uncharger
[08:49:00] Running Uncharger
[08:49:00] Running Uncharger
[08:49:00] Running Uncharger
[08:49:00] Running Uncharger
[08:49:

...

[08:49:05] WARNING: Proton(s) added/removed

[08:49:05] WARNING: Proton(s) added/removed

[08:49:05] WARNING: Omitted undefined stereo

[08:49:05] WARNING: Omitted undefined stereo

[08:49:05] WARNING: Omitted undefined stereo

[08:49:05] WARNING: Proton(s) added/removed

[08:49:05] WARNING: Charges were rearranged

[08:49:05] WARNING: Omitted undefined stereo

[08:49:05] WARNING: Charges were rearranged; Proton(s) added/removed

[08:49:05] WARNING: Proton(s) added/removed

[08:49:05] WARNING: Omitted undefined stereo

[08:49:05] WARNING: Omitted undefined stereo

[08:49:05] WARNING: Omitted undefined stereo

[08:49:05] WARNING: Charges were rearranged

[08:49:05] WARNING: Charges were rearranged

[08:49:05] WARNING: Omitted undefined stereo

[08:49:05] WARNING: Omitted undefined stereo

[08:49:05] WARNING: Omitted undefined stereo

[08:49:05] WARNING: Charges were rearranged

[08:49:05] WARNING: Proton(s) added/removed

[08:49:05] WARNING: Omitted undefined stereo

[08:49:05] WARNING:

[08:49:05] WARNING: Omitted undefined stereo

[08:49:05] WARNING: Omitted undefined stereo

[08:49:05] WARNING: Omitted undefined stereo

[08:49:05] WARNING: Omitted undefined stereo

[08:49:05] WARNING: Omitted undefined stereo

[08:49:05] WARNING: Proton(s) added/removed

[08:49:05] WARNING: Omitted undefined stereo

[08:49:05] WARNING: Omitted undefined stereo

[08:49:05] WARNING: Omitted undefined stereo

[08:49:05] WARNING: Omitted undefined stereo

[08:49:05] WARNING: Proton(s) added/removed

[08:49:05] WARNING: Proton(s) added/removed

[08:49:05] WARNING: Omitted undefined stereo

[08:49:05] WARNING: Omitted undefined stereo

[08:49:05] WARNING: Omitted undefined stereo

[08:49:05] WARNING: Charges were rearranged; Omitted undefined stereo

[08:49:05] WARNING: Omitted undefined stereo

[08:49:05] WARNING: Charges were rearranged

[08:49:05] WARNING: Charges were rearranged; Omitted undefined stereo

[08:49:05] WARNING: Charges were rearranged

[08:49:05] WARNING: Accepted unusua

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Proton(s) added/removed

[08:49:06] WARNING: Proton(s) added/removed

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Proton(s) added/removed

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Proton(s) added/removed

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Charges were rearranged

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Metal was disconnected; Proton(s) added/removed

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Omitted undefined stereo




FINAL SUMMARY

Corrected outcomes: 4,932 usable trials
Corrected_Outcome
PASS             3997
FAIL_EFFICACY     694
FAIL_SAFETY       222
FAIL_BOTH          19

Training (with features): 2,900 trials, 480 drugs
Wrong SMILES in features: 18 drugs

Reprocessing trial-level: 1704 rows, 540 drugs
Reprocessing drug-level: 568 rows
  With SMILES: 568
  With usable trials: 542
  With drug targets: 210
  With disease targets: 350
  Max disease target genes: 342

File sizes:
  trial-level: 13952 KB
  drug-level:  7561 KB


[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Proton(s) added/removed

[08:49:06] WARNING: Proton(s) added/removed



---
## 14. Unified Feature Computation from Raw Pipeline Outputs

Compute ALL biological features from raw gabe-run/run2/run3 pipeline outputs.
This replaces the ad-hoc scripts that built `STAR_complete.csv`.

**Raw data sources (local):**

| Run | Binding | Tissue | Network | Notes |
|-----|---------|--------|---------|-------|
| gabe-run  | 404 drugs | 404 drugs | — | Original run |
| gabe-run2 | 707 drugs | — | 1,390 files | Supplemental (underscored InChIKeys) |
| gabe-run3 | 493 drugs | 485 drugs | 425 drugs | New 550-drug run |
| **Total** | **~900 unique** | **~889** | **~900** | After dedup |

**Feature groups computed:**
1. Tissue interaction features (79) — from `biological_properties.tsv`
2. Binding specificity features (24) — from `drug_scores.tsv`
3. Toxicity binding features (23) — from `drug_scores.tsv` + safety target lists
4. Essential gene binding (14) — from `drug_scores.tsv` + essential gene list
5. Network enrichment features (36) — from `network_enrichment.tsv` + `network_interactions.tsv`
6. Drug network features (22) — from network enrichment (drug-level only)

Disease-target binding and DruMAP added separately.

In [22]:
# === 14a. Load ALL raw pipeline data from unified folder ===
#
# All pipeline outputs are consolidated in data/raw/pipeline_all/ with normalized
# filenames: {IK27}_{type}.tsv (IK27 with underscores). When new runs complete,
# just drop files into the right subdirectory and re-run this notebook.

import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('..').resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
MODELS_DIR = PROJECT_ROOT / 'data' / 'models'
SOURCES_DIR = PROJECT_ROOT / 'data' / 'sources'
MODELS_DIR.mkdir(exist_ok=True)

def normalize_ik(ik):
    return ik.replace('_', '-')

PIPELINE_DIR = RAW_DIR / 'pipeline_all'

# --- 1. Load all files from unified directory ---
import re

def extract_ik_from_unified(filename):
    m = re.match(r'^([A-Z0-9]{14}_[A-Z0-9]{10}_[A-Z0-9])_', filename)
    if m:
        return m.group(1).replace('_', '-')
    return None

bio_files = {}
for f in sorted((PIPELINE_DIR / 'tissue').glob('*_biological_properties.tsv')):
    ik = extract_ik_from_unified(f.name)
    if ik:
        bio_files[ik] = str(f)

score_files = {}
for f in sorted((PIPELINE_DIR / 'binding').glob('*_drug_scores.tsv')):
    ik = extract_ik_from_unified(f.name)
    if ik:
        score_files[ik] = str(f)

net_enrich_files = {}
for f in sorted((PIPELINE_DIR / 'network').glob('*_network_enrichment.tsv')):
    ik = extract_ik_from_unified(f.name)
    if ik:
        net_enrich_files[ik] = str(f)

net_interact_files = {}
for f in sorted((PIPELINE_DIR / 'network').glob('*_network_interactions.tsv')):
    ik = extract_ik_from_unified(f.name)
    if ik:
        net_interact_files[ik] = str(f)

print(f'Unified pipeline data loaded from {PIPELINE_DIR}:')
print(f'  Binding scores:     {len(score_files)} drugs')
print(f'  Tissue/bio props:   {len(bio_files)} drugs')
print(f'  Network enrichment: {len(net_enrich_files)} drugs')
print(f'  Network interactions: {len(net_interact_files)} drugs')

# Load corrected outcomes (needed for smart matching and ik_to_smiles)
co = pd.read_csv(SOURCES_DIR / '12_trials_corrected_outcomes.csv')

# --- Build ik_to_smiles mapping (needed by downstream feature cells) ---
from rdkit import Chem
from rdkit.Chem.inchi import MolToInchi, InchiToInchiKey

ik_to_smiles = {}
for smi in co['SMILES'].dropna().unique():
    mol = Chem.MolFromSmiles(str(smi))
    if mol:
        inch = MolToInchi(mol)
        if inch:
            ik = InchiToInchiKey(inch)
            ik_to_smiles[ik] = smi
# Also add pipeline IKs from score_files
for ik in score_files:
    if ik not in ik_to_smiles:
        ik_to_smiles[ik] = None  # placeholder

# Achiral stereo constant (InChIKey stereo layer for non-stereo molecules)
ACHIRAL_STEREO = 'UHFFFAOYSA'

# MANE transcript subset — the ~19K transcripts shared by run1/run2/run4.
# Run3 used the full 111K proteome, so its files must be filtered to this subset
# for comparable features. Use a run1 file (19,352 transcripts) as the reference.
MANE_TRANSCRIPTS = set()
_r1_path = RAW_DIR / 'gabe-run' / 'output' / 'drug-target' / 'binding'
_r1_files = list(_r1_path.glob('*_drug_scores.tsv')) if _r1_path.exists() else []
if _r1_files:
    _ref = pd.read_csv(str(_r1_files[0]), sep='\t')
    if 'Transcript' in _ref.columns:
        MANE_TRANSCRIPTS = set(_ref['Transcript'])
if MANE_TRANSCRIPTS:
    print(f'MANE transcript subset: {len(MANE_TRANSCRIPTS)} transcripts (from gabe-run reference)')
else:
    print('WARNING: Could not determine MANE transcript subset — features may not be normalized')


def build_pipeline_ik_index(score_files, bio_files, net_enrich_files):
    """Build full-IK and IK14→[full_IK] indexes from pipeline filenames."""
    from collections import defaultdict
    pipeline_full_iks = set()
    pipeline_ik14_to_full = defaultdict(list)
    for ik in set(list(score_files.keys()) + list(bio_files.keys()) + list(net_enrich_files.keys())):
        pipeline_full_iks.add(ik)
        pipeline_ik14_to_full[ik[:14]].append(ik)
    for k in pipeline_ik14_to_full:
        pipeline_ik14_to_full[k] = list(set(pipeline_ik14_to_full[k]))
    return pipeline_full_iks, dict(pipeline_ik14_to_full)

def build_blocked_ik14s(trial_smiles_list):
    """Find IK14s where multiple stereoisomers exist in trial data."""
    from collections import defaultdict
    ik14_stereos = defaultdict(set)
    for smi in trial_smiles_list:
        mol = Chem.MolFromSmiles(str(smi))
        if not mol: continue
        inch = rdkinchi.MolToInchi(mol)
        if not inch: continue
        ik = rdkinchi.InchiToInchiKey(inch)
        ik14_stereos[ik[:14]].add(ik[15:25])
    return {ik14 for ik14, stereos in ik14_stereos.items() if len(stereos) > 1}

def smart_match(trial_ik, pipeline_full_iks, pipeline_ik14_to_full, blocked_ik14s):
    """Match trial InChIKey to pipeline data with stereoisomer safety.
    
    Returns: (matched_pipeline_ik, match_type) or (None, reason)
    
    1. Full InChIKey match → best
    2. IK14 match, one side achiral, IK14 NOT blocked → acceptable  
    3. IK14 blocked (stereoisomer collision in trial data) → reject
    4. Both have stereo but differ → reject
    """
    if trial_ik in pipeline_full_iks:
        return trial_ik, 'full_ik'
    
    trial_ik14 = trial_ik[:14]
    trial_stereo = trial_ik[15:25]
    
    if trial_ik14 in blocked_ik14s:
        return None, 'blocked_stereoisomer'
    
    candidates = pipeline_ik14_to_full.get(trial_ik14, [])
    if not candidates:
        return None, 'no_ik14_match'
    
    trial_is_achiral = (trial_stereo == ACHIRAL_STEREO[:10])
    for cik in candidates:
        if trial_stereo == cik[15:25]:
            return cik, 'full_ik_via_ik14'
        if trial_is_achiral or (cik[15:25] == ACHIRAL_STEREO[:10]):
            return cik, 'ik14_achiral'
    
    return None, 'stereo_conflict'

# Build the indexes
pipeline_full_iks, pipeline_ik14_to_full = build_pipeline_ik_index(score_files, bio_files, net_enrich_files)
trial_smiles = co['SMILES'].dropna().unique()
blocked_ik14s = build_blocked_ik14s(trial_smiles)
print(f'Smart match index: {len(pipeline_full_iks)} pipeline IKs, {len(blocked_ik14s)} blocked IK14s (stereoisomers)')


Unified pipeline data loaded from <repo>/data/raw/pipeline_all:
  Binding scores:     1427 drugs
  Tissue/bio props:   1423 drugs
  Network enrichment: 1471 drugs
  Network interactions: 1471 drugs


[08:49:06] WARNING: Metal was disconnected; Proton(s) added/removed

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Charges were rearranged

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Charges were rearranged

[08:49:06] WARNING: Metal was disconnected; Proton(s) added/removed

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] bond type above 3 (17) is treated as unspecified!
[08:49:06] bond type above 3 (17) is treated as unspecified!
[08:49:06] ERROR: Unrecognized bond type: 0

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Charges were rearranged

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Charges were rearranged

[08:49:06] WARNING: Charges were rearranged; Omitted undefined stereo

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Omitted un

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Charges were rearranged

[08:49:06] WARNING: Charges were rearranged; Omitted undefined stereo

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Proton(s) added/removed

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Charges were rearranged

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Charges were rearranged

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WARNING: Omitted undefined stereo

[08:49:06] WA

Smart match index: 1536 pipeline IKs, 16 blocked IK14s (stereoisomers)


[08:49:07] WARNING: Omitted undefined stereo

[08:49:07] WARNING: Charges were rearranged

[08:49:07] WARNING: Omitted undefined stereo

[08:49:07] WARNING: Omitted undefined stereo

[08:49:07] WARNING: Omitted undefined stereo

[08:49:07] WARNING: Omitted undefined stereo

[08:49:07] WARNING: Omitted undefined stereo

[08:49:07] WARNING: Omitted undefined stereo

[08:49:07] WARNING: Omitted undefined stereo

[08:49:07] WARNING: Charges were rearranged

[08:49:07] WARNING: Charges were rearranged; Omitted undefined stereo

[08:49:07] WARNING: Omitted undefined stereo

[08:49:07] WARNING: Omitted undefined stereo

[08:49:07] WARNING: Omitted undefined stereo

[08:49:07] WARNING: Proton(s) added/removed

[08:49:07] WARNING: Omitted undefined stereo

[08:49:07] WARNING: Omitted undefined stereo

[08:49:07] WARNING: Charges were rearranged

[08:49:07] WARNING: Omitted undefined stereo

[08:49:07] WARNING: Charges were rearranged

[08:49:07] WARNING: Omitted undefined stereo

[08:49:07] WAR

In [23]:
# === 14b. Compute tissue interaction features (79 features) ===
#
# Source: biological_properties.tsv files (gabe-run + gabe-run3)
# These files contain pre-computed tissue × Binding interaction scores from the pipeline.
# Each file = one drug, one row with ~100 columns covering 14 tissue types.
# We just load and merge — the heavy computation was done in the pipeline.

tissue_rows = []
for ik, filepath in sorted(bio_files.items()):
    try:
        df = pd.read_csv(filepath, sep='\t')
        if len(df) > 0:
            row = df.iloc[0].to_dict()
            row['InChIKey'] = ik
            row['SMILES'] = ik_to_smiles.get(ik, None)
            tissue_rows.append(row)
    except Exception as e:
        pass  # Skip corrupt files

tissue_df = pd.DataFrame(tissue_rows)
# Drop the Drug_ID column (it's the InChIKey, already captured)
if 'Drug_ID' in tissue_df.columns:
    tissue_df = tissue_df.drop(columns=['Drug_ID'])

print(f'=== TISSUE INTERACTION FEATURES ===')
print(f'Drugs loaded: {len(tissue_df)}')
print(f'With SMILES:  {tissue_df["SMILES"].notna().sum()}')
print(f'Columns: {len(tissue_df.columns)}')
tissue_cols = [c for c in tissue_df.columns if c not in ('InChIKey', 'SMILES')]
print(f'Feature columns: {len(tissue_cols)}')
print(f'Sample: {tissue_cols[:5]}...')

# NULL organ interaction values mean ZERO binding (no targets above threshold),
# NOT missing data. The pipeline only writes a value when at least one target
# is above threshold; otherwise the column is absent / NaN. Imputing the
# median (~0.25 for drugs that do bind) falsely signals average binding.
# Fix: fill NaN → 0 for all max/min/mean_*_interaction and weighted_score_* columns.
_organ_null_zero = [c for c in tissue_df.columns if (
    any(c.startswith(p) for p in ('max_', 'min_', 'mean_')) and '_interaction' in c
) or c.startswith('weighted_score_')]
tissue_df[_organ_null_zero] = tissue_df[_organ_null_zero].fillna(0)
print(f'Filled NaN→0 in {len(_organ_null_zero)} organ binding columns')

tissue_df.to_csv(MODELS_DIR / 'tissue_interaction_features_v5.csv', index=False)
print(f'Saved: tissue_interaction_features_v5.csv')

=== TISSUE INTERACTION FEATURES ===
Drugs loaded: 1423
With SMILES:  825
Columns: 125
Feature columns: 123
Sample: ['n_enst_heart_above_target_interaction', 'n_enst_gasrto_intestin_above_target_interaction', 'n_enst_female_tissues_above_target_interaction', 'n_enst_male_tissues_above_target_interaction', 'n_enst_liver_above_target_interaction']...
Saved: tissue_interaction_features_v5.csv


In [24]:
# === 14c. Compute Binding specificity + toxicity + essential gene features ===
#
# All three feature groups read from the same drug_scores.tsv files.
# We load each file once, then compute all three in a single pass.
#
# Binding specificity (24 features): binding distribution stats across all transcripts
# Toxicity binding (23 features): binding to known safety pharmacology targets
# Essential gene binding (14 features): binding to housekeeping/vital genes
#
# BIND_THRESHOLD = 0.5 (Binding score above which binding is considered significant)

import sys
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

BIND_THRESHOLD = 0.5

# Load target gene lists from the compute scripts
from compute_essential_gene_features import ESSENTIAL_GENES, VITAL_ORGAN_GENES
from compute_safety_features import ANTI_TARGETS
ANTITARGET_GENES = list(ANTI_TARGETS.keys())

# Load gene → ENST mapping
gene_enst_df = pd.read_csv(SOURCES_DIR / 'pipeline_gene_to_enst.csv')
gene_to_enst = defaultdict(list)
for _, row in gene_enst_df.iterrows():
    if pd.notna(row['enst_id']):
        gene_to_enst[row['gene_symbol']].append(row['enst_id'])

# Build ENST sets for each target group
essential_ensts = set()
for gene in ESSENTIAL_GENES:
    essential_ensts.update(gene_to_enst.get(gene, []))

antitarget_ensts = set()
for gene in ANTITARGET_GENES:
    antitarget_ensts.update(gene_to_enst.get(gene, []))

vital_organ_ensts = {}  # organ → set of ENST
for organ, genes in VITAL_ORGAN_GENES.items():
    vital_organ_ensts[organ] = set()
    for gene in genes:
        vital_organ_ensts[organ].update(gene_to_enst.get(gene, []))

print(f'Target sets: essential={len(essential_ensts)} ENST, '
      f'antitarget={len(antitarget_ensts)} ENST, '
      f'organs={", ".join(f"{k}={len(v)}" for k,v in vital_organ_ensts.items())}')

# --- Define MANE subset transcripts for cross-run consistency ---
# gabe-run/run2 used 19,352 transcripts (MANE subset)
# gabe-run3 used 111,037 transcripts (full MANE)
# To ensure features are comparable across runs, filter gabe-run3 scores
# to the same 19,352 MANE subset transcripts.
_mane_ref_file = list(bio_files.values())[0] if bio_files else list(score_files.values())[0]
# Get MANE transcript set from a gabe-run/run2 file
# Use the MANE_TRANSCRIPTS set from cell 36 (loaded from gabe-run reference, ~19K)
if MANE_TRANSCRIPTS and len(MANE_TRANSCRIPTS) > 0:
    print(f'MANE subset: {len(MANE_TRANSCRIPTS)} transcripts (from cell 36)')
else:
    # Fallback: find a 19K-transcript file in the unified folder
    for ik, path in score_files.items():
        _ref = pd.read_csv(path, sep='\t')
        if len(_ref) < 25000:  # 19K files, not 111K
            MANE_TRANSCRIPTS = set(_ref['Transcript'])
            print(f'MANE subset: {len(MANE_TRANSCRIPTS)} transcripts (from {os.path.basename(path)[:30]})')
            break
    else:
        MANE_TRANSCRIPTS = None
        print('WARNING: No MANE reference found')

# --- Process all drug_scores files ---
binding_rows = []
tox_rows = []
essential_rows = []

for i, (ik, filepath) in enumerate(sorted(score_files.items())):
    smi = ik_to_smiles.get(ik)
    try:
        df = pd.read_csv(filepath, sep='\t')
        # Filter to MANE subset if this is a gabe-run3 file (111K → 19K transcripts)
        if MANE_TRANSCRIPTS and len(df) > 25000:  # run3 files have >100K rows
            df = df[df['Transcript'].isin(MANE_TRANSCRIPTS)]
        scores = df['Score'].values
        transcripts = df['Transcript'].values
        
        if len(scores) == 0:
            continue
        
        # --- Binding specificity features ---
        n_total = len(scores)
        bound = scores > BIND_THRESHOLD
        n_bound = bound.sum()
        
        binding_rows.append({
            'InChIKey': ik, 'SMILES': smi,
            'binding_n_total_transcripts': n_total,
            'binding_n_above_50': (scores > 0.5).sum(),
            'binding_n_above_70': (scores > 0.7).sum(),
            'binding_n_above_80': (scores > 0.8).sum(),
            'binding_n_above_90': (scores > 0.9).sum(),
            'binding_frac_above_50': (scores > 0.5).mean(),
            'binding_score_max': scores.max(),
            'binding_score_mean': scores.mean(),
            'binding_score_median': np.median(scores),
            'binding_score_std': scores.std(),
            'binding_score_p90': np.percentile(scores, 90),
            'binding_score_p95': np.percentile(scores, 95),
            'binding_score_p99': np.percentile(scores, 99),
            'binding_score_iqr': np.percentile(scores, 75) - np.percentile(scores, 25),
            'binding_score_skew': float(pd.Series(scores).skew()),
            'binding_score_kurtosis': float(pd.Series(scores).kurtosis()),
            'binding_score_entropy': float(-np.sum(np.clip(scores, 1e-10, 1) * np.log2(np.clip(scores, 1e-10, 1))) / max(n_total, 1)),
            'binding_top10_concentration': scores[np.argsort(scores)[-10:]].sum() / max(scores.sum(), 1e-10),
            'binding_top50_concentration': scores[np.argsort(scores)[-50:]].sum() / max(scores.sum(), 1e-10) if n_total >= 50 else 1.0,
            'binding_drug_bind_max': scores.max(),
            'binding_drug_bind_mean': scores[bound].mean() if n_bound > 0 else 0,
            'binding_drug_frac_bound': n_bound / n_total if n_total > 0 else 0,
            'binding_drug_n_bound': n_bound,
        })
        
        # --- Toxicity binding features ---
        # Score lookup by transcript
        score_dict = dict(zip(transcripts, scores))
        
        anti_scores = [score_dict.get(e, 0) for e in antitarget_ensts if e in score_dict]
        tox_row = {'InChIKey': ik, 'SMILES': smi}
        
        if anti_scores:
            anti_arr = np.array(anti_scores)
            tox_row['tox_antitarget_max_bind'] = anti_arr.max()
            tox_row['tox_antitarget_mean_bind'] = anti_arr.mean()
            tox_row['tox_antitarget_n_bound'] = (anti_arr > BIND_THRESHOLD).sum()
            tox_row['tox_antitarget_burden'] = anti_arr[anti_arr > BIND_THRESHOLD].sum()
        else:
            tox_row['tox_antitarget_max_bind'] = 0
            tox_row['tox_antitarget_mean_bind'] = 0
            tox_row['tox_antitarget_n_bound'] = 0
            tox_row['tox_antitarget_burden'] = 0
        
        # Per-organ toxicity
        for organ, ensts in [('cardiac', vital_organ_ensts.get('heart', set())),
                              ('hepatic', vital_organ_ensts.get('liver', set())),
                              ('renal', vital_organ_ensts.get('kidney', set()))]:
            organ_scores = [score_dict.get(e, 0) for e in ensts if e in score_dict]
            if organ_scores:
                oarr = np.array(organ_scores)
                tox_row[f'tox_{organ}_max_bind'] = oarr.max()
                tox_row[f'tox_{organ}_mean_bind'] = oarr.mean()
                tox_row[f'tox_{organ}_n_bound'] = (oarr > BIND_THRESHOLD).sum()
                tox_row[f'tox_{organ}_burden'] = oarr[oarr > BIND_THRESHOLD].sum()
            else:
                for suffix in ['max_bind', 'mean_bind', 'n_bound', 'burden']:
                    tox_row[f'tox_{organ}_{suffix}'] = 0
        
        # DNA damage targets (from safety genes)
        dna_genes = ['TP53', 'BRCA1', 'BRCA2', 'ATM', 'ATR', 'CHEK1', 'CHEK2', 'RAD51', 'PARP1']
        dna_ensts = set()
        for g in dna_genes:
            dna_ensts.update(gene_to_enst.get(g, []))
        dna_scores = [score_dict.get(e, 0) for e in dna_ensts if e in score_dict]
        if dna_scores:
            darr = np.array(dna_scores)
            tox_row['tox_dna_damage_max_bind'] = darr.max()
            tox_row['tox_dna_damage_n_bound'] = (darr > BIND_THRESHOLD).sum()
            tox_row['tox_dna_damage_burden'] = darr[darr > BIND_THRESHOLD].sum()
        else:
            tox_row['tox_dna_damage_max_bind'] = 0
            tox_row['tox_dna_damage_n_bound'] = 0
            tox_row['tox_dna_damage_burden'] = 0
        
        # Overall selectivity ratio
        if n_bound > 0 and anti_scores:
            on_target = scores[bound].mean()
            off_target = np.mean(anti_scores) if anti_scores else 0
            tox_row['tox_vs_overall_ratio'] = off_target / on_target if on_target > 0 else 0
        else:
            tox_row['tox_vs_overall_ratio'] = 0
        
        tox_rows.append(tox_row)
        
        # --- Essential gene features ---
        ess_scores = [score_dict.get(e, 0) for e in essential_ensts if e in score_dict]
        ess_row = {'InChIKey': ik, 'SMILES': smi}
        
        if ess_scores:
            earr = np.array(ess_scores)
            ess_bound = earr > BIND_THRESHOLD
            ess_row['essential_n_bound'] = ess_bound.sum()
            ess_row['essential_frac_bound'] = ess_bound.sum() / max(n_bound, 1)
            ess_row['essential_bind_burden'] = earr[ess_bound].sum()
            ess_row['essential_bind_mean'] = earr[ess_bound].mean() if ess_bound.sum() > 0 else 0
            ess_row['essential_bind_max'] = earr.max()
        else:
            ess_row['essential_n_bound'] = 0
            ess_row['essential_frac_bound'] = 0
            ess_row['essential_bind_burden'] = 0
            ess_row['essential_bind_mean'] = 0
            ess_row['essential_bind_max'] = 0
        
        # Per-organ vital gene burden
        for organ, ensts in vital_organ_ensts.items():
            organ_scores = [score_dict.get(e, 0) for e in ensts if e in score_dict]
            if organ_scores:
                oarr = np.array(organ_scores)
                ess_row[f'{organ}_vital_bind_burden'] = oarr[oarr > BIND_THRESHOLD].sum()
            else:
                ess_row[f'{organ}_vital_bind_burden'] = 0
        
        # Organ risk summary
        organ_burdens = [ess_row.get(f'{o}_vital_bind_burden', 0) for o in vital_organ_ensts]
        ess_row['max_organ_vital_burden'] = max(organ_burdens) if organ_burdens else 0
        ess_row['organ_burden_spread'] = max(organ_burdens) - min(organ_burdens) if organ_burdens else 0
        nonzero = [b for b in organ_burdens if b > 0]
        median_nz = np.median(nonzero) if nonzero else 0
        ess_row['n_organs_at_risk'] = sum(1 for b in organ_burdens if b > median_nz) if median_nz > 0 else 0
        
        essential_rows.append(ess_row)
        
    except Exception as e:
        if i < 3:  # Only print first few errors
            print(f'  Error processing {ik}: {e}')

    if (i + 1) % 200 == 0:
        print(f'  Processed {i+1}/{len(score_files)} drugs...')

binding_df = pd.DataFrame(binding_rows)
tox_df = pd.DataFrame(tox_rows)
essential_df = pd.DataFrame(essential_rows)

print(f'\n=== BINDING SPECIFICITY FEATURES ===')
print(f'Drugs: {len(binding_df)}, Features: {len([c for c in binding_df.columns if c.startswith("binding_")])}')

print(f'\n=== TOXICITY BINDING FEATURES ===')
print(f'Drugs: {len(tox_df)}, Features: {len([c for c in tox_df.columns if c.startswith("tox_")])}')

print(f'\n=== ESSENTIAL GENE FEATURES ===')
print(f'Drugs: {len(essential_df)}, Features: {len([c for c in essential_df.columns if c not in ("InChIKey","SMILES")])}')

binding_df.to_csv(MODELS_DIR / 'binding_specificity_features_v5.csv', index=False)
tox_df.to_csv(MODELS_DIR / 'toxicity_binding_features_v5.csv', index=False)
essential_df.to_csv(MODELS_DIR / 'essential_gene_features_v5.csv', index=False)
print(f'\nSaved: binding_specificity_features_v5.csv, toxicity_binding_features_v5.csv, essential_gene_features_v5.csv')

Target sets: essential=1758 ENST, antitarget=325 ENST, organs=heart=960, brain=874, liver=830, kidney=684
MANE subset: 19352 transcripts (from AAGBDNPEEQZKHW_UHFFFAOYSA_N_dr)


  Processed 200/1427 drugs...


  Processed 400/1427 drugs...


  Processed 600/1427 drugs...


  Processed 800/1427 drugs...


  Processed 1000/1427 drugs...


  Processed 1200/1427 drugs...


  Processed 1400/1427 drugs...



=== BINDING SPECIFICITY FEATURES ===
Drugs: 1427, Features: 23

=== TOXICITY BINDING FEATURES ===
Drugs: 1427, Features: 20

=== ESSENTIAL GENE FEATURES ===
Drugs: 1427, Features: 12

Saved: binding_specificity_features_v5.csv, toxicity_binding_features_v5.csv, essential_gene_features_v5.csv


In [25]:
# === 14d. Network enrichment features (36 features) ===
#
# Comprehensive network analysis from STRING-DB enrichment and PPI data.
# Three main signal categories:
#   1. Disease-match (efficacy): drug targets overlap with disease pathways
#   2. Off-target spread (safety): drug affects pathways beyond disease biology
#   3. PPI reachability: drug targets can reach disease genes via protein network
#
# Requires: network enrichment/interaction files in pipeline_all/network/
# Disease targets from OpenTargets (cached in data/cache/disease_targets_cache.json)

import os, re, json as _json
import networkx as nx
import requests, time

PIPELINE_DIR_NF = RAW_DIR / 'pipeline_all'
FDR_THRESHOLD = 0.05

# === Load disease targets from OpenTargets cache ===
_dt_cache_path = PROJECT_ROOT / 'data' / 'cache' / 'disease_targets_cache.json'
with open(_dt_cache_path) as _f:
    _dt_cache = _json.load(_f)

# Build disease name → gene symbols mapping
_disease_to_genes = {}
for k, v in _dt_cache.items():
    if k.startswith('targets:') and isinstance(v, dict) and v.get('targets'):
        dname = v.get('disease_name', '').lower().strip()
        symbols = {t['symbol'] for t in v['targets'][:50]}  # top 50 by OpenTargets score
        if dname and symbols:
            _disease_to_genes[dname] = symbols

print(f'Disease targets loaded: {len(_disease_to_genes)} diseases')

# Gene symbol → ENST mapping
_gene_enst = pd.read_csv(SOURCES_DIR / 'pipeline_gene_to_enst.csv')
_gene_to_enst = {}
for _, row in _gene_enst.iterrows():
    if pd.notna(row.get('enst_id')):
        _gene_to_enst.setdefault(row['gene_symbol'], set()).add(row['enst_id'])

# Synonym map: common trial disease names → OpenTargets canonical names.
# 35% of training diseases get zero disease-pathway features because terms
# like "COPD" or "Depression" don't overlap ≥2 words with OT canonical names.
_DISEASE_SYNONYMS = {
    "copd": "chronic obstructive pulmonary disease",
    "depression": "major depressive disorder",
    "major depression": "major depressive disorder",
    "diabetes": "diabetes mellitus",
    "diabetes mellitus": "diabetes mellitus",
    "migraine": "migraine disorder",
    "covid19": "covid-19",
    "covid-19": "covid-19",
    "sars-cov infection": "covid-19",
    "sars-cov-2": "covid-19",
    "atopic dermatitis": "atopic eczema",
    "erosive esophagitis": "gastroesophageal reflux disease",
    "pancreatic cancer": "pancreatic adenocarcinoma",
    "postoperative pain": "pain",
    "gastric ulcers; duodenal ulcers": "peptic ulcer disease",
    "duodenal ulcers; gastric ulcers": "peptic ulcer disease",
    "asthma; wheezing": "asthma",
    "plaque psoriasis": "psoriasis",
}

def _get_disease_target_ensts(disease_text):
    """Get ENST IDs for disease targets, matching trial disease text to OpenTargets."""
    if pd.isna(disease_text) or not str(disease_text).strip():
        return set()
    dt = str(disease_text).lower().strip()
    # Apply synonym normalization before any matching
    dt_lookup = _DISEASE_SYNONYMS.get(dt, dt)
    # Try exact match (with and without synonym normalization), then partial
    if dt_lookup in _disease_to_genes:
        symbols = _disease_to_genes[dt_lookup]
    elif dt in _disease_to_genes:
        symbols = _disease_to_genes[dt]
    else:
        # Partial match: find the best overlapping disease name
        best_match = None
        best_overlap = 0
        dt_words = set(dt_lookup.split())
        for dname, syms in _disease_to_genes.items():
            overlap = len(dt_words & set(dname.split()))
            if overlap > best_overlap:
                best_overlap = overlap
                best_match = dname
        symbols = _disease_to_genes.get(best_match, set()) if best_overlap >= 2 else set()
    
    # Convert symbols to ENSTs
    ensts = set()
    for sym in symbols:
        ensts.update(_gene_to_enst.get(sym, set()))
    return ensts

# === STRING-DB Fallback ===
_sig_int_files = {}
for f in sorted((PIPELINE_DIR_NF / 'binding').glob('*_significant_interactions.tsv')):
    m = re.match(r'^([A-Z0-9]{14}_[A-Z0-9]{10}_[A-Z0-9])_', f.name)
    if m: _sig_int_files[m.group(1).replace('_', '-')] = str(f)

_net_iks = set()
for f in (PIPELINE_DIR_NF / 'network').glob('*_network_enrichment.tsv'):
    m = re.match(r'^([A-Z0-9]{14}_[A-Z0-9]{10}_[A-Z0-9])_', f.name)
    if m: _net_iks.add(m.group(1).replace('_', '-'))

_need_string = {ik: path for ik, path in _sig_int_files.items() if ik not in _net_iks}
if _need_string:
    print(f'STRING-DB fallback: {len(_need_string)} drugs need network queries')
    _s_ok = 0
    for ik, path in sorted(_need_string.items()):
        try:
            _df = pd.read_csv(path, sep='\t')
            if len(_df) == 0: continue
            _df = _df.sort_values('binding_score', ascending=False).drop_duplicates('transcipt')
            if len(_df) > 450: _df = _df.iloc[:450]
            _above = _df[_df['binding_score'] > 0.6]
            if len(_above) == 0: continue
            _plist = '%0d'.join(_above['transcipt'].unique().tolist())
            _r1 = requests.get(f'https://string-db.org/api/tsv/enrichment?identifiers={_plist}&species=9606', timeout=60)
            time.sleep(0.5)
            _r2 = requests.get(f'https://string-db.org/api/tsv/network?identifiers={_plist}&species=9606', timeout=60)
            time.sleep(0.5)
            _ik_u = ik.replace('-', '_')
            for _r, _suffix in [(_r1, 'network_enrichment'), (_r2, 'network_interactions')]:
                _lines = _r.text.split('\n')
                _data = [l.split('\t') for l in _lines]
                if len(_data) > 1:
                    _out = pd.DataFrame(_data[1:-1] if _data[-1] == [''] else _data[1:], columns=_data[0])
                    _out.to_csv(PIPELINE_DIR_NF / 'network' / f'{_ik_u}_{_suffix}.tsv', sep='\t', index=False)
            _s_ok += 1
        except: pass
    if _s_ok: print(f'  STRING-DB fallback completed: {_s_ok} drugs')

# === Reload network files after fallback ===
net_enrich_files = {}
net_interact_files = {}
for f in sorted((PIPELINE_DIR_NF / 'network').glob('*_network_enrichment.tsv')):
    m = re.match(r'^([A-Z0-9]{14}_[A-Z0-9]{10}_[A-Z0-9])_', f.name)
    if m: net_enrich_files[m.group(1).replace('_', '-')] = str(f)
for f in sorted((PIPELINE_DIR_NF / 'network').glob('*_network_interactions.tsv')):
    m = re.match(r'^([A-Z0-9]{14}_[A-Z0-9]{10}_[A-Z0-9])_', f.name)
    if m: net_interact_files[m.group(1).replace('_', '-')] = str(f)
print(f'Network files: {len(net_enrich_files)} enrichment, {len(net_interact_files)} interactions')

# === Feature extraction (full v3 logic) ===

def _build_enst_to_symbol(enrich_df):
    mapping = {}
    if enrich_df is None or 'inputGenes' not in enrich_df.columns:
        return mapping
    for _, row in enrich_df.iterrows():
        gs = row.get('inputGenes', '')
        ns = row.get('preferredNames', '')
        if pd.notna(gs) and pd.notna(ns):
            for g, n in zip(str(gs).split(','), str(ns).split(',')):
                mapping[g.strip()] = n.strip()
    return mapping

def _build_ppi_graph(interact_df):
    G = nx.Graph()
    if interact_df is None: return G
    for _, row in interact_df.iterrows():
        a, b = row.get('preferredName_A'), row.get('preferredName_B')
        if pd.notna(a) and pd.notna(b):
            G.add_edge(str(a), str(b),
                       score=float(row.get('score', 0)),
                       escore=float(row.get('escore', 0)),
                       dscore=float(row.get('dscore', 0)),
                       tscore=float(row.get('tscore', 0)))
    return G

MECHANISM_KEYWORDS = {
    'dna_damage': ['dna repair', 'dna replication', 'dna damage', 'nucleotide excision',
                   'double-strand break', 'recombinational repair', 'mismatch repair'],
    'cell_cycle': ['cell cycle', 'mitotic', 'mitosis', 'cell division', 'spindle',
                   'chromosome segregation'],
    'epigenetic': ['histone', 'chromatin', 'nucleosome', 'deacetylase',
                   'acetyltransferase', 'methyltransferase', 'chromatin remodeling'],
    'immune': ['immune', 'toll-like', 'defense response', 'inflammatory', 'cytokine',
               'interferon', 'antigen', 'nf-kappa', 'innate immune'],
    'apoptosis': ['apoptosis', 'apoptotic', 'programmed cell death', 'caspase'],
}

def extract_network_features(enrich_path, interact_path, disease_ensts):
    """Extract all 36 network features for one drug-disease pair."""
    feats = {}
    try:
        enrich_df = pd.read_csv(enrich_path, sep='\t')
        interact_df = pd.read_csv(interact_path, sep='\t') if interact_path and os.path.exists(interact_path) else None
    except: return None
    
    if len(enrich_df) == 0: return None
    
    # Parse significant pathways
    fdr_col = 'fdr' if 'fdr' in enrich_df.columns else 'p_value'
    if fdr_col not in enrich_df.columns: return None
    enrich_df[fdr_col] = pd.to_numeric(enrich_df[fdr_col], errors='coerce')
    sig_df = enrich_df[enrich_df[fdr_col] < FDR_THRESHOLD]
    
    enst_to_sym = _build_enst_to_symbol(enrich_df)
    disease_symbols = {enst_to_sym.get(e, '') for e in disease_ensts} - {''}
    n_disease = len(disease_ensts)
    has_disease = n_disease > 0
    
    # Pathway gene sets
    pathway_gene_sets = []
    pathway_fdrs = []
    all_pathway_genes = set()
    for _, row in sig_df.iterrows():
        gs = row.get('inputGenes', '')
        if pd.notna(gs) and str(gs).strip():
            genes = set(str(gs).split(','))
            pathway_gene_sets.append(genes)
            pathway_fdrs.append(float(row[fdr_col]))
            all_pathway_genes.update(genes)
    n_pathways = len(pathway_gene_sets)
    
    # 1. PATHWAY-DISEASE MATCH
    if has_disease and n_pathways > 0:
        disease_in_pathways = disease_ensts & all_pathway_genes
        feats['net_n_disease_genes_in_pathways'] = len(disease_in_pathways)
        feats['net_frac_disease_covered'] = len(disease_in_pathways) / n_disease
        pw_with_disease = [i for i, pg in enumerate(pathway_gene_sets) if pg & disease_ensts]
        feats['net_n_pathways_with_disease'] = len(pw_with_disease)
        feats['net_frac_pathways_with_disease'] = len(pw_with_disease) / n_pathways
        if pw_with_disease:
            ds = [-np.log10(max(pathway_fdrs[i], 1e-300)) for i in pw_with_disease]
            feats['net_disease_pathway_score_max'] = max(ds)
            feats['net_disease_pathway_score_mean'] = np.mean(ds)
        else:
            feats['net_disease_pathway_score_max'] = 0.0
            feats['net_disease_pathway_score_mean'] = 0.0
        feats['net_has_disease_pathway_overlap'] = int(len(pw_with_disease) > 0)
    else:
        for k in ['net_n_disease_genes_in_pathways', 'net_frac_disease_covered',
                   'net_n_pathways_with_disease', 'net_frac_pathways_with_disease',
                   'net_disease_pathway_score_max', 'net_disease_pathway_score_mean',
                   'net_has_disease_pathway_overlap']:
            feats[k] = 0
    
    # 2. PPI REACHABILITY
    G = _build_ppi_graph(interact_df)
    ppi_nodes = set(G.nodes())
    disease_in_ppi = disease_symbols & ppi_nodes
    feats['net_n_disease_in_ppi'] = len(disease_in_ppi)
    feats['net_frac_disease_in_ppi'] = len(disease_in_ppi) / n_disease if has_disease else 0
    feats['net_n_unique_ppi_proteins'] = len(ppi_nodes)
    
    # 3. OFF-TARGET SPREAD
    feats['net_n_enriched_total'] = n_pathways
    if n_pathways > 0:
        n_off = n_pathways - len([pg for pg in pathway_gene_sets if pg & disease_ensts]) if has_disease else n_pathways
        feats['net_n_off_target_pathways'] = n_off
        feats['net_frac_off_target_pathways'] = n_off / n_pathways
        feats['net_spread_ratio'] = len(ppi_nodes) / max(1, n_pathways)
        if len(ppi_nodes) > 1:
            feats['net_network_density'] = len(G.edges()) / (len(ppi_nodes) * (len(ppi_nodes)-1) / 2)
        else:
            feats['net_network_density'] = 0
    else:
        feats['net_n_off_target_pathways'] = 0
        feats['net_frac_off_target_pathways'] = 0
        feats['net_spread_ratio'] = 0
        feats['net_network_density'] = 0
    
    # 4. ENRICHMENT SUMMARY
    if n_pathways > 0:
        scores = -np.log10(sig_df[fdr_col].clip(lower=1e-300))
        feats['net_max_enrichment_score'] = float(scores.max())
        feats['net_min_pvalue'] = float(sig_df[fdr_col].min())
        feats['net_mean_pvalue'] = float(sig_df[fdr_col].mean())
        if 'category' in sig_df.columns:
            feats['net_enrichment_diversity'] = sig_df['category'].nunique()
        else:
            feats['net_enrichment_diversity'] = 0
    else:
        feats['net_max_enrichment_score'] = 0
        feats['net_min_pvalue'] = 1.0
        feats['net_mean_pvalue'] = 1.0
        feats['net_enrichment_diversity'] = 0
    
    # 5. MECHANISM TYPE
    if n_pathways > 0 and 'description' in sig_df.columns:
        descriptions = sig_df['description'].dropna().str.lower().tolist()
        for mech, keywords in MECHANISM_KEYWORDS.items():
            hits = sum(1 for d in descriptions if any(k in d for k in keywords))
            feats[f'net_mech_{mech}_n'] = hits
            feats[f'net_mech_{mech}_frac'] = hits / n_pathways
    else:
        for mech in MECHANISM_KEYWORDS:
            feats[f'net_mech_{mech}_n'] = 0
            feats[f'net_mech_{mech}_frac'] = 0
    
    return feats

# === Process all drugs ===
print(f'Computing network features for {len(net_enrich_files)} drugs...')

# Build trial disease mapping: feature_IK → disease text (for disease target lookup)
# We need to compute per-trial features because disease targets are trial-specific
_trial_diseases = {}
for _, row in co.iterrows():
    smi = row.get('SMILES')
    disease = row.get('Disease', '')
    if pd.notna(smi) and pd.notna(disease):
        _trial_diseases.setdefault(str(smi), set()).add(str(disease))

net_rows = []
n_processed = 0
for ik in sorted(net_enrich_files.keys()):
    enrich_path = net_enrich_files[ik]
    interact_path = net_interact_files.get(ik)
    smi = ik_to_smiles.get(ik)
    
    # Get disease targets for this drug's trials
    diseases = _trial_diseases.get(smi, set()) if smi else set()
    # Merge all disease targets across trials for this drug
    all_disease_ensts = set()
    for d in diseases:
        all_disease_ensts.update(_get_disease_target_ensts(d))
    
    feats = extract_network_features(enrich_path, interact_path, all_disease_ensts)
    if feats:
        feats['InChIKey'] = ik
        feats['SMILES'] = smi
        net_rows.append(feats)
        n_processed += 1
    
    if n_processed % 200 == 0 and n_processed > 0:
        print(f'  Processed {n_processed} drugs...')

net_df = pd.DataFrame(net_rows)
print(f'Computed: {n_processed} drugs, {len(net_df.columns)-2} features')

# Remap IKs to match trial feature_IKs
from rdkit import Chem as _C
from rdkit.Chem.inchi import MolToInchi as _MI, InchiToInchiKey as _IK
_trial_ik14_map = {}
for smi in co['SMILES'].dropna().unique():
    mol = _C.MolFromSmiles(str(smi))
    if mol:
        inch = _MI(mol)
        if inch:
            full_ik = _IK(inch)
            _trial_ik14_map[full_ik[:14]] = full_ik

_remapped = 0
for idx, row in net_df.iterrows():
    ik = row['InChIKey']
    if pd.isna(ik): continue
    ik14 = str(ik)[:14]
    if ik14 in _trial_ik14_map:
        trial_ik = _trial_ik14_map[ik14]
        if trial_ik != ik:
            net_df.at[idx, 'InChIKey'] = trial_ik
            _remapped += 1
if _remapped:
    print(f'Remapped {_remapped} network IKs to match trial feature_IKs')

net_df.to_csv(MODELS_DIR / 'network_enrichment_features_v5.csv', index=False)
print(f'Saved: network_enrichment_features_v5.csv ({len(net_df)} drugs, {len(net_df.columns)-2} features)')


Disease targets loaded: 1285 diseases


Network files: 1471 enrichment, 1471 interactions
Computing network features for 1471 drugs...


  Processed 200 drugs...


  Processed 400 drugs...


  Processed 600 drugs...


...

  Processed 1200 drugs...


  Processed 1400 drugs...


Computed: 1434 drugs, 29 features


[08:50:36] WARNING: Metal was disconnected; Proton(s) added/removed

[08:50:36] WARNING: Omitted undefined stereo

[08:50:36] WARNING: Omitted undefined stereo

[08:50:36] WARNING: Charges were rearranged

[08:50:36] WARNING: Omitted undefined stereo

[08:50:36] WARNING: Charges were rearranged

[08:50:36] WARNING: Metal was disconnected; Proton(s) added/removed

[08:50:36] WARNING: Omitted undefined stereo

[08:50:36] bond type above 3 (17) is treated as unspecified!
[08:50:36] bond type above 3 (17) is treated as unspecified!
[08:50:36] ERROR: Unrecognized bond type: 0

[08:50:36] WARNING: Omitted undefined stereo

[08:50:36] WARNING: Charges were rearranged

[08:50:36] WARNING: Omitted undefined stereo

[08:50:36] WARNING: Charges were rearranged

[08:50:36] WARNING: Charges were rearranged; Omitted undefined stereo

[08:50:36] WARNING: Omitted undefined stereo

[08:50:36] WARNING: Omitted undefined stereo

[08:50:36] WARNING: Omitted undefined stereo

[08:50:36] WARNING: Omitted un

Remapped 69 network IKs to match trial feature_IKs
Saved: network_enrichment_features_v5.csv (1434 drugs, 29 features)


In [26]:
# === 14e. Merge all features into unified training dataset ===
#
# Simple approach: features are keyed by pipeline InChIKey (from filenames).
# Trial data has SMILES. We build a SMILES → pipeline InChIKey mapping
# from the ik_to_smiles dict (built in 14a), then join.
#
# IMPORTANT: Uses smart_match to handle stereoisomers (CLAUDE.md rule 7).

from rdkit import Chem
from rdkit.Chem import inchi as rdkinchi
import warnings
warnings.filterwarnings('ignore')

# Load corrected outcomes
co = pd.read_csv(SOURCES_DIR / '12_trials_corrected_outcomes.csv')
usable = co[co['Corrected_Outcome'].isin(['PASS', 'FAIL_SAFETY', 'FAIL_EFFICACY', 'FAIL_BOTH'])].copy()

# Map trial SMILES → pipeline InChIKey via smart_match
# smart_match, pipeline_full_iks, pipeline_ik14_to_full, blocked_ik14s from cell 14a
def get_pipeline_ik_for_trial(smi):
    if not smi or pd.isna(smi): return None
    mol = Chem.MolFromSmiles(str(smi))
    if not mol: return None
    inch = rdkinchi.MolToInchi(mol)
    if not inch: return None
    trial_ik = rdkinchi.InchiToInchiKey(inch)
    matched_ik, _ = smart_match(trial_ik, pipeline_full_iks, pipeline_ik14_to_full, blocked_ik14s)
    # CRITICAL: strip any -drug suffix from gabe-run2 filenames
    if matched_ik and len(matched_ik) > 27:
        matched_ik = matched_ik[:27]
    return matched_ik

usable['feature_IK'] = usable['SMILES'].apply(get_pipeline_ik_for_trial)
n_matched = usable['feature_IK'].notna().sum()
print(f'Trial drugs matched to pipeline: {n_matched}/{usable["SMILES"].notna().sum()} trials, '
      f'{usable["feature_IK"].nunique()} unique drugs')

# Recover IK mismatches: some training drugs have pipeline files under different IKs
# (e.g., pre-neutralization IK vs post-neutralization IK, same molecule)
import json as _json
_recovery_path = SOURCES_DIR / 'feature_ik_recovery_map.json'
if _recovery_path.exists():
    with open(_recovery_path) as _f:
        _recovery = _json.load(_f)
    _n_recovered = 0
    for _group, _smi_to_ik in _recovery.items():
        for idx, row in usable[usable['feature_IK'].isna()].iterrows():
            if row['SMILES'] in _smi_to_ik:
                usable.at[idx, 'feature_IK'] = _smi_to_ik[row['SMILES']]
                _n_recovered += 1
    if _n_recovered > 0:
        n_matched = usable['feature_IK'].notna().sum()
        print(f'IK recovery: +{_n_recovered} trial rows recovered')
        print(f'Trial drugs matched after recovery: {n_matched}/{usable["SMILES"].notna().sum()} trials, '
              f'{usable["feature_IK"].nunique()} unique drugs')

# Load all feature CSVs
tissue = pd.read_csv(MODELS_DIR / 'tissue_interaction_features_v5.csv')
binding = pd.read_csv(MODELS_DIR / 'binding_specificity_features_v5.csv')
tox = pd.read_csv(MODELS_DIR / 'toxicity_binding_features_v5.csv')
essential = pd.read_csv(MODELS_DIR / 'essential_gene_features_v5.csv')
network = pd.read_csv(MODELS_DIR / 'network_enrichment_features_v5.csv')

# DruMAP — match by canonical SMILES since DruMAP was run on corrected SMILES
drumap = pd.read_csv(SOURCES_DIR / 'drumap_combined.csv')
drumap_feats = drumap[['smiles', 'clint_reg', 'fup_rat_reg', 'fup_reg', 'kpbrain_reg',
                        'papp_caco2_reg', 'vd_reg', 'cyp1a2_prob', 'cyp2c9_prob',
                        'cyp2d6_prob', 'cyp3a4_prob']].copy()
drumap_feats = drumap_feats.rename(columns={
    'smiles': 'SMILES', 'clint_reg': 'drumap_clint', 'fup_rat_reg': 'drumap_fup_rat',
    'fup_reg': 'drumap_fup', 'kpbrain_reg': 'drumap_kpbrain',
    'papp_caco2_reg': 'drumap_papp_caco2', 'vd_reg': 'drumap_vd',
    'cyp1a2_prob': 'drumap_cyp1a2', 'cyp2c9_prob': 'drumap_cyp2c9',
    'cyp2d6_prob': 'drumap_cyp2d6', 'cyp3a4_prob': 'drumap_cyp3a4',
})
for col in ['drumap_cyp1a2', 'drumap_cyp2c9', 'drumap_cyp2d6', 'drumap_cyp3a4']:
    drumap_feats[col] = drumap_feats[col].map({'substrate': 1, 'non-substrate': 0}).astype(float)
# Map DruMAP SMILES to feature_IK
drumap_feats['feature_IK'] = drumap_feats['SMILES'].apply(get_pipeline_ik_for_trial)

# All feature CSVs use InChIKey as their key → rename to feature_IK for joining
for name, df in [('tissue', tissue), ('binding', binding), ('tox', tox), 
                  ('essential', essential), ('network', network)]:
    if 'InChIKey' in df.columns:
        df['feature_IK'] = df['InChIKey'].apply(lambda x: str(x)[:27] if pd.notna(x) else None)
    elif 'SMILES' in df.columns:
        df['feature_IK'] = df['SMILES'].apply(get_pipeline_ik_for_trial)
    n = df['feature_IK'].notna().sum()
    print(f'  {name:10s}: {n}/{len(df)} with feature_IK')

# Build drug-level feature table via outer join on feature_IK
all_features = [('tissue', tissue), ('binding', binding), ('tox', tox),
                ('essential', essential), ('network', network), ('drumap', drumap_feats)]

drug_features = None
for name, df in all_features:
    feat_cols = [c for c in df.columns if c not in ('SMILES', 'InChIKey', 'Drug_ID')]
    deduped = df[df['feature_IK'].notna()][feat_cols].drop_duplicates('feature_IK')
    if drug_features is None:
        drug_features = deduped
    else:
        drug_features = drug_features.merge(deduped, on='feature_IK', how='outer',
                                             suffixes=('', f'_{name}_dup'))
        dup_cols = [c for c in drug_features.columns if c.endswith('_dup')]
        if dup_cols:
            drug_features = drug_features.drop(columns=dup_cols)

print(f'\nDrug-level features: {len(drug_features)} drugs, {len(drug_features.columns)} columns')

# DEBUG: check overlap before join
trial_fiks = set(usable[usable['feature_IK'].notna()]['feature_IK'].unique())
feat_fiks = set(drug_features['feature_IK'].dropna())
overlap = trial_fiks & feat_fiks
print(f'Trial feature_IKs: {len(trial_fiks)}, Feature feature_IKs: {len(feat_fiks)}, Overlap: {len(overlap)}')

# Inner join trial data with drug features
train = usable[usable['feature_IK'].notna()].merge(drug_features, on='feature_IK', how='inner')

# Verify feature completeness
feature_groups = {
    'tissue': [c for c in train.columns if any(x in c for x in ['_interaction', 'n_enst_', 'n_high_protein', 'share_affected', 'weighted_score']) and 'binding' not in c],
    'binding': [c for c in train.columns if c.startswith('binding_')],
    'toxicity': [c for c in train.columns if c.startswith('tox_')],
    'essential': [c for c in train.columns if c.startswith('essential_') or 'vital_bind' in c or c in ('max_organ_vital_burden', 'organ_burden_spread', 'n_organs_at_risk')],
    'network': [c for c in train.columns if c.startswith('net_')],
    'drumap': [c for c in train.columns if c.startswith('drumap_')],
}

print(f'\nTraining dataset:')
print(f'  Trials: {len(train)}')
print(f'  Drugs:  {train["feature_IK"].nunique()}')
print(f'  Outcomes:')
print(train['Corrected_Outcome'].value_counts().to_string())
print(f'\nFeature completeness (% trials with non-null):')
for group, cols in feature_groups.items():
    if cols:
        pct = 100 * train[cols].notna().any(axis=1).mean()
        print(f'  {group:10s} ({len(cols):3d} cols): {pct:5.1f}%')
total = sum(len(v) for v in feature_groups.values())
print(f'  TOTAL: {total} features')

# =====================================================================
# Compute disease context + trial design features from trial metadata
# =====================================================================
# These were the #1 safety predictor module in v3 (standalone AUC 0.719).
# Computed from disease text and AACT trial data, not from the pipeline.

# --- Disease repopulation (Jun 12 2026): fix upstream join that dropped Disease text ---
# 189 trials (incl. the entire elagolix/endometriosis cluster) had Disease==NaN inherited from
# 12_trials_corrected_outcomes.csv, which silently zeroed all disease_is_* flags. Conditions were
# pulled from the ct.gov v2 conditionsModule (all 189 resolved) and committed as a correction file.
# This is a DATA-CORRECTNESS fix (Disease must never be NaN); measured efficacy-AUC impact is neutral
# because the disease_is_* flags are coarse. See notes/efficacy_FP_audit_jun12.md.
_disease_repop = pd.read_csv(SOURCES_DIR / 'disease_repopulation_jun12.csv')
_repop_map = dict(zip(_disease_repop['NCT_ID'], _disease_repop['Disease']))
_before_nan = train['Disease'].isna().sum()
_fill_mask = train['Disease'].isna() & train['NCT_ID'].isin(_repop_map)
train.loc[_fill_mask, 'Disease'] = train.loc[_fill_mask, 'NCT_ID'].map(_repop_map)
print(f'Disease repopulation: filled {int(_fill_mask.sum())} rows; Disease NaN {int(_before_nan)} -> {int(train["Disease"].isna().sum())}')

disease_text = train['Disease'].fillna('').str.lower()

# Disease context (5 binary flags)
train['disease_is_oncology'] = disease_text.str.contains(
    'cancer|carcinoma|lymphoma|leukemia|melanoma|sarcoma|myeloma|glioma|'
    'glioblastoma|neoplasm|tumor|nsclc|sclc|neuroblastoma|mesothelioma|'
    'oncology|metastat|malignant', regex=True).astype(int)
train['disease_is_infectious'] = disease_text.str.contains(
    'hiv|hepatitis|hcv|hbv|influenza|covid|sars|tuberculosis|malaria|'
    'bacterial|fungal|infection|pneumonia|sepsis|cmv|herpes|rsv|dengue', regex=True).astype(int)
train['disease_is_cns'] = disease_text.str.contains(
    'alzheimer|parkinson|epilepsy|seizure|schizophreni|depress|bipolar|psychiatr|mood disorder|obsessive|ptsd|adhd|autism|huntington|'
    'anxiety|dementia|multiple sclerosis|neuropath|migraine|stroke|brain', regex=True).astype(int)
train['disease_is_cardiac'] = disease_text.str.contains(
    'heart failure|atrial fibrillation|hypertension|coronary|myocardial|'
    'arrhythmia|angina|cardiac|cardiovascular|atherosclerosis', regex=True).astype(int)
train['disease_is_autoimmune'] = disease_text.str.contains(
    'rheumatoid|lupus|psoriasis|crohn|colitis|autoimmune|immunolog|atopic|eczema|dermatitis|hidradenitis|alopecia areata|vitiligo|'
    'ankylosing|scleroderma|vasculitis|pemphigus', regex=True).astype(int)

# Patient vulnerability (3 flags)
train['disease_is_metastatic'] = disease_text.str.contains(
    'metastat|advanced|stage iv|stage 4|unresectable|refractory', regex=True).astype(int)
train['disease_is_transplant'] = disease_text.str.contains(
    'transplant|graft|rejection', regex=True).astype(int)
train['disease_is_severe'] = disease_text.str.contains(
    'severe|critical|intensive care|icu|acute respiratory distress|ards|'
    'life.threatening|end.stage|terminal', regex=True).astype(int)

# Trial design (from AACT data)
# has_black_box: drug has FDA black box warning (from ChEMBL 36)
import sqlite3
_chembl_db = PROJECT_ROOT / 'data' / 'cache' / 'chembl_36' / 'chembl_36_sqlite' / 'chembl_36.db'
if _chembl_db.exists():
    _conn = sqlite3.connect(str(_chembl_db))
    _bbw = pd.read_sql('SELECT chembl_id FROM molecule_dictionary WHERE black_box_warning = 1', _conn)
    _conn.close()
    _chembl_lu = pd.read_csv(SOURCES_DIR / 'chembl_smiles_lookup.csv')
    _bbw_chembl_ids = set(_bbw['chembl_id'])
    _bbw_drugs = set(_chembl_lu[_chembl_lu['chembl_id'].isin(_bbw_chembl_ids)]['Drug_Clean'])
    train['has_black_box'] = train['Drug_Clean'].isin(_bbw_drugs).astype(int)
    print(f'Black box warning: {train["has_black_box"].sum()} trials ({train[train["has_black_box"]==1]["Drug_Clean"].nunique()} drugs)')
else:
    train['has_black_box'] = 0
    print('WARNING: ChEMBL DB not found, has_black_box = 0')

# is_combination: multiple drugs in the trial
# Count drugs per NCT_ID from the corrected outcomes
drugs_per_trial = co.groupby('NCT_ID')['Drug_Clean'].nunique().to_dict()
train['trial_n_drugs'] = train['NCT_ID'].map(drugs_per_trial).fillna(1).astype(int)
train['is_combination'] = (train['trial_n_drugs'] > 1).astype(int)

n_onc = train['disease_is_oncology'].sum()
n_inf = train['disease_is_infectious'].sum()
n_cns = train['disease_is_cns'].sum()
n_combo = train['is_combination'].sum()
print(f'Disease context: oncology={n_onc}, infectious={n_inf}, CNS={n_cns}')
print(f'Trial design: combinations={n_combo}, multi-drug trials={n_combo}')

# =====================================================================
# Flag anti-pathogen drugs and exclude supportive care mispairings
# =====================================================================

# Anti-pathogen drugs target pathogen proteins (not human). Our pipeline captures
# human-protein binding only, so efficacy mechanism is invisible. Keep for safety
# (off-target human binding IS the safety signal), exclude from efficacy.
ANTI_PATHOGEN_DRUGS = {
    # Antibiotics (target bacterial proteins)
    'amoxicillin', 'aztreonam', 'ceftaroline fosamil', 'dalbavancin',
    'delafloxacin', 'linezolid', 'meropenem', 'omadacycline', 'oxacillin',
    'tigecycline', 'clofazimine',
    # Antivirals (target viral proteins)
    'ABI-H2158', 'Inarigivir soproxil', 'LCQ908',
    'PBI-0451 (Pomotrelvir)', 'Shionogi Protease Inhibitor (S-217622)',
    'baloxavir marboxil', 'doravirine', 'favipiravir', 'foscarnet',
    'letermovir', 'maribavir', 'ribavirin', 'sofosbuvir',
    # Antiparasitics (target parasite proteins)
    'Artefenomel', 'artesunate',
}
train['is_anti_pathogen'] = train['Drug_Clean'].isin(ANTI_PATHOGEN_DRUGS).astype(int)
n_ap = train['is_anti_pathogen'].sum()
print(f'\nAnti-pathogen drugs flagged: {n_ap} trials ({n_ap/len(train)*100:.1f}%)')
print(f'  These will be EXCLUDED from efficacy training (kept for safety)')

# Endogenous hormones/molecules — identical to what the body produces.
# Their binding to human targets is physiologically normal, not a drug effect.
# Pipeline can't distinguish therapeutic from baseline binding.
# Synthetic mimetics (prednisone, desmopressin, etc.) are chemically different
# and produce meaningful binding predictions — they are KEPT.
ENDOGENOUS_HORMONES = {
    'testosterone', 'testosterone enanthate', 'testosterone undecanoate',
    'estradiol', 'progesterone', 'hydrocortisone', 'cortisone acetate',
    'melatonin', 'oxytocin', 'epinephrine', 'norepinephrine', 'dopamine',
    'calcitriol', 'vitamin D3', 'vasopressin',
    'levothyroxine', 'dinoprostone', 'epoprostenol',
}
train['is_endogenous'] = train['Drug_Clean'].isin(ENDOGENOUS_HORMONES).astype(int)
n_endo = train['is_endogenous'].sum()
print(f'Endogenous hormones flagged: {n_endo} trials ({n_endo/len(train)*100:.1f}%)')
print(f'  These will be EXCLUDED from efficacy training (binding = normal physiology)')

# Supportive care drugs mispaired with cancer diseases.
# These are analgesics/anesthetics/antiemetics tested for side-effect management
# but paired with cancer as the disease. The drug-disease features are wrong.
MISPAIRED_NCTS = {
    'NCT03267680', 'NCT01067144', 'NCT05680870', 'NCT04237090',
    'NCT04032119', 'NCT04327063', 'NCT03375515', 'NCT03269344',
    'NCT03131713', 'NCT02927379', 'NCT02740127', 'NCT02650791',
    'NCT02480114', 'NCT02330926', 'NCT02408393', 'NCT01651182',
    'NCT01814553', 'NCT01793480', 'NCT01718613', 'NCT01326325',
    'NCT04495894',
}
SUPPORTIVE_CARE_DRUGS = {
    'acetaminophen', 'diphenhydramine', 'famotidine', 'methylprednisolone',
    'bupivacaine', 'ropivacaine', 'ketamine', 'ketorolac', 'morphine',
    'fentanyl', 'gabapentin', 'ibuprofen', 'hydromorphone', 'methadone',
    'loperamide', 'propofol', 'cetirizine', 'omeprazole', 'epinephrine',
    'norepinephrine', 'tranexamic acid',
}
mispair_mask = (train['NCT_ID'].isin(MISPAIRED_NCTS)) & (train['Drug_Clean'].isin(SUPPORTIVE_CARE_DRUGS))
train['is_mispaired_supportive'] = mispair_mask.astype(int)
n_mp = train['is_mispaired_supportive'].sum()
print(f'Supportive care mispairings flagged: {n_mp} trials')
print(f'  These will be EXCLUDED from efficacy training (wrong drug-disease pairing)')

# Manual trial exclusions from disease curation review
# - Procedural/anesthesia/healthy volunteer: exclude from EFFICACY only (keep for safety)
# - Multi-drug: exclude from both
_excl_path = SOURCES_DIR / 'manual_trial_exclusions.csv'
if _excl_path.exists():
    _excl = pd.read_csv(_excl_path)
    _excl_keys = set(zip(_excl['NCT_ID'], _excl['Drug_Clean']))
    _excl_safety_keys = set(zip(_excl[_excl['exclude_safety']==True]['NCT_ID'], 
                                _excl[_excl['exclude_safety']==True]['Drug_Clean']))
    _excl_efficacy_keys = set(zip(_excl[_excl['exclude_efficacy']==True]['NCT_ID'],
                                  _excl[_excl['exclude_efficacy']==True]['Drug_Clean']))
    
    train['is_procedural_exclude'] = train.apply(
        lambda r: int((r['NCT_ID'], r['Drug_Clean']) in _excl_efficacy_keys), axis=1)
    train['is_multi_drug_exclude'] = train.apply(
        lambda r: int((r['NCT_ID'], r['Drug_Clean']) in _excl_safety_keys), axis=1)
    
    n_proc = train['is_procedural_exclude'].sum()
    n_multi = train['is_multi_drug_exclude'].sum()
    print(f'Manual exclusions: {n_proc} procedural/other (efficacy only), {n_multi} multi-drug (both)')

# Also flag "Healthy" volunteer trials (no disease)
healthy_mask = train['Disease'].fillna('').str.contains('Healthy', case=False)
train['is_healthy_volunteer'] = healthy_mask.astype(int)
n_hv = healthy_mask.sum()
if n_hv > 0:
    print(f'Healthy volunteer trials flagged: {n_hv} (excluded from efficacy)')

# Fill partial tissue NaN with 0 (biological zero: no HPA expression = no interaction)
# These are NOT missing data — they represent tissues where HPA has no expression
# data for the specific protein, so no interaction was detected.
tissue_fill_cols = [c for c in train.columns if any(c.startswith(p) for p in 
    ['max_', 'min_', 'mean_', 'share_', 'weighted_']) and train[c].dtype in ['float64', 'int64']]
# Only fill for drugs that HAVE tissue features (not drugs with no pipeline data)
_has_tissue = train[tissue_fill_cols].notna().any(axis=1)
for c in tissue_fill_cols:
    train.loc[_has_tissue & train[c].isna(), c] = 0.0
n_filled = (_has_tissue & ~train[tissue_fill_cols].notna().all(axis=1)).sum()
# Recount after fill
n_partial_after = (_has_tissue & ~train[tissue_fill_cols].notna().all(axis=1)).sum()
print(f'Tissue NaN filled with 0 (biological zeros): {n_filled} trials had partial → {n_partial_after} remain')

# Drop constant and redundant columns
# n_high_protein_in_* are HPA tissue-level counts, same for every drug
# Sensory tissue features have 96% NaN (HPA data gap, not pipeline issue)
# Disease-level network features are identical to drug-level (verified)
drop_cols = [c for c in train.columns if c.startswith('n_high_protein_in_')]
# Sensory tissue features have 36% NaN — HPA lacks sensory tissue expression data
drop_cols += [c for c in train.columns if 'sensory' in c]
drop_cols += [c for c in train.columns if c.startswith('net_disease_')]
# Also drop any column that is constant (zero variance) or mostly NaN
for c in train.columns:
    if c in drop_cols:
        continue
    if train[c].dtype not in ['float64', 'int64']:
        continue
    n_valid = train[c].notna().sum()
    if n_valid == 0:  # all NaN
        drop_cols.append(c)
    elif n_valid < 0.05 * len(train):  # <5% non-null
        drop_cols.append(c)
    elif train[c].dropna().nunique() <= 1:  # constant
        drop_cols.append(c)
if drop_cols:
    train = train.drop(columns=[c for c in drop_cols if c in train.columns])
    print(f'Dropped {len(drop_cols)} constant/redundant columns')

# Save
out_path = SOURCES_DIR / 'training_dataset_v5_unified.csv'
train.to_csv(out_path, index=False)
print(f'\nSaved: {out_path}')
print(f'Size: {out_path.stat().st_size / 1024 / 1024:.1f} MB')


[08:50:36] WARNING: Metal was disconnected; Proton(s) added/removed

[08:50:36] WARNING: Omitted undefined stereo

[08:50:36] WARNING: Metal was disconnected; Proton(s) added/removed

[08:50:36] WARNING: Metal was disconnected; Proton(s) added/removed

[08:50:36] WARNING: Omitted undefined stereo

[08:50:36] WARNING: Charges were rearranged

[08:50:36] WARNING: Omitted undefined stereo

[08:50:36] WARNING: Charges were rearranged

[08:50:36] WARNING: Metal was disconnected; Proton(s) added/removed

[08:50:36] WARNING: Omitted undefined stereo

[08:50:36] WARNING: Charges were rearranged

[08:50:36] bond type above 3 (17) is treated as unspecified!
[08:50:36] bond type above 3 (17) is treated as unspecified!
[08:50:36] ERROR: Unrecognized bond type: 0

[08:50:36] WARNING: Omitted undefined stereo

[08:50:36] WARNING: Omitted undefined stereo

[08:50:36] WARNING: Omitted undefined stereo

[08:50:36] WARNING: Omitted undefined stereo

[08:50:36] WARNING: Charges were rearranged

[08:50:36

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Charges were rearranged; Omitted undefined stereo

[08:50:37] WARNING: Charges were rearranged; Omitted undefined stereo

[08:50:37] WARNING: Charges were rearranged; Omitted undefined stereo

[08:50:37] WARNING: Charges were rearranged

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Charges were rearranged; Omitted undefined stereo

[08:50:37] WARNING: Charges were rearranged; Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Charges were rearranged; Omitted undefined stereo

[08:50:37] WARNING: Charges were rearranged; Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitte

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Charges were rearranged

[08:50:37] WARNING: Charges were rearranged

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Charges were rearranged

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Charges were rearranged; Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] W

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Charges were rearranged

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Charges were rearranged

[08:50:37] WARNING: Charges were rearranged

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined stereo

[08:50:37] WARNING: Omitted undefined

...

Trial drugs matched to pipeline: 4204/4666 trials, 829 unique drugs


[08:50:38] WARNING: Proton(s) added/removed

[08:50:38] WARNING: Omitted undefined stereo

[08:50:38] WARNING: Omitted undefined stereo

[08:50:38] WARNING: Omitted undefined stereo

[08:50:38] WARNING: Omitted undefined stereo

[08:50:38] WARNING: Omitted undefined stereo

[08:50:38] WARNING: Omitted undefined stereo

[08:50:38] WARNING: Charges were rearranged; Proton(s) added/removed

[08:50:38] WARNING: Omitted undefined stereo

[08:50:38] WARNING: Omitted undefined stereo

[08:50:38] WARNING: Omitted undefined stereo

[08:50:38] WARNING: Omitted undefined stereo

[08:50:38] WARNING: Omitted undefined stereo

[08:50:38] WARNING: Proton(s) added/removed

[08:50:38] WARNING: Omitted undefined stereo

[08:50:38] WARNING: Charges were rearranged; Omitted undefined stereo

[08:50:38] WARNING: Charges were rearranged

[08:50:38] WARNING: Omitted undefined stereo

[08:50:38] WARNING: Omitted undefined stereo

[08:50:38] WARNING: Omitted undefined stereo

[08:50:38] WARNING: Charges were r

  tissue    : 1423/1423 with feature_IK
  binding   : 1427/1427 with feature_IK
  tox       : 1427/1427 with feature_IK
  essential : 1427/1427 with feature_IK
  network   : 1434/1434 with feature_IK

Drug-level features: 1505 drugs, 218 columns
Trial feature_IKs: 829, Feature feature_IKs: 1505, Overlap: 829

Training dataset:
  Trials: 4204
  Drugs:  829
  Outcomes:
Corrected_Outcome
PASS             3622
FAIL_EFFICACY     426
FAIL_SAFETY       142
FAIL_BOTH          14

Feature completeness (% trials with non-null):
  tissue     (116 cols):  99.4%
  binding    ( 23 cols):  99.5%
  toxicity   ( 20 cols):  99.5%
  essential  ( 12 cols):  99.5%
  network    ( 29 cols):  93.6%
  drumap     ( 10 cols): 100.0%
  TOTAL: 210 features


Black box warning: 1778 trials (254 drugs)
Disease context: oncology=973, infectious=524, CNS=312
Trial design: combinations=1755, multi-drug trials=1755

Anti-pathogen drugs flagged: 184 trials (4.4%)
  These will be EXCLUDED from efficacy training (kept for safety)
Endogenous hormones flagged: 86 trials (2.0%)
  These will be EXCLUDED from efficacy training (binding = normal physiology)
Supportive care mispairings flagged: 27 trials
  These will be EXCLUDED from efficacy training (wrong drug-disease pairing)
Manual exclusions: 110 procedural/other (efficacy only), 3 multi-drug (both)
Healthy volunteer trials flagged: 5 (excluded from efficacy)
Tissue NaN filled with 0 (biological zeros): 0 trials had partial → 0 remain
Dropped 38 constant/redundant columns



Saved: <repo>/data/sources/training_dataset_v5_unified.csv
Size: 9.5 MB


In [ ]:
# === 14g. Direct disease-target engagement features (3 features) ===
#
# Captures cases the pathway-aggregation features miss: single-target drugs that
# bind the disease's actual target gene directly but don't trigger pathway-level
# enrichment (e.g., CFTR modulators for cystic fibrosis, ivacaftor at 0.55 vs CFTR
# but net_has_disease_pathway_overlap = 0).
#
# Method: for each trial, look up the top-10 OpenTargets disease genes, map to
# transcripts, find the drug's max Binding score against those transcripts.
# Bypasses significance thresholds and STRING-DB pathway enrichment.

import json, pandas as pd, numpy as np
from pathlib import Path

disease_cache = json.load(open(PROJECT_ROOT / 'data' / 'cache' / 'disease_targets_cache.json'))
gene_to_enst_cache = json.load(open(PROJECT_ROOT / 'data' / 'cache' / 'gene_to_enst_cache.json'))

# Disease cache has two layers:
#   search:<lower_name> → disease ID string (e.g., "MONDO_0009061")
#   targets:<ID> → dict with 'targets' key = list of {gene, ensembl_id, score}
# Also: <Title Case Disease> → list directly (older entries)

def _extract_targets(v, top_k=10):
    """Extract list of top-K gene dicts from whatever cache value form."""
    if v is None: return []
    if isinstance(v, list):
        return v[:top_k]
    if isinstance(v, dict):
        t = v.get('targets')
        if isinstance(t, list): return t[:top_k]
    return []

def normalize_disease_name(s):
    return ' '.join(str(s).lower().replace('-',' ').replace('_',' ').replace(',',' ').replace("'",'').split())

# Build a normalized search index
_search_norm_to_key = {}
for k in disease_cache:
    if not k.startswith('search:'): continue
    raw = k[len('search:'):]
    for piece in raw.split(';'):
        n = normalize_disease_name(piece)
        if n and n not in _search_norm_to_key:
            _search_norm_to_key[n] = k

def get_disease_top_genes(disease_str, top_k=10):
    if disease_str is None or pd.isna(disease_str): return []
    # Strategy 1: direct title-case lookup
    v = disease_cache.get(str(disease_str))
    out = _extract_targets(v, top_k)
    if out: return out
    # Strategy 2: search:<lower> → ID → targets:<ID>
    norm = normalize_disease_name(disease_str)
    sk = _search_norm_to_key.get(norm)
    if sk:
        did = disease_cache.get(sk)
        if isinstance(did, str):
            # Some search: values are bare IDs, others are "ID targets"; take first token
            did = did.split()[0] if did else did
            out = _extract_targets(disease_cache.get(f'targets:{did}'), top_k)
            if out: return out
    # Strategy 3: substring fallback against normalized search keys
    for n, k in _search_norm_to_key.items():
        if (norm in n or n in norm) and len(n) > 4:
            did = disease_cache.get(k)
            if isinstance(did, str):
                did = did.split()[0]
                out = _extract_targets(disease_cache.get(f'targets:{did}'), top_k)
                if out: return out
    return []

# Drug score files indexed by IK14
binding_dir = PROJECT_ROOT / 'data' / 'raw' / 'pipeline_all' / 'binding'
ik14_to_file = {}
for f in binding_dir.glob('*_drug_scores.tsv'):
    ik14_to_file.setdefault(f.name[:14], f)
print(f'Indexed {len(ik14_to_file)} drug_scores.tsv files by IK14')

_score_cache = {}
def get_drug_scores(ik14):
    if ik14 in _score_cache: return _score_cache[ik14]
    f = ik14_to_file.get(ik14)
    if f is None:
        _score_cache[ik14] = None; return None
    d = pd.read_csv(f, sep='\t')
    tx_col = next(c for c in d.columns if c.lower() in ('transcript','transcipt'))
    score_col = next(c for c in d.columns if c.lower()=='score')
    _score_cache[ik14] = dict(zip(d[tx_col], d[score_col]))
    return _score_cache[ik14]

from rdkit import Chem
def _smiles_ik14(s):
    try:
        m = Chem.MolFromSmiles(s)
        if m: return Chem.MolToInchiKey(m)[:14]
    except: pass
    return None

train_path = SOURCES_DIR / 'training_dataset_v5_unified.csv'
train = pd.read_csv(train_path, low_memory=False)
print(f'Loaded {train_path.name}: {len(train)} trials')

# Drop any existing direct_target columns before recomputing
new_cols = ['direct_target_max','direct_target_mean','direct_target_n_engaged_05','direct_target_n_genes_matched']
train = train.drop(columns=[c for c in new_cols if c in train.columns])

rows = []
n_no_drug = 0; n_no_disease = 0; n_no_gene_match = 0
for idx, row in train.iterrows():
    ik14 = _smiles_ik14(row['SMILES'])
    scores = get_drug_scores(ik14) if ik14 else None
    if scores is None:
        n_no_drug += 1
        rows.append({'_idx': idx, 'direct_target_max': np.nan, 'direct_target_mean': np.nan,
                     'direct_target_n_engaged_05': np.nan, 'direct_target_n_genes_matched': 0})
        continue
    targets = get_disease_top_genes(row['Disease'])
    if not targets:
        n_no_disease += 1
        rows.append({'_idx': idx, 'direct_target_max': np.nan, 'direct_target_mean': np.nan,
                     'direct_target_n_engaged_05': np.nan, 'direct_target_n_genes_matched': 0})
        continue
    per_gene_max = []
    for t in targets:
        gene = (t.get('gene') or t.get('symbol')) if isinstance(t, dict) else None
        if not gene: continue
        ensts = gene_to_enst_cache.get(gene, [])
        if not ensts: continue
        gmax = max((scores.get(e, 0) for e in ensts), default=0)
        per_gene_max.append(gmax)
    if not per_gene_max:
        n_no_gene_match += 1
        rows.append({'_idx': idx, 'direct_target_max': np.nan, 'direct_target_mean': np.nan,
                     'direct_target_n_engaged_05': np.nan, 'direct_target_n_genes_matched': 0})
        continue
    rows.append({
        '_idx': idx,
        'direct_target_max': float(max(per_gene_max)),
        'direct_target_mean': float(np.mean(per_gene_max)),
        'direct_target_n_engaged_05': int(sum(1 for x in per_gene_max if x >= 0.5)),
        'direct_target_n_genes_matched': len(per_gene_max),
    })

feat = pd.DataFrame(rows).set_index('_idx')
covered = feat.direct_target_max.notna().sum()
print(f'\nDirect-target feature coverage: {covered}/{len(train)} trials = {100*covered/len(train):.1f}%')
print(f'  Skipped — no drug score file: {n_no_drug}')
print(f'  Skipped — no disease cache hit: {n_no_disease}')
print(f'  Skipped — no gene→ENST mapping: {n_no_gene_match}')
if covered:
    print('\ndirect_target_max distribution (non-NaN):')
    print(feat.direct_target_max.describe([.25,.5,.75]))

# Sanity spot-checks
print('\nSanity checks (single-target drugs that were previously zero):')
for drug, disease in [('ivacaftor','cystic fibrosis'),('lumacaftor','cystic fibrosis'),('imatinib','myeloid'),('olaparib','ovarian'),('metoprolol','copd')]:
    sub = train[train.Drug_Clean.str.lower().str.contains(drug, na=False) & train.Disease.str.lower().str.contains(disease, na=False)].head(1)
    if len(sub) and sub.index[0] in feat.index:
        r = feat.loc[sub.index[0]]
        print(f'  {drug:12s}-{disease[:18]:18s}  direct_target_max={r.direct_target_max:.3f}  matched_genes={int(r.direct_target_n_genes_matched)}')
    else:
        print(f'  {drug:12s}-{disease[:18]:18s}  no matching trial')

# Merge into trial features and resave
train = train.join(feat[new_cols])
train.to_csv(train_path, index=False)
print(f'\nSaved updated {train_path.name} with 4 new direct_target_* columns ({len(train)} trials × {len(train.columns)} columns)')


In [ ]:
# === 14h. Organ-specific toxicity panels (dermatologic, musculoskeletal, GI) ===
#
# Closes SIDER concordance gap for TKI dermatologic, statin muscle, NSAID GI failures.
# Each panel: curated gene list → Binding binding aggregation → 4 features.
# No new pipeline runs needed; uses local drug_scores.tsv files.
#
# Validation criteria per panel:
#   1. Coverage ≥85% of trained drugs
#   2. Expected drug class shows elevated values (Cohen's d > 0.5 vs all-drug)
#   3. SIDER concordance lift within affected class (Spearman ρ > 0.3)

import json, pandas as pd, numpy as np
from pathlib import Path

gene_to_enst_cache = json.load(open(PROJECT_ROOT / 'data' / 'cache' / 'gene_to_enst_cache.json'))

# --- Panel gene lists ---
DERM_GENES = {
    'EGFR','ERBB2','ERBB3','ERBB4',                       # EGFR family — canonical TKI rash mechanism
    'KRT5','KRT14','KRT10','KRT1',                        # keratinocyte structure
    'FLG','LOR','IVL','CDSN',                              # skin barrier
    'TYR','TYRP1','MITF','MC1R',                          # follicular / pigmentation
    'TSLP','IL17A','IL23A','IL36G',                       # skin inflammation effectors
    'KDR','FLT1',                                          # VEGFR-mediated skin effects
}
MUSCLE_GENES = {
    'HMGCR','MVD','GGCX','PMVK',                          # statin target + biosynthesis
    'SLCO1B1','SLCO1B3','ABCG2','CYP3A4','CYP2C9',        # statin transporters / DDI
    'COQ2','COQ10A','COQ10B','MT-CYB','NDUFA9',           # mitochondrial respiratory chain
    'DMD','DYSF','CAV3','RYR1','CACNA1S',                 # muscle structural / channel
    'ACADVL','PPARGC1A','AMPD1','GATM',                   # energy metabolism
}
GI_GENES = {
    'PTGS1','PTGS2',                                       # COX-1, COX-2 (NSAID targets)
    'ATP4A','ATP4B','MUC5AC','MUC6','TFF1','TFF2','TFF3', # gastric mucosa
    'CDH17','CLDN2','CLDN3','CLDN4','OCLN','TJP1',        # intestinal barrier
    'HTR3A','HTR4','CHRM3',                                # GI motility / receptors
    # bile/lipid removed — NPC1L1/ABCB11/SLC10A2 contaminate statin GI signal; they are hepatobiliary not gastric mucosa
}

PANELS = {'dermatologic': DERM_GENES, 'muscle': MUSCLE_GENES, 'gi': GI_GENES}

# --- Map genes → ENSTs for each panel ---
panel_ensts = {}
for panel_name, genes in PANELS.items():
    ensts = set()
    n_mapped = 0
    for g in genes:
        gene_ensts = gene_to_enst_cache.get(g, [])
        if gene_ensts:
            n_mapped += 1
            ensts.update(gene_ensts)
    panel_ensts[panel_name] = ensts
    print(f'Panel {panel_name}: {n_mapped}/{len(genes)} genes mapped, {len(ensts)} transcripts')

# --- Drug score files indexed by IK14 ---
binding_dir = PROJECT_ROOT / 'data' / 'raw' / 'pipeline_all' / 'binding'
ik14_to_file = {f.name[:14]: f for f in binding_dir.glob('*_drug_scores.tsv')}

_score_cache = {}
def get_drug_scores(ik14):
    if ik14 in _score_cache: return _score_cache[ik14]
    f = ik14_to_file.get(ik14)
    if f is None:
        _score_cache[ik14] = None; return None
    d = pd.read_csv(f, sep='\t')
    tx_col = next(c for c in d.columns if c.lower() in ('transcript','transcipt'))
    score_col = next(c for c in d.columns if c.lower()=='score')
    _score_cache[ik14] = dict(zip(d[tx_col], d[score_col]))
    return _score_cache[ik14]

from rdkit import Chem
def _smiles_ik14(s):
    try:
        m = Chem.MolFromSmiles(s)
        if m: return Chem.MolToInchiKey(m)[:14]
    except: pass
    return None

# --- Compute features per trial ---
train_path = SOURCES_DIR / 'training_dataset_v5_unified.csv'
train = pd.read_csv(train_path, low_memory=False)
print(f'\nLoaded {train_path.name}: {len(train)} trials')

# Drop existing panel columns to allow idempotent reruns
panel_cols = [f'tox_{p}_{stat}' for p in PANELS for stat in ('max_bind','mean_bind','n_bound','burden')]
train = train.drop(columns=[c for c in panel_cols if c in train.columns])

rows = []
n_no_drug = 0
for idx, row in train.iterrows():
    ik14 = _smiles_ik14(row['SMILES'])
    scores = get_drug_scores(ik14) if ik14 else None
    if scores is None:
        n_no_drug += 1
        rows.append({'_idx': idx, **{c: np.nan for c in panel_cols}})
        continue
    out = {'_idx': idx}
    for panel_name, ensts in panel_ensts.items():
        per_gene_max = []
        # group ENSTs back by gene for per-gene max
        for gene in PANELS[panel_name]:
            gene_ensts = gene_to_enst_cache.get(gene, [])
            if not gene_ensts: continue
            gmax = max((scores.get(e, 0) for e in gene_ensts), default=0)
            per_gene_max.append(gmax)
        if not per_gene_max:
            for stat in ('max_bind','mean_bind','n_bound','burden'):
                out[f'tox_{panel_name}_{stat}'] = np.nan
            continue
        out[f'tox_{panel_name}_max_bind'] = float(max(per_gene_max))
        out[f'tox_{panel_name}_mean_bind'] = float(np.mean(per_gene_max))
        out[f'tox_{panel_name}_n_bound'] = int(sum(1 for x in per_gene_max if x >= 0.7))
        out[f'tox_{panel_name}_burden'] = float(sum(max(0, x-0.5) for x in per_gene_max))
    rows.append(out)

feat = pd.DataFrame(rows).set_index('_idx')
print(f'\nPanel feature coverage (trials with drug_scores file): {len(train) - n_no_drug}/{len(train)} = {100*(len(train)-n_no_drug)/len(train):.1f}%')
for panel in PANELS:
    print(f'  tox_{panel}_max_bind distribution: mean={feat[f"tox_{panel}_max_bind"].mean():.3f}  median={feat[f"tox_{panel}_max_bind"].median():.3f}')

# --- Validation: drug-class signatures ---
print('\n=== Drug-class signatures (mean tox_<panel>_max_bind per class vs all-drug mean) ===')
classes = {
    'TKI': ['dasatinib','erlotinib','sorafenib','cabozantinib','gefitinib','imatinib','nilotinib','regorafenib','sunitinib','vandetanib','lenvatinib','neratinib','lapatinib','ribociclib','alpelisib','afatinib','osimertinib'],
    'statin': ['atorvastatin','fluvastatin','lovastatin','pitavastatin','simvastatin','rosuvastatin','pravastatin','cerivastatin'],
    'NSAID': ['aspirin','celecoxib','diclofenac','etoricoxib','ibuprofen','ketorolac','naproxen','rofecoxib','meloxicam','indomethacin'],
    'corticosteroid': ['prednisone','dexamethasone','prednisolone','methylprednisolone','hydrocortisone'],
}
m = train.join(feat[panel_cols])
for cls_name, drugs in classes.items():
    sub = m[m.Drug_Clean.str.lower().isin([d.lower() for d in drugs])]
    print(f'\n  {cls_name} (n={len(sub)} trials, {sub.Drug_Clean.nunique()} unique drugs):')
    for panel in PANELS:
        cv = m[f'tox_{panel}_max_bind'].dropna()
        cls_v = sub[f'tox_{panel}_max_bind'].dropna()
        if len(cls_v):
            pooled_sd = cv.std()
            d = (cls_v.mean() - cv.mean()) / pooled_sd if pooled_sd else 0
            mark = '✓' if abs(d) > 0.5 else ' '
            print(f'    tox_{panel}_max_bind: class={cls_v.mean():.3f}  all={cv.mean():.3f}  Cohen d={d:+.2f}  {mark}')

# Spot-check ivacaftor, EGFR drug, NSAID
print('\nSpot checks:')
for drug in ['ivacaftor','erlotinib','aspirin','simvastatin','olaparib','metoprolol']:
    sub = m[m.Drug_Clean.str.lower()==drug.lower()].head(1)
    if len(sub):
        s = sub.iloc[0]
        print(f'  {drug:15s}  derm_max={s.tox_dermatologic_max_bind:.3f}  muscle_max={s.tox_muscle_max_bind:.3f}  gi_max={s.tox_gi_max_bind:.3f}')

# Merge back and save
train = train.join(feat[panel_cols])
train.to_csv(train_path, index=False)
print(f'\nSaved {train_path.name} with 12 new panel features ({len(train)} trials × {len(train.columns)} columns)')


---

### 14i. Arm-level decomposition (V6, May 22 2026)

Earlier cells produce a per-(NCT, Drug_Clean) row structure that silently does the wrong
thing for multi-drug trials:
- *Head-to-head trials* (e.g., NCT01006980 "Vemurafenib vs Dacarbazine") get split into
  two rows. Both rows carry the trial-level FAIL outcome, but the dacarbazine row really
  represents the active comparator — not a drug the trial was testing.
- *Add-on regimen trials* (e.g., NCT01080391 "CRd vs Rd": Carfilzomib added to
  Lenalidomide-Dex) get split per-drug; the model trains on Dexamethasone as if it were
  the failing investigational compound when really only Carfilzomib was new.

The arm-level rebuild produces one row per *trial arm* with:
- `All_Drugs` (semicolon-separated): every drug a patient on this arm actually receives
- `Investigational_Drugs`: the subset of `All_Drugs` flagged as the trial's investigational
  addition (computed from CT.gov arm_groups + intervention names + backbone subtraction)
- `Arm_Type`: from CT.gov `armGroupType` (EXPERIMENTAL / ACTIVE_COMPARATOR / PLACEBO_COMPARATOR / …)
- Aggregated pipeline features: per-target MAX across the arm's drugs (cumulative effect)
- Existing per-trial flags (disease context, is_combination, has_black_box, etc.)

The implementation lives in `scripts/{fetch_trial_descriptions,decompose_trials_to_arms,build_arm_level_dataset}.py`.
Cells below execute them in order. All three scripts are idempotent — the fetcher uses a
disk cache, decomposition is deterministic, and the feature builder reads on-disk inputs.


In [ ]:
import subprocess  # fix: was relying on later cell 87 import
# === 14i-0a. Augment chembl_smiles_lookup via local ChEMBL 36 SQLite ===
# Scans every drug name in trial_descriptions.json for names that don't resolve in
# the lookup built by cell 18, then queries ChEMBL SQLite (molecule_dictionary +
# molecule_synonyms) for SMILES. Idempotent; appends to chembl_smiles_lookup.csv.
# Catches ~575 entries on first run (codes, brand names, synonyms).
result = subprocess.run(
    ['python3', str(PROJECT_ROOT / 'scripts' / 'augment_chembl_via_sqlite.py')],
    capture_output=True, text=True
)
print(result.stdout[-1000:] if result.stdout else result.stderr[-1000:])


In [ ]:
# === 14i-0b. Augment chembl_smiles_lookup via PubChem bulk flat files ===
# Streams CID-Synonym-filtered.gz + CID-SMILES.gz (~2.4 GB cached at
# data/cache/pubchem_bulk/) to resolve remaining names. NO REST API calls —
# PubChem REST rate-limits aggressively. Bulk files cached after first download.
# Catches ~100 additional entries (mostly experimental compounds + abbreviations).
result = subprocess.run(
    ['python3', str(PROJECT_ROOT / 'scripts' / 'augment_pubchem_via_bulk.py')],
    capture_output=True, text=True, timeout=1800
)
print(result.stdout[-1500:] if result.stdout else result.stderr[-1500:])


In [ ]:
# === 14i-1. Fetch CT.gov descriptions + arm structure for every trial ===
# Cached at data/cache/trial_descriptions.json. Idempotent — only fetches NCTs
# not already in the cache. ~5-10 min on a cold cache; ~1s when fully cached.
import subprocess, json
result = subprocess.run(
    ['python3', str(PROJECT_ROOT / 'scripts' / 'fetch_trial_descriptions.py'), 'all'],
    capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else result.stderr[-500:])
with open(PROJECT_ROOT / 'data' / 'cache' / 'trial_descriptions.json') as _f:
    _cache = json.load(_f)
print(f'Cached NCTs: {len(_cache)}')


In [ ]:
# === 14i-2. Decompose every trial into per-arm rows ===
# Logic: each arm has Arm_Type from CT.gov; backbone drugs (intersection across
# all arms) are subtracted from EXPERIMENTAL arms to identify Investigational_Drugs.
# Outputs data/sources/trial_arms.csv (one row per arm).
result = subprocess.run(
    ['python3', str(PROJECT_ROOT / 'scripts' / 'decompose_trials_to_arms.py')],
    capture_output=True, text=True
)
print(result.stdout[-1200:] if result.stdout else result.stderr[-1200:])

arms = pd.read_csv(PROJECT_ROOT / 'data' / 'sources' / 'trial_arms.csv', low_memory=False)
print()
print(f'trial_arms.csv: {len(arms)} arm rows × {len(arms.columns)} cols, '
      f'{arms.NCT_ID.nunique()} unique NCTs')
print('Arm_Type breakdown:')
print(arms.Arm_Type.value_counts().to_string())


In [ ]:
# === 14i-2b. Post-decompose augment: resolve drugs that only appear AFTER ===
# === FDC dictionary expansion + regimen-string splitting at decompose time. ===
# Examples: 'ABT-450/r/ABT-267,ABT-333' → Paritaprevir, Ombitasvir, etc. — these
# component names need lookup entries before build_arm_level can attach features.
# Then re-run decompose so the now-resolved names get proper SMILES + IK in inv.
result = subprocess.run(
    ['python3', str(PROJECT_ROOT / 'scripts' / 'augment_post_decompose.py')],
    capture_output=True, text=True
)
print(result.stdout[-1000:] if result.stdout else result.stderr[-1000:])

print()
print('Re-running decompose with augmented lookup...')
result = subprocess.run(
    ['python3', str(PROJECT_ROOT / 'scripts' / 'decompose_trials_to_arms.py')],
    capture_output=True, text=True
)
print(result.stdout[-600:] if result.stdout else result.stderr[-600:])


In [ ]:
# === 14i-3. Build arm-level training dataset with aggregated features ===
# For each arm: parse All_InChIKeys → list of drug IK27s; look up per-drug pipeline
# features; aggregate via element-wise MAX across drugs (cumulative-effect semantics).
# Also computes disease_is_*, is_combination, trial_n_drugs, has_black_box, is_anti_pathogen,
# is_endogenous as arm-level OR across drugs.
# Outputs data/sources/training_dataset_arm_level.csv.
#
# Per-drug edge-case fixes applied inside build_arm_level_dataset.py (May 31 2026):
#
# FEATURE_IK_ALIASES — 3 drugs whose arm InChIKey stereo/isotope layer differs from
# the pipeline feature row for the same connectivity; the difference is scientifically
# immaterial (deuteration; R-enantiomer vs racemate; alternate stereo encoding), so
# the feature lookup is silently redirected to the equivalent compound while each
# drug's true identity is preserved in All_Drugs/All_InChIKeys:
#   • Donafenib  → sorafenib features  (deuterated sorafenib; same binding; 5 inv arms)
#   • Arbaclofen → baclofen features   (R-baclofen vs racemic; same GABA-B; 4 inv arms)
#   • Folinic acid → defined-stereo feature row (3 investigational arms)
# Ephedrine intentionally NOT aliased — it is a true diastereomer of pseudoephedrine.
#
# KNOWN_LARGE_PEPTIDE_IK14 — exenatide (39-aa GLP-1 peptide, IK14 HTQBXNHDCUEHJF)
# slipped past inv_biologic_only (it has a peptide SMILES and carries an IK14) and
# past the MW>900 inv_large_peptide check (absent from chembl_smiles_lookup). Fixed
# via a per-investigational-IK14 direct check; exenatide arms now flagged and excluded.
result = subprocess.run(
    ['python3', str(PROJECT_ROOT / 'scripts' / 'build_arm_level_dataset.py')],
    capture_output=True, text=True
)
print(result.stdout[-1200:] if result.stdout else result.stderr[-1200:])


In [ ]:
# === 14i-4. Inspect and validate the arm-level dataset ===
arm_df = pd.read_csv(PROJECT_ROOT / 'data' / 'sources' / 'training_dataset_arm_level.csv',
                     low_memory=False)
print(f'arm_level dataset: {arm_df.shape}')
print()
training_ready = arm_df[
    (arm_df.Corrected_Outcome.isin(['PASS','FAIL_EFFICACY','FAIL_SAFETY','FAIL_BOTH'])) &
    (arm_df['n_investigational'] > 0) &
    (arm_df['net_has_disease_pathway_overlap'].notna())
]
print(f'Training-ready arms (has investigational drug + features + valid outcome): '
      f'{len(training_ready)} from {training_ready.NCT_ID.nunique()} NCTs')
print()
print('Outcome distribution:')
print(training_ready['Corrected_Outcome'].value_counts().to_string())

# Spot-check three classic confounded trials
print()
print('=== Spot checks ===')
for nct in ['NCT01006980', 'NCT01080391', 'NCT01078454', 'NCT03926624', 'NCT01556347']:
    sub = arm_df[arm_df.NCT_ID == nct]
    if len(sub) == 0: continue
    print(f'{nct}: {sub.Trial_Title.iloc[0][:80]}')
    for _, r in sub.iterrows():
        inv = r.Investigational_Drugs if isinstance(r.Investigational_Drugs, str) else '(comparator)'
        print(f'  [{r.Arm_Type}] all={r.All_Drugs[:80] if isinstance(r.All_Drugs, str) else "—"} | inv={inv[:60] if isinstance(inv, str) else inv}')


In [ ]:
# === 14i-5. Regenerate the team review xlsx (preserving reviewer annotations) ===
# Reads the previous review xlsx to extract each reviewer's arm-partition and any
# existing right/wrong + notes annotations, then rebuilds tabs from current data.
# Output: data/review/training_dataset_arm_level_REVIEW-<date>.xlsx (4 reviewer
# tabs + Training Ready + All Arms + Suspect Review + Compared to v5 + Provenance).
result = subprocess.run(
    ['python3', str(PROJECT_ROOT / 'scripts' / 'regenerate_review_xlsx.py')],
    capture_output=True, text=True
)
print(result.stdout[-1000:] if result.stdout else result.stderr[-1000:])


In [ ]:
# === 14i-6. Auto-update reviewer verdicts based on objective criteria ===
# Reads the regenerated review xlsx and marks rows 'right' when current data
# meets clear correctness criteria (control arm correctly identified, single
# test drug with complete chemistry, methodology trial properly flagged, etc.).
# Uses plain 'right' verdicts; rationale lives in the notes column.
# Net effect on first run: ~3,500 of 3,700 reviewer rows marked right; ~120
# residual need human eyes for genuinely ambiguous cases.
result = subprocess.run(
    ['python3', str(PROJECT_ROOT / 'scripts' / 'auto_update_review_verdicts.py')],
    capture_output=True, text=True
)
print(result.stdout[-1200:] if result.stdout else result.stderr[-1200:])


In [27]:
# === 14f. Pipeline Validity Audit & Reprocessing List ===
#
# CRITICAL: The old SMILES were wrong for ~99 drugs. The pipeline computed
# binding scores, tissue interactions, and network enrichment for the WRONG
# molecules. We must NOT use those features.
#
# This cell:
# 1. Computes the CORRECT InChIKey for every trial drug (from ChEMBL SMILES)
# 2. Checks if first 14 chars of InChIKey (connectivity layer) match pipeline files
#    (stereochemistry layer differs between RDKit versions — only connectivity matters)
# 3. Drugs without matching pipeline data → need reprocessing
# 4. Generates the definitive reprocessing list for the science team
# 5. Only drugs WITH matching correct-IK pipeline data enter training

from rdkit import Chem
from rdkit.Chem import inchi as rdkinchi
import warnings
warnings.filterwarnings('ignore')

co = pd.read_csv(SOURCES_DIR / '12_trials_corrected_outcomes.csv')
usable = co[co['Corrected_Outcome'].isin(['PASS','FAIL_SAFETY','FAIL_EFFICACY','FAIL_BOTH'])]
chembl_lookup = pd.read_csv(SOURCES_DIR / 'chembl_smiles_lookup.csv')
chembl_ids = dict(zip(chembl_lookup['Drug_Clean'], chembl_lookup['chembl_id']))
chembl_names = dict(zip(chembl_lookup['Drug_Clean'], chembl_lookup['chembl_pref_name']))

# Step 1: Correct InChIKey for every trial drug
# Use smart_match for pipeline matching (handles stereoisomers)
trial_drug_ik = {}  # SMILES → pipeline IK (matched) or IK14 (unmatched)
trial_drug_full_ik = {}  # SMILES → full trial InChIKey (for output)
have_pipeline = {}
need_pipeline = {}
match_stats = {}

for smi in usable['SMILES'].dropna().unique():
    mol = Chem.MolFromSmiles(str(smi))
    if not mol: continue
    inch = rdkinchi.MolToInchi(mol)
    if not inch: continue
    full_ik = rdkinchi.InchiToInchiKey(inch)
    trial_drug_full_ik[smi] = full_ik
    
    matched_ik, match_type = smart_match(full_ik, pipeline_full_iks, pipeline_ik14_to_full, blocked_ik14s)
    match_stats[match_type] = match_stats.get(match_type, 0) + 1
    
    if matched_ik:
        have_pipeline[smi] = full_ik
        trial_drug_ik[smi] = full_ik[:14]
    else:
        need_pipeline[smi] = full_ik
        trial_drug_ik[smi] = full_ik[:14]

print(f'Trial drugs with valid InChIKey: {len(trial_drug_full_ik)}')
print(f'Smart match results:')
for mt, n in sorted(match_stats.items(), key=lambda x: -x[1]):
    print(f'  {mt}: {n}')

# Steps 2-3 handled by smart_match above (from cell 14a)
print(f'\nDrugs WITH matching pipeline data: {len(have_pipeline)}')
print(f'Drugs NEEDING pipeline run: {len(need_pipeline)}')

# Step 4: Generate reprocessing list
smi_to_drug = dict(zip(usable['SMILES'], usable.get('Drug_Clean', usable.get('Drug', ''))))
rows = []
for smi, ik in sorted(need_pipeline.items(), key=lambda x: -len(usable[usable['SMILES'] == x[0]])):
    drug = smi_to_drug.get(smi, '')
    n_trials = len(usable[usable['SMILES'] == smi])
    outcomes = usable[usable['SMILES'] == smi]['Corrected_Outcome'].value_counts().to_dict()
    rows.append({
        'Drug_Clean': drug,
        'SMILES': smi,
        'InChIKey': trial_drug_full_ik.get(smi, ik),
        'ChEMBL_ID': chembl_ids.get(drug, ''),
        'Resolved_Name': chembl_names.get(drug, ''),
        'N_Trials': n_trials,
        'Outcomes': str(outcomes),
    })

need_df = pd.DataFrame(rows)
# Exclude non-drugs
need_df = need_df[~need_df['Drug_Clean'].isin(['Sugar pill', 'Rovalpituzumab tesirine'])]
# NOTE: drugs_needing_full_pipeline_run.csv is now written authoritatively by the
# ARM-LEVEL step (scripts/rebuild_pipeline_reprocessing_files.py, run later in the
# Arm-Level Pipeline section). That version is aligned with actual arm-level feature
# coverage (drugs that WILL become trainable once reprocessed). This legacy trial-
# level computation is kept for the audit print below but no longer writes the file.

print(f'\n=== FOR SCIENCE TEAM: FULL PIPELINE REPROCESSING ===')
print(f'Drugs: {len(need_df)}')
print(f'Trials affected: {need_df["N_Trials"].sum()}')
print(f'File: drugs_needing_full_pipeline_run.csv')
print(f'Columns: Drug_Clean, SMILES, InChIKey, ChEMBL_ID, N_Trials, Outcomes')
print(f'\nThese drugs need: Binding binding + tissue interaction + network enrichment')
print(f'Feed SMILES and InChIKey to the Omic pipeline.')
print(f'\nTop 10 by trial count:')
for _, row in need_df.head(10).iterrows():
    print(f'  {row["Drug_Clean"]}: {row["N_Trials"]} trials')

# Step 5: Count what we CAN train on RIGHT NOW
trainable_smiles = set(have_pipeline.keys())
trainable_trials = usable[usable['SMILES'].isin(trainable_smiles)]
print(f'\n=== CURRENTLY TRAINABLE (correct pipeline data) ===')
print(f'Drugs: {len(trainable_smiles)}')
print(f'Trials: {len(trainable_trials)}')
print(trainable_trials['Corrected_Outcome'].value_counts().to_string())

# === Generate target-enriched reprocessing files ===

# Gene symbol → ENST mapping for disease targets
gene_enst_df = pd.read_csv(SOURCES_DIR / 'pipeline_gene_to_enst.csv')
# Use FIRST (canonical) ENST per gene — not all splice variants
gene_to_enst_map = {}
for _, row in gene_enst_df.iterrows():
    if pd.notna(row['enst_id']) and row['gene_symbol'] not in gene_to_enst_map:
        gene_to_enst_map[row['gene_symbol']] = [row['enst_id']]

def symbols_to_enst(symbols_str):
    """Convert semicolon-separated gene symbols to ENST IDs (1 per gene)."""
    if not symbols_str or pd.isna(symbols_str) or symbols_str == '':
        return ''
    enst_ids = []
    for sym in str(symbols_str).split(';'):
        ids = gene_to_enst_map.get(sym.strip(), [])
        if ids:
            enst_ids.append(ids[0])  # One canonical ENST per gene
    return ';'.join(enst_ids)
# These are the files the science team actually needs: drug + disease targets.

import json as json_mod

# Load target data
drug_targets_df = pd.read_csv(SOURCES_DIR / 'pipeline_drugs_with_targets.csv')
drug_targets_map = dict(zip(drug_targets_df['drug_name'], drug_targets_df['drug_targets_enst']))

disease_cache = json_mod.load(open(PROJECT_ROOT / 'data' / 'cache' / 'disease_targets_cache.json'))
disease_name_to_id = {k.replace('search:', ''): v 
                      for k, v in disease_cache.items() 
                      if k.startswith('search:') and isinstance(v, str)}

def get_disease_targets_top50(disease):
    if not disease or pd.isna(disease):
        return ''
    did = disease_name_to_id.get(str(disease).lower())
    if not did:
        return ''
    entry = disease_cache.get(f'targets:{did}')
    if not entry or not isinstance(entry, dict):
        return ''
    targets = entry.get('targets', [])
    if not targets:
        return ''
    top = sorted(targets, key=lambda t: t.get('score', 0), reverse=True)[:50]
    return ';'.join(t['symbol'] for t in top)

chembl_lookup_local = pd.read_csv(SOURCES_DIR / 'chembl_smiles_lookup.csv')
chembl_ids_local = dict(zip(chembl_lookup_local['Drug_Clean'], chembl_lookup_local['chembl_id']))
# chembl_names already defined above
# Build Drug_Clean → SMILES mapping for ALL need_pipeline drugs
# Multiple Drug_Clean values may share the same SMILES or IK14
need_smiles = set(need_pipeline.keys())
# Get ALL (Drug_Clean, SMILES) pairs that need reprocessing
need_drug_smi_pairs = usable[usable['SMILES'].isin(need_smiles)][['Drug_Clean','SMILES']].drop_duplicates()
smi_to_drug_local = dict(zip(usable['SMILES'], usable.get('Drug_Clean', usable.get('Drug', ''))))

# Trial-level: one row per NCT_ID for drugs needing reprocessing
trial_rows = []
for _, row in usable.iterrows():
    smi = row.get('SMILES')
    if not smi or smi not in need_smiles:
        continue
    drug = row.get('Drug_Clean', row.get('Drug', ''))
    disease = row.get('Disease', '')
    dt = get_disease_targets_top50(disease)
    drug_tgt = drug_targets_map.get(drug, '')
    if pd.isna(drug_tgt): drug_tgt = ''
    
    trial_rows.append({
        'NCT_ID': row['NCT_ID'],
        'Drug_Clean': drug,
        'SMILES': smi,
        'InChIKey': trial_drug_full_ik.get(smi, ''),
        'ChEMBL_ID': chembl_ids_local.get(drug, ''),
        'Disease': disease if pd.notna(disease) else '',
        'Corrected_Outcome': row['Corrected_Outcome'],
        'Drug_Targets_ENST': str(drug_tgt),
        'Disease_Targets_Symbols': dt,
        'Disease_Targets_ENST': symbols_to_enst(dt),
    })

trial_df = pd.DataFrame(trial_rows)
trial_df.to_csv(SOURCES_DIR / 'pipeline_reprocessing_trial_level.csv', index=False)
print(f'\nTrial-level reprocessing: {len(trial_df)} rows, {trial_df["Drug_Clean"].nunique()} drugs')

# Drug-level: one row per drug, aggregated targets
drug_rows = []
for smi, ik14 in sorted(need_pipeline.items()):
    drug = smi_to_drug_local.get(smi, '')
    drug_trials = usable[usable['SMILES'] == smi]
    diseases = sorted(drug_trials['Disease'].dropna().unique())
    
    all_symbols = set()
    for d in diseases:
        dt = get_disease_targets_top50(d)
        if dt: all_symbols.update(dt.split(';'))
    all_symbols.discard('')
    
    drug_tgt = drug_targets_map.get(drug, '')
    if pd.isna(drug_tgt): drug_tgt = ''
    
    drug_rows.append({
        'Drug_Clean': drug,
        'SMILES': smi,
        'InChIKey': trial_drug_full_ik.get(smi, ik14),
        'ChEMBL_ID': chembl_ids_local.get(drug, ''),
        'Diseases': '|'.join(diseases),
        'N_Diseases': len(diseases),
        'N_Usable_Trials': len(drug_trials),
        'Drug_Targets_ENST': str(drug_tgt),
        'Disease_Targets_Symbols': ';'.join(sorted(all_symbols)),
        'Disease_Targets_ENST': symbols_to_enst(';'.join(sorted(all_symbols))),
        'N_Disease_Target_Genes': len(all_symbols),
    })

drug_df = pd.DataFrame(drug_rows)
drug_df.to_csv(SOURCES_DIR / 'pipeline_reprocessing_drug_level.csv', index=False)
print(f'Drug-level reprocessing: {len(drug_df)} rows')
print(f'  With drug targets: {(drug_df["Drug_Targets_ENST"] != "").sum()}')
print(f'  With disease targets: {(drug_df["N_Disease_Target_Genes"] > 0).sum()}')
print(f'  File sizes: trial={os.path.getsize(SOURCES_DIR / "pipeline_reprocessing_trial_level.csv")//1024} KB, '
      f'drug={os.path.getsize(SOURCES_DIR / "pipeline_reprocessing_drug_level.csv")//1024} KB')


[08:50:40] WARNING: Metal was disconnected; Proton(s) added/removed

[08:50:40] WARNING: Omitted undefined stereo

[08:50:40] WARNING: Omitted undefined stereo

[08:50:40] WARNING: Charges were rearranged

[08:50:40] WARNING: Omitted undefined stereo

[08:50:40] WARNING: Charges were rearranged

[08:50:40] WARNING: Metal was disconnected; Proton(s) added/removed

[08:50:40] WARNING: Omitted undefined stereo

[08:50:40] bond type above 3 (17) is treated as unspecified!
[08:50:40] bond type above 3 (17) is treated as unspecified!
[08:50:40] ERROR: Unrecognized bond type: 0

[08:50:40] WARNING: Omitted undefined stereo

[08:50:40] WARNING: Charges were rearranged

[08:50:40] WARNING: Omitted undefined stereo

[08:50:40] WARNING: Charges were rearranged

[08:50:40] WARNING: Charges were rearranged; Omitted undefined stereo

[08:50:40] WARNING: Omitted undefined stereo

[08:50:40] WARNING: Omitted undefined stereo

[08:50:40] WARNING: Omitted undefined stereo

[08:50:40] WARNING: Omitted un

[08:50:41] WARNING: Omitted undefined stereo

[08:50:41] WARNING: Omitted undefined stereo

[08:50:41] WARNING: Omitted undefined stereo

[08:50:41] WARNING: Omitted undefined stereo

[08:50:41] WARNING: Charges were rearranged

[08:50:41] WARNING: Charges were rearranged; Omitted undefined stereo

[08:50:41] WARNING: Omitted undefined stereo

[08:50:41] WARNING: Omitted undefined stereo

[08:50:41] WARNING: Omitted undefined stereo

[08:50:41] WARNING: Proton(s) added/removed

[08:50:41] WARNING: Omitted undefined stereo

[08:50:41] WARNING: Omitted undefined stereo

[08:50:41] WARNING: Charges were rearranged

[08:50:41] WARNING: Omitted undefined stereo

[08:50:41] WARNING: Charges were rearranged

[08:50:41] WARNING: Omitted undefined stereo

[08:50:41] WARNING: Omitted undefined stereo

[08:50:41] WARNING: Omitted undefined stereo

[08:50:41] WARNING: Omitted undefined stereo

[08:50:41] WARNING: Omitted undefined stereo

[08:50:41] WARNING: Omitted undefined stereo

[08:50:41] WA

Trial drugs with valid InChIKey: 912
Smart match results:
  full_ik: 827
  no_ik14_match: 82
  ik14_achiral: 2
  blocked_stereoisomer: 1

Drugs WITH matching pipeline data: 829
Drugs NEEDING pipeline run: 83

=== FOR SCIENCE TEAM: FULL PIPELINE REPROCESSING ===
Drugs: 83
Trials affected: 455
File: drugs_needing_full_pipeline_run.csv
Columns: Drug_Clean, SMILES, InChIKey, ChEMBL_ID, N_Trials, Outcomes

These drugs need: Binding binding + tissue interaction + network enrichment
Feed SMILES and InChIKey to the Omic pipeline.

Top 10 by trial count:
  paclitaxel: 132 trials
  atorvastatin: 33 trials
  azithromycin: 30 trials
  everolimus: 19 trials
  vincristine: 17 trials
  vinorelbine: 13 trials
  lurasidone: 13 trials
  Taxotere: 12 trials
  elagolix: 10 trials
  montelukast: 9 trials

=== CURRENTLY TRAINABLE (correct pipeline data) ===
Drugs: 829
Trials: 4204
Corrected_Outcome
PASS             3622
FAIL_EFFICACY     426
FAIL_SAFETY       142
FAIL_BOTH          14



Trial-level reprocessing: 455 rows, 83 drugs
Drug-level reprocessing: 83 rows
  With drug targets: 1
  With disease targets: 37
  File sizes: trial=372 KB, drug=109 KB


---
## 14g. External Drug Safety Validation (Withdrawn2DB, EMA, FDA)

**Purpose:** Use external withdrawal databases as a VALIDATION SET, not training labels.

**Key principle (CLAUDE.md):** A trial outcome is a fact about that trial. Post-marketing
withdrawal is a fact about the drug's future. Using future information to relabel past
trial outcomes is temporal leakage.

**What we do:**
1. Cross-reference trial PASS drugs against Withdrawn2DB (647 drugs with toxicity types)
2. Identify PASS-labeled drugs that were later withdrawn for safety
3. Present for manual review — DO NOT auto-reclassify
4. After training, evaluate model's safety scores for these drugs (external validation)
5. Check FDA accelerated approval withdrawals in AACT for coverage gaps
6. Enrich existing FAIL_SAFETY trials with organ-specific toxicity from Withdrawn2DB

In [28]:
# === 14g. Cross-reference trials with external withdrawal databases ===
#
# Load Withdrawn2DB (647 drugs with SMILES, InChIKey, toxicity type)
# Match to our trial drugs by InChIKey-14
# Output: list of PASS drugs that were later withdrawn — FOR MANUAL REVIEW

import pandas as pd
from rdkit import Chem
from rdkit.Chem import inchi as rdkinchi
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('..').resolve()
SOURCES_DIR = PROJECT_ROOT / 'data' / 'sources'

# Load Withdrawn2DB
w2db = pd.read_csv(PROJECT_ROOT / 'data' / 'raw' / 'Caline-Drug-Lists' / 'Withdrawn2DB.csv')
print(f'Withdrawn2DB: {len(w2db)} drugs')

# Compute InChIKey-14 for Withdrawn2DB drugs
def ik14(smi):
    if not smi or pd.isna(smi): return None
    mol = Chem.MolFromSmiles(str(smi))
    if not mol: return None
    inch = rdkinchi.MolToInchi(mol)
    if not inch: return None
    return rdkinchi.InchiToInchiKey(inch)[:14]

w2db['IK14'] = w2db['inchikey'].apply(lambda x: str(x)[:14] if pd.notna(x) and len(str(x)) >= 14 else None)
# Also compute from SMILES as fallback
w2db_ik14_from_smi = w2db['smiles'].apply(ik14)
w2db['IK14'] = w2db['IK14'].fillna(w2db_ik14_from_smi)
print(f'With IK14: {w2db["IK14"].notna().sum()}')

# Build lookup: IK14 → withdrawal info
withdrawn_lookup = {}
for _, row in w2db.iterrows():
    if pd.notna(row['IK14']):
        withdrawn_lookup[row['IK14']] = {
            'drugname': row.get('drugname', ''),
            'toxtype': row.get('toxtype', ''),
            'toxclass': row.get('toxclass', ''),
            'firstwithdrawn': row.get('firstwithdrawn', ''),
            'lastwithdrawn': row.get('lastwithdrawn', ''),
            'withdrawalCountries': row.get('withdrawalCountries', ''),
        }

# Load our trial data
co = pd.read_csv(SOURCES_DIR / '12_trials_corrected_outcomes.csv')
usable = co[co['Corrected_Outcome'].isin(['PASS', 'FAIL_SAFETY', 'FAIL_EFFICACY', 'FAIL_BOTH'])].copy()
usable['IK14'] = usable['SMILES'].apply(ik14)

# Cross-reference
pass_trials = usable[usable['Corrected_Outcome'] == 'PASS']
pass_withdrawn = []

for _, row in pass_trials.iterrows():
    ik = row.get('IK14')
    if ik and ik in withdrawn_lookup:
        w = withdrawn_lookup[ik]
        pass_withdrawn.append({
            'NCT_ID': row['NCT_ID'],
            'Drug_Clean': row.get('Drug_Clean', row.get('Drug', '')),
            'Disease': row.get('Disease', ''),
            'Start_Year': row.get('Start_Year', ''),
            'Corrected_Outcome': row['Corrected_Outcome'],
            'W2DB_Name': w['drugname'],
            'Toxicity_Type': w['toxtype'],
            'Toxicity_Class': w['toxclass'],
            'First_Withdrawn': w['firstwithdrawn'],
            'Withdrawal_Countries': str(w['withdrawalCountries'])[:50],
        })

pw_df = pd.DataFrame(pass_withdrawn)
print(f'\n=== PASS DRUGS LATER WITHDRAWN (for manual review) ===')
print(f'Trials: {len(pw_df)}')
print(f'Unique drugs: {pw_df["Drug_Clean"].nunique()}')

# Also check FAIL_SAFETY overlap
fail_safety = usable[usable['Corrected_Outcome'].isin(['FAIL_SAFETY', 'FAIL_BOTH'])]
fail_in_w2db = fail_safety[fail_safety['IK14'].isin(withdrawn_lookup.keys())]
print(f'\nFAIL_SAFETY drugs also in Withdrawn2DB: {fail_in_w2db["Drug_Clean"].nunique()} (for organ-specific enrichment)')

# Print the PASS+Withdrawn list for manual review
print(f'\n{"="*80}')
print(f'FOR MANUAL REVIEW: {pw_df["Drug_Clean"].nunique()} PASS drugs that were later withdrawn')
print(f'These are NOT reclassified. Labels remain PASS (trial-derived).')
print(f'After training, we evaluate if the model assigns high safety risk to these drugs.')
print(f'{"="*80}')

drug_summary = pw_df.groupby('Drug_Clean').agg(
    N_Trials=('NCT_ID', 'count'),
    Diseases=('Disease', lambda x: '|'.join(x.dropna().unique()[:3])),
    Toxicity=('Toxicity_Type', 'first'),
    Withdrawn=('First_Withdrawn', 'first'),
    Countries=('Withdrawal_Countries', 'first'),
).sort_values('N_Trials', ascending=False)

for drug, row in drug_summary.iterrows():
    print(f'  {drug}: {row["N_Trials"]} trials, {row["Diseases"][:40]}')
    print(f'    Withdrawn: {row["Withdrawn"]} for {row["Toxicity"]} ({row["Countries"]})')

# Save validation set
pw_df.to_csv(SOURCES_DIR / 'validation_withdrawn_pass_drugs.csv', index=False)
print(f'\nSaved: validation_withdrawn_pass_drugs.csv ({len(pw_df)} rows)')

# Also check FDA accelerated approval withdrawals
try:
    fda_aa = pd.read_excel(PROJECT_ROOT / 'data' / 'raw' / 'Caline-Drug-Lists' / 'FDA' / 
                            'Withdrawn Cancer Accelerated Approvals FDA.xlsx')
    fda_drugs = fda_aa.iloc[:, 0].dropna().unique()  # First column = drug names
    print(f'\n=== FDA WITHDRAWN ACCELERATED APPROVALS ===')
    print(f'Cancer drugs: {len(fda_drugs)}')
    
    # Check which are in our AACT data
    aact = pd.read_csv(SOURCES_DIR / '07_aact_terminated_classified.csv', low_memory=False)
    for drug in sorted(fda_drugs)[:15]:
        drug_clean = str(drug).lower().split(' (')[0].strip()
        in_aact = aact[aact['intervention_name'].str.lower().str.contains(drug_clean, na=False)]
        in_trials = usable[usable['Drug_Clean'].str.lower() == drug_clean]
        terminated = in_aact[in_aact['failure_category'].isin(['safety', 'efficacy'])]
        print(f'  {drug}: {len(in_trials)} trials in data, {len(terminated)} terminated in AACT')
except Exception as e:
    print(f'FDA file error: {e}')

Withdrawn2DB: 647 drugs


With IK14: 636


[08:50:43] WARNING: Omitted undefined stereo

[08:50:43] WARNING: Omitted undefined stereo

[08:50:43] WARNING: Omitted undefined stereo

[08:50:43] WARNING: Charges were rearranged

[08:50:43] WARNING: Charges were rearranged

[08:50:43] WARNING: Omitted undefined stereo

[08:50:43] WARNING: Omitted undefined stereo

[08:50:43] WARNING: Omitted undefined stereo

[08:50:43] WARNING: Omitted undefined stereo

[08:50:43] WARNING: Omitted undefined stereo

[08:50:43] WARNING: Omitted undefined stereo

[08:50:43] WARNING: Charges were rearranged; Omitted undefined stereo

[08:50:43] WARNING: Proton(s) added/removed

[08:50:43] WARNING: Charges were rearranged; Omitted undefined stereo

[08:50:43] WARNING: Omitted undefined stereo

[08:50:43] WARNING: Omitted undefined stereo

[08:50:43] WARNING: Omitted undefined stereo

[08:50:43] WARNING: Omitted undefined stereo

[08:50:43] WARNING: Charges were rearranged; Omitted undefined stereo

[08:50:43] WARNING: Omitted undefined stereo

[08:50:4

[08:50:43] WARNING: Omitted undefined stereo

[08:50:43] WARNING: Charges were rearranged

[08:50:43] bond type above 3 (17) is treated as unspecified!
[08:50:43] bond type above 3 (17) is treated as unspecified!
[08:50:43] ERROR: Unrecognized bond type: 0

[08:50:43] WARNING: Omitted undefined stereo

[08:50:43] WARNING: Omitted undefined stereo

[08:50:43] WARNING: Omitted undefined stereo

[08:50:43] WARNING: Omitted undefined stereo

[08:50:43] WARNING: Charges were rearranged

[08:50:43] bond type above 3 (17) is treated as unspecified!
[08:50:43] bond type above 3 (17) is treated as unspecified!
[08:50:43] ERROR: Unrecognized bond type: 0

[08:50:43] WARNING: Omitted undefined stereo

[08:50:43] WARNING: Charges were rearranged

[08:50:43] WARNING: Omitted undefined stereo

[08:50:43] WARNING: Omitted undefined stereo

[08:50:43] WARNING: Metal was disconnected; Proton(s) added/removed

[08:50:43] WARNING: Charges were rearranged

[08:50:43] WARNING: Charges were rearranged

[08:

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Charges were rearranged

[08:50:44] WARNING: Charges were rearranged

[08:50:44] WARNING: Charges were rearranged

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Charges were rearranged; Omitted undefined stereo

[08:50:44] WARNING: Charges were rearranged

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Charges were rearranged

[08:50:44] WARNING: Charges were rearranged

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Charges were rearranged

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNI

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Charges were rearranged

[08:50:44] WARNING: Charges were rearranged

[08:50:44] WARNING: Charges were rearranged

[08:50:44] WARNING: Charges were rearranged

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Charges were rearranged

[08:50:44] WARNING: Charges were rearranged

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined st

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Charges were rearranged

[08:50:44] WARNING: Charges were rearranged

[08:50:44] WARNING: Charges were rearranged

[08:50:44] WARNING: Proton(s) added/removed

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Charges were rearranged

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Charges were rearranged

[08:50:44] WARNING: Charges were rearranged

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Charges were rearranged

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined ster

[08:50:44] WARNING: Charges were rearranged

[08:50:44] WARNING: Charges were rearranged

[08:50:44] WARNING: Charges were rearranged

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Charges were rearranged

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Charges were rearranged; Omitted undefined stereo

[08:50:44] WARNING: Charges were rearranged; Omitted undefined stereo

[08:50:44] WARNING: Charges were rearranged; Omitted undefined stereo

[08:50:44] WARNING: Charges were rearranged; Omitted undefined stereo

[08:50:44] WARNING: Charges were rearranged; Omitted undefined stereo

[08:50:44] WARNING: Charges were rearranged; Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitted undefined stereo

[08:50:44] WARNING: Omitte


=== PASS DRUGS LATER WITHDRAWN (for manual review) ===
Trials: 497
Unique drugs: 73

FAIL_SAFETY drugs also in Withdrawn2DB: 11 (for organ-specific enrichment)

FOR MANUAL REVIEW: 73 PASS drugs that were later withdrawn
These are NOT reclassified. Labels remain PASS (trial-derived).
After training, we evaluate if the model assigns high safety risk to these drugs.
  paclitaxel: 117 trials, Urothelial Cancer|Ovarian Cancer|Anal Ca
    Withdrawn: nan for None (EU)
  ribavirin: 58 trials, Dengue Hemorrhagic Fever|Cirrhosis|hepat
    Withdrawn: 2014.0 for respiratory (EU;USA)
  clopidogrel: 33 trials, Stent Thrombosis; Myocardial Infarction,
    Withdrawn: nan for None (EU)
  ticagrelor: 20 trials, Acute Ischaemic Stroke; Transient Ischae
    Withdrawn: nan for None (EU)
  fentanyl: 16 trials, Procedural Pain|Acute Pain; Ambulances|A
    Withdrawn: nan for None (EU)
  duloxetine: 16 trials, Improving Outcomes in People With Knee O
    Withdrawn: nan for None (EU)
  buprenorphine: 14 trials

  Copiktra (duvelisib): 0 trials in data, 0 terminated in AACT
  Exkivity (mobocertinib): 0 trials in data, 0 terminated in AACT
  Farydak (panobinostat): 0 trials in data, 0 terminated in AACT
  Gavreto (pralsetinib): 0 trials in data, 0 terminated in AACT
  Imbruvica (ibrutinib): 0 trials in data, 0 terminated in AACT
  Imfinzi (durvalumab): 0 trials in data, 0 terminated in AACT
  Iressa (gefitinib): 0 trials in data, 1 terminated in AACT
  Istodax (romidepsin): 0 trials in data, 0 terminated in AACT
  Keytruda (pembrolizumab): 0 trials in data, 0 terminated in AACT
  Lartruvo (olaratumab): 0 trials in data, 0 terminated in AACT


---
## 15. Disease Context Features

**Key insight**: the disease being treated strongly predicts safety failure risk.
Oncology trials fail for safety at 14.2% vs 4.8% for non-oncology.

Disease context features (binary flags from trial disease field):
- `disease_is_oncology` — Cancer, Leukemia, Lymphoma, Melanoma, etc.
- `disease_is_infectious` — HIV, Hepatitis, COVID, etc.
- `disease_is_cns` — Alzheimer's, Parkinson's, Depression, etc.
- `disease_is_cardiac` — Heart failure, Atrial fibrillation, etc.
- `disease_is_autoimmune` — Arthritis, Lupus, Psoriasis, GVHD, etc.

**52 drugs** pass for some diseases but fail for safety in others — 
same drug, different outcome depending on disease context.
This is the pair-level signal that drug-only features miss.

In [29]:
# Disease context analysis
df_ctx = pd.read_csv(SOURCES_DIR / 'training_dataset_v5_unified.csv', low_memory=False)
df_s = df_ctx[df_ctx['Corrected_Outcome'].isin(['PASS', 'FAIL_SAFETY', 'FAIL_BOTH'])].copy()
y = df_s['Corrected_Outcome'].isin(['FAIL_SAFETY', 'FAIL_BOTH'])

print('Safety failure rate by disease context:')
for col in ['disease_is_oncology', 'disease_is_infectious', 'disease_is_cns', 
            'disease_is_cardiac', 'disease_is_autoimmune']:
    if col in df_s.columns:
        mask = df_s[col] == 1
        if mask.sum() > 10:
            rate = y[mask].mean()
            print(f'  {col.replace("disease_is_",""):<15} {mask.sum():>5} trials  {y[mask].sum():>4} failures  ({rate*100:.1f}%)')

# Drugs that pass for some diseases but fail for others
drugs_mixed = set()
for drug, group in df_s.groupby('SMILES'):
    outcomes = set(group['Corrected_Outcome'])
    if 'PASS' in outcomes and ('FAIL_SAFETY' in outcomes or 'FAIL_BOTH' in outcomes):
        drugs_mixed.add(drug)
print(f'\nDrugs with mixed safety outcomes by disease: {len(drugs_mixed)}')

Safety failure rate by disease context:
  oncology          774 trials    94 failures  (12.1%)
  infectious        485 trials     6 failures  (1.2%)
  cns               283 trials     9 failures  (3.2%)
  cardiac           224 trials     5 failures  (2.2%)
  autoimmune        137 trials     2 failures  (1.5%)



Drugs with mixed safety outcomes by disease: 48


---
## 16. Label Quality Fixes (Error Analysis)

OOF predictions from the model were used to identify potential mislabels.
Trials where the model is most confident the label is wrong were manually reviewed.

Fixes applied:
- **NCT01712659** (ruxolitinib): "terminated after death of Lead investigator" — PI died, not drug safety. Excluded.
- **NCT01342016** (prednisone): "safety concern of active control drug" — the OTHER drug failed, not prednisone. Excluded.

Also identified but not yet fixed:
- Several combination therapy trials where the control drug (prednisone, methylprednisolone) is listed as the Drug, but the experimental drug caused the safety failure.

**Phase leakage caught**: `phase_numeric` perfectly separates PASS (all Phase 3) from FAIL (mostly Phase 2) by definition. Excluded from features.

---
## 18. Drug Context Features & Leakage Audit

Four drug context features were tested. **Three were found to leak outcome information** 
and were removed after audit:

| Feature | Verdict | Reason |
|---------|---------|--------|
| `is_combination` | **KEEP** | Trial design (multi-drug), not outcome |
| `drug_is_code` | **REMOVED** | Survivorship bias — drugs that fail keep codes, drugs that pass get named |
| `log_drug_n_trials` | **REMOVED** | Reverse causation — failure → termination → fewer trials |
| `log_drug_n_diseases` | **REMOVED** | Same — success causes expansion to more diseases |

**Lesson**: Any feature that encodes "how successful is this drug" is the outcome in disguise.
Drug trial count (1 trial = 85% fail, 20+ trials = 2.5% fail) perfectly reflects that 
failed drugs get stopped while successful ones continue.

In [30]:
# Drug context features
df_drug_ctx = pd.read_csv(SOURCES_DIR / 'training_dataset_v5_unified.csv', low_memory=False)
df_s = df_drug_ctx[df_drug_ctx['Corrected_Outcome'].isin(['PASS', 'FAIL_SAFETY', 'FAIL_BOTH'])].copy()
y = df_s['Corrected_Outcome'].isin(['FAIL_SAFETY', 'FAIL_BOTH'])

print('Safety failure rate by drug context:')
for col, label in [('is_combination', 'Combination therapy'),
                    ('drug_is_code', 'Code-named drug')]:
    if col in df_s.columns:
        mask = df_s[col] == 1
        rate = y[mask].mean() if mask.sum() > 0 else 0
        print(f'  {label}: {mask.sum()} trials, {y[mask].sum()} failures ({rate*100:.0f}%)')

print()
for threshold in [1, 5, 10, 20]:
    mask = df_s['log_drug_n_trials'] >= np.log1p(threshold) if 'log_drug_n_trials' in df_s.columns else pd.Series(False, index=df_s.index)
    if mask.sum() > 10:
        rate = y[mask].mean()
        print(f'  ≥{threshold} trials: {mask.sum()} trials, safety fail rate {rate*100:.1f}%')

Safety failure rate by drug context:
  Combination therapy: 1534 trials, 98 failures (6%)



---
## 20. Organ-Specific Toxicity Classification

All 120 unique safety failure NCT IDs were manually classified by organ/mechanism.
73% were classified to specific organs using:
1. `why_stopped` text (hepatotoxicity, cardiac, etc.)
2. Drug pharmacology knowledge (known class effects)
3. Literature research (ClinicalTrials.gov results, PubMed publications)

Key corrections from literature:
- NCT01717898 (BEZ235): grade 4 pneumonitis → pulmonary
- NCT04165031 (LY3499446): hemolysis → hematological (KRAS G12C)
- NCT01792635 (PF-05175157): QTc prolongation → cardiac
- NCT01199731: drug was fosdevirine (seizures), NOT etravirine as listed

In [31]:
# Load organ classifications
organ_clf = pd.read_csv(SOURCES_DIR / 'safety_failure_organ_classification.csv')
print(f'Organ-specific toxicity classification ({len(organ_clf)} trials):')
print()
print(organ_clf['tox_type'].value_counts().to_string())
print(f'\nOrgan-specific: {(organ_clf["tox_type"] != "general_tox").sum()} / {len(organ_clf)} '
      f'({100*(organ_clf["tox_type"] != "general_tox").mean():.0f}%)')

Organ-specific toxicity classification (120 trials):

tox_type
general_tox     32
hepato          12
hemato          10
neuro            9
preclinical      8
cardio           8
death_unspec     7
benefit_risk     6
combo_tox        6
pulm             5
GI               5
immune           4
nephro           3
dermal           2
hemorrhage       2
NOT_SAFETY       1

Organ-specific: 88 / 120 (73%)


---
## 21. Feature Exploration Summary

### What helped safety prediction (honest improvements):
| Feature | Safety Δ | Efficacy Δ | Why it works |
|---------|----------|------------|-------------|
| Disease context (is_oncology etc.) | +0.060 | +0.018 | Oncology = 3× failure rate |
| Toxicity binding (DNA damage, cardiac) | +0.018 | +0.018 | Direct organ toxicity risk |
| Patient vulnerability (metastatic, transplant) | +0.030 | +0.027 | Sicker patients = more toxicity |
| Multi-drug trial flag | ~+0.005 | ~+0.005 | Combo regimens are more toxic |
| DruMAP (f_up, CL_int) | marginal | marginal | Lower protein binding = more exposure |

### What was caught as leakage:
| Feature | Issue | How detected |
|---------|-------|-------------|
| drug_is_code | Survivorship bias (failed drugs keep codes) | 100% failure rate |
| log_drug_n_trials | Reverse causation (failure → fewer trials) | 85% failure for 1-trial drugs |
| log_drug_n_diseases | Same (success → more diseases tested) | 85% failure for 1-disease drugs |
| phase_numeric | Definitional (PASS = Phase 3 by definition) | Perfect separation |
| pass/fail_votes, CT_TOX, FDA_APPROVED | Direct outcome labels | AUC = 1.000 |

### What didn't help:
- Interaction terms (drug features × disease type) — noise
- Organ-matched risk scores — disease flags already capture the signal
- Structural alerts (18 SMARTS patterns) — too sparse at this sample size
- Anti-target panel binding — same direction issue as other binding features

### Current best model:
- **Safety: 0.748 ± 0.025** (quick 1-seed, 208 features)
- **Efficacy: 0.802 ± 0.053** (quick 1-seed)
- Dataset: 2,968 trials, 464 drugs with complete features
- 550 more drugs awaiting bioinformatics pipeline runs

---
## 22. Key Reversal: ADMET Features Predict Safety

With corrected labels, ADMET features that were previously declared "useless" now 
show strong significance for safety prediction:

| Feature | p-value | Direction | Mechanism |
|---------|---------|-----------|-----------|
| AMES | <0.0001 | ↑ in fail | Mutagenicity → genotoxicity |
| DILI | <0.0001 | ↑ in fail | Liver injury prediction |
| hERG | <0.0001 | ↑ in fail | Cardiac channel → QT prolongation |
| Pgp | 0.009 | ↑ in fail | Drug efflux pump substrate |
| LD50 | 0.04 | ↑ in fail | Acute toxicity |
| PPBR | 0.03 | ↑ in fail | Higher protein binding (paradoxical) |
| VDss | 0.01 | ↑ in fail | Volume of distribution |

**Why the reversal?** The old "useless" finding was based on labels where 86% of 
safety failures were from DILIrank/Withdrawn databases (approved drugs with known toxicity).
Those drugs had ALREADY passed pre-clinical ADMET screens — so ADMET features couldn't 
distinguish them from other approved drugs. With real trial failures, ADMET correctly
identifies drugs with pre-clinical toxicity signals that go on to fail in trials.

Adding 7 significant ADMET features: Safety **+0.013** AUC (0.740 → 0.753).

---
## 23. Disease-Mechanism Alignment Protects Against Safety Failure

Drugs with disease pathway overlap fail for safety at **4.3%**.  
Drugs without overlap fail at **16.8%** (4× higher).

This means the same "pathway alignment" signal that predicts efficacy ALSO protects 
against safety: drugs acting on disease-relevant targets have focused activity,
while drugs without disease alignment have off-target activity that causes toxicity.

Joint any-failure model (safety + efficacy combined): **AUC 0.819** — 
better than either alone because off-target biology predicts both failure modes.

---
## 27. Top-20 Feature Leakage Audit — All Clean

Every feature audited for outcome leakage, data availability confounds, and reverse causation.

| # | Feature | Source | NaN gap | Verdict |
|---|---------|--------|---------|---------|
| 1-11 | net_* (pathway alignment) | STRING-DB × OpenTargets | 15.5% vs 12.1% | **CLEAN** — molecular biology |
| 12 | binding_disease_n_bound_total | Binding × OpenTargets | 0% | **CLEAN** but hurts model (noise) |
| 13 | trial_n_drugs | Trial design | 0% | **CLEAN** — works in oncology (0.69 AUC), noise in non-onc |
| 14 | disease_is_oncology | Disease text | 0% | **CLEAN** — disease property, not outcome |
| 15 | AMES | ADMETlab2.0 from SMILES | 48% vs 35% | **CLEAN** — gap is data completeness, AUC=0.50 for NaN itself |
| 16 | DILI | ADMETlab2.0 from SMILES | same | **CLEAN** — same reason |
| 17 | hERG | ADMETlab2.0 from SMILES | same | **CLEAN** — same reason |
| 18-20 | binding_disease_* | Binding × OpenTargets | 0% | **CLEAN** — binding predictions |

**ADMET NaN gap explained**: 35% of drugs lack ADMET because they were newly recovered 
via SMILES lookups and haven't been run through ADMETlab2.0 yet. The NaN indicator itself 
has AUC=0.500 (no confound). Among drugs WITH ADMET data, the signal is confirmed real 
(AUC 0.63-0.64).

**LOO findings**: `binding_disease_n_bound_total` and `trial_n_drugs` have negative LOO 
contribution (removing them improves AUC). This is overfitting from correlated features, 
not leakage. The 10 network features capture overlapping signal.

**Final verdict**: All 20 features are legitimate pre-trial information. 
Model: **Safety 0.797, Efficacy 0.806** with 20 features.

---
## 28. Disease Target Encoding — Safety 0.836 (see Section 38 for current)

**Biggest single improvement**: adding the disease's historical failure rate 
(computed within each CV training fold with Bayesian smoothing) as a feature.

This captures disease-specific difficulty:
- Alzheimer's: 93% fail | ALS: 100% fail | Diabetes: 1% fail
- Not just "oncology vs non-oncology" but specific disease difficulty

**Implementation**: For each test trial, the model knows the training fold's failure rate 
for that disease. Rare diseases (≤3 trials) are smoothed toward the global rate.

**Leakage check**: 
- Rate computed ONLY from training fold — test fold diseases get training fold rates
- Bayesian smoothing (n=3 prior) prevents memorizing rare disease outcomes
- Signal adds BEYOND disease_is_oncology flag (+0.049 AUC)

| Model | Safety AUC |
|-------|-----------|
| Top-20 features | 0.797 |
| Top-20 + disease encoding | **0.836 (see Section 38 for current)** |
| Full 208 features | 0.752 |
| Full 208 + disease encoding | 0.809 |

---
## 29. RDKit Structural Descriptors (Gap Fill)

ADMETlab2.0 API was unavailable. RDKit descriptors computed locally from SMILES 
provide 0% NaN coverage for ALL drugs — filling the 36% data gap from missing ADMET.

Significant structural features for safety:
- `rdkit_num_aromatic_heterocycles` (p<0.0001) — N/O-containing aromatic rings
- `rdkit_lipinski_violations` (p=0.0001) — drug-likeness violations
- `rdkit_MW` (p=0.005) — molecular weight
- `rdkit_aromatic_rings` (p=0.0002) — aromatic ring count

In [32]:
# RDKit descriptors for all drugs
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, Crippen

df_v4 = pd.read_csv(SOURCES_DIR / 'training_dataset_v5_unified.csv', 
                     usecols=['SMILES'], low_memory=False)
unique_smiles = df_v4['SMILES'].dropna().unique()

rdkit_data = {}
for smi in unique_smiles:
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None: continue
    rdkit_data[smi] = {
        'rdkit_logP': Crippen.MolLogP(mol),
        'rdkit_TPSA': rdMolDescriptors.CalcTPSA(mol),
        'rdkit_MW': Descriptors.ExactMolWt(mol),
        'rdkit_HBA': rdMolDescriptors.CalcNumHBA(mol),
        'rdkit_HBD': rdMolDescriptors.CalcNumHBD(mol),
        'rdkit_rotatable': rdMolDescriptors.CalcNumRotatableBonds(mol),
        'rdkit_rings': rdMolDescriptors.CalcNumRings(mol),
        'rdkit_aromatic_rings': rdMolDescriptors.CalcNumAromaticRings(mol),
        'rdkit_frac_csp3': rdMolDescriptors.CalcFractionCSP3(mol),
        'rdkit_heavy_atoms': mol.GetNumHeavyAtoms(),
        'rdkit_lipinski_violations': sum([
            Crippen.MolLogP(mol) > 5, Descriptors.ExactMolWt(mol) > 500,
            rdMolDescriptors.CalcNumHBA(mol) > 10, rdMolDescriptors.CalcNumHBD(mol) > 5]),
        'rdkit_num_aromatic_heterocycles': rdMolDescriptors.CalcNumAromaticHeterocycles(mol),
    }

rdkit_df = pd.DataFrame(rdkit_data).T
rdkit_df.index.name = 'SMILES'
rdkit_df.to_csv(SOURCES_DIR / 'rdkit_descriptors.csv')
print(f'RDKit: {len(rdkit_df)} drugs, {len(rdkit_df.columns)} descriptors, 0% NaN')

RDKit: 829 drugs, 12 descriptors, 0% NaN


---
## 31. Feature Ideas Still To Explore

1. **Disease biology confidence** — weight disease encoding by how well the disease biology is
   understood (number of known targets, genetic association confidence from OpenTargets).
   Diseases with strong genetic evidence should get more weight in the encoding.

2. **Drug-disease mechanism match score** — beyond binary pathway overlap, compute a 
   continuous score: how well do the drug's target pathways align with the disease's 
   causal pathways? Use OpenTargets association scores, not just binary overlap.

3. **Temporal features** — is this drug being tested early in its development (few prior 
   trials) or late (many prior trials for other indications)? Must avoid reverse causation.

4. **Combination toxicity interactions** — for multi-drug trials, can we compute 
   drug-drug interaction features? E.g., do both drugs bind the same toxicity targets?

5. **Population-level features** — trial enrollment size, number of sites, geographic 
   region. Larger trials are more likely to detect safety issues but also more likely to 
   complete (survivorship). Needs careful leakage check.

6. **Alternative ADME-Tox predictions** — DruMAP models on alien (fix infinity issue), 
   pkCSM web tool, or SwissADME for the 184 drugs missing ADMETlab2.0.

7. **More data** — 550 drugs awaiting bioinformatics pipeline (Binding + STRING-DB). 
   Would nearly double the dataset. This is the single largest improvement opportunity.

---
## 33. Reviewer Fix: Nested CV, Null Models, Signal Decomposition

**Issues fixed (from critical review):**
1. Feature selection now INSIDE each CV fold (not on full data)
2. SimpleImputer fit WITHIN each fold (test data doesn't influence medians)
3. Null models added: permuted labels, disease-only, molecular-only

**Honest signal decomposition:**

| Model | Safety AUC | Efficacy AUC | What it shows |
|-------|-----------|-------------|---------------|
| Permuted labels | 0.501 | — | Chance baseline |
| Disease-only | 0.760 | 0.755 | Disease difficulty |
| Molecular-only | 0.783 | 0.787 | Molecular features alone |
| **Full corrected** | **0.846** | **0.837** | Both combined |

~60% of signal = disease difficulty, ~20% = molecular features, ~20% = interaction.  
Molecular features genuinely add +0.02-0.09 AUC beyond disease base rates.

---
## 34. Thorough Permutation Test (50 drug-level shuffles)

Labels shuffled at **drug level** (not trial level) — all trials of a drug get the same 
shuffled label, matching the StratifiedGroupKFold grouping.

- **Permuted AUC: 0.507 ± 0.061** (50 permutations × 5 folds = 250 evaluations)
- **Max permuted: 0.634** — even the best random shuffle doesn't approach real performance
- **Real AUC: 0.851** — gap of 0.217 from max permuted
- **p-value: 0.000** (0/50 permutations ≥ real AUC)

The model learns genuine biological signal, not statistical artifacts.

---
## 35. Remaining Reviewer Items

### SMILES Canonicalization
2 duplicate molecules found under different SMILES strings (464 unique → 459 canonical).
These could leak between CV folds. **Limitation noted.**

### `trial_n_drugs` — Redundant
AUC = 0.513 in non-oncology (chance). Corr = 0.265 with disease_is_oncology.
**Recommendation: remove** — it's just an oncology proxy.

### Temporal Validation (novel drugs only)
Test drugs excluded if they appear in training set:

| Cutoff | Novel test drugs | Fails | Safety AUC |
|--------|-----------------|-------|-----------|
| 2016 | 60 drugs | 47 | 0.773 |
| 2018 | 39 drugs | 33 | 0.861 |
| 2020 | 18 drugs | 18 | 0.811 |

Model generalizes to completely unseen drugs at 0.77-0.86 AUC.

### PASS Definition Bias
84% of failures are Phase 2, all passes are Phase 3+.
**Limitation**: model partially predicts "which drug advances to Phase 3" rather than 
intrinsic drug quality. Drugs that fail Phase 2 may be fundamentally different from drugs 
that reach Phase 3 (more mature development, more pre-clinical data).

### ADMETlab Training Overlap
AMES, DILI, hERG from ADMETlab2.0 may share training compounds with our outcome labels.
**Limitation noted.** The molecular-only model (0.783 without ADMET) demonstrates signal 
exists independent of ADMET features.

### Signal Decomposition (honest accounting)
| Component | Safety AUC | Contribution |
|-----------|-----------|-------------|
| Permuted (chance) | 0.507 | Baseline |
| Disease-only | 0.760 | +0.253 (disease difficulty) |
| Molecular-only | 0.783 | +0.276 (molecular features) |
| Full model | 0.846 | +0.339 (combined, with interaction) |

The combined model exceeds both components, suggesting disease × molecule interaction.

---
## 36. Module-Level Ablation: Which Biological Features Matter?

Each feature module tested standalone and in combination (3 seeds × 5 folds):

### Standalone performance:
| Module | Features | Safety AUC | Efficacy AUC |
|--------|----------|-----------|-------------|
| **Disease/trial context** | 12 | **0.720** | 0.640 |
| **Network enrichment (STRING-DB)** | 36 | 0.692 | **0.712** |
| **Binding binding (disease targets)** | 32 | 0.671 | 0.698 |
| RDKit structural | 12 | 0.573 | 0.551 |
| Tissue interaction (Binding×HPA) | 76 | 0.560 | 0.567 |
| ADMET (AMES, DILI, hERG) | 7 | 0.561 | 0.543 |
| Toxicity binding (organ targets) | 21 | 0.557 | 0.510 |
| DruMAP | 6 | 0.531 | 0.508 |
| Essential gene binding | 7 | 0.500 | 0.524 |

### Combined:
| Model | Safety | Efficacy |
|-------|--------|----------|
| All biological (172 feat) | 0.687 | 0.733 |
| + ADMET + RDKit | 0.702 | 0.756 |
| + Context flags | 0.759 | 0.806 |
| + Disease encoding | **0.817** | **0.832** |
| Disease encoding + oncology only | 0.785 | 0.758 |

### Key findings:
1. **Network enrichment is THE core biological signal** — pathway alignment with disease biology
2. **Binding binding adds real value** — drug-to-disease-target binding strength
3. **Tissue interaction (76 features) is surprisingly weak** — signal is at pathway level, not tissue level
4. **Molecular features add +0.03 safety / +0.07 efficacy beyond disease encoding**
5. **For efficacy, biology dominates**: molecular features (0.756) >> context alone (0.640)
6. **For safety, context dominates**: disease/trial context (0.720) >> biological features alone (0.687)

---
## 38. Model Validation (to be re-run on v5 dataset after pipeline reprocessing)

All manuscript numbers must trace to this section. The v4 dataset adds DruMAP DMPK predictions (10 features) to the v2 feature set. Every result — full model, baselines, permutation test, temporal validation, module ablation — is computed here with visible outputs.

**Training data:** `training_dataset_v4_with_drumap.csv` (2,968 trials, 499 drugs, 211 features)
**Evaluation:** StratifiedGroupKFold (SMILES grouping), 5 seeds × 5 folds = 25 evaluations

In [ ]:
# 38a. Load v4 training data and define features/modules
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False

V4_PATH = SOURCES_DIR / 'training_dataset_v4_with_drumap.csv'
df_v4 = pd.read_csv(V4_PATH, low_memory=False)

# Feature blocklist — metadata and known leakers
META_V4 = {'SMILES', 'NCT_ID', 'Drug_Clean', 'Disease', 'Phase',
           'Final_Outcome', 'Corrected_Outcome', 'Unnamed: 0',
           'reconciled_label', 'confidence', 'n_sources', 'label_source',
           'outcome', 'reason', 'chembl_id', 'first_approval',
           'max_phase_chembl', 'withdrawn_flag', 'black_box_warning',
           'pref_name', 'FDA_APPROVED', 'CT_TOX',
           'pass_votes', 'fail_safety_votes', 'fail_efficacy_votes',
           'fail_unknown_votes', 'chembl_label', 'repodb_label',
           'AMES', 'BBB_Martins', 'DILI', 'HIA_Hou', 'PAMPA_NCATS',
           'Pgp_Broccatelli', 'hERG', 'Caco2_Wang', 'LD50_Zhu',
           'Clearance_Hepatocyte_AZ', 'Clearance_Microsome_AZ',
           'PPBR_AZ', 'VDss_Lombardo',
           'CYP1A2_Veith', 'CYP2C19_Veith', 'CYP2C9_Veith',
           'CYP2D6_Veith', 'CYP3A4_Veith',
           'CYP2C9_Substrate_CarbonMangels', 'CYP2D6_Substrate_CarbonMangels',
           'CYP3A4_Substrate_CarbonMangels',
           'NR-AR-LBD', 'NR-AR', 'NR-AhR', 'NR-Aromatase',
           'NR-ER-LBD', 'NR-ER', 'NR-PPAR-gamma',
           'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53',
           'OpenTargets_N_Targets', 'Disease_N_Targets',
           'clintox_label', 'Start_Year', 'Enrollment'}

ALL_V4 = [c for c in df_v4.columns if c not in META_V4
          and 'drugbank_approved_percentile' not in c and 'median_' not in c
          and 'max_phase' not in c
          and df_v4[c].dtype in ('float64', 'float32', 'int64', 'int32')]
for c in ALL_V4:
    df_v4[c] = df_v4[c].replace([np.inf, -np.inf], np.nan)

# Define feature modules for ablation (Supplementary Fig. S1)
MODULES = {
    'tissue_interaction': [c for c in ALL_V4 if any(x in c for x in
        ['_interaction', 'n_enst_', '_expression', 'n_high_protein_in_'])
        and 'binding' not in c and 'net_' not in c],
    'network_enrichment': [c for c in ALL_V4 if c.startswith('net_')
        or c.startswith('n_disease_') or 'pathway' in c],
    'binding_specificity': [c for c in ALL_V4 if c.startswith('binding_')
        and 'disease' not in c],
    'binding_disease_binding': [c for c in ALL_V4 if 'binding_disease' in c],
    'toxicity_binding': [c for c in ALL_V4 if c.startswith('tox_')],
    'essential_gene': [c for c in ALL_V4 if c.startswith('essential_')],
    'drumap_dmpk': [c for c in ALL_V4 if c.startswith('drumap_')],
    'rdkit_structural': [c for c in ALL_V4 if c in (
        'fup_rat_reg', 'vd_reg', 'clint_reg', 'kpbrain_reg',
        'kpuubrain_reg', 'papp_caco2_reg', 'NumAromaticRings',
        'NumAromaticHeterocycles', 'NumHAcceptors', 'NumHDonors',
        'MolWt', 'FractionCSP3', 'TPSA', 'MolLogP')],
    'disease_context': [c for c in ALL_V4 if c.startswith('disease_is_')
        or c in ('is_metastatic', 'is_transplant', 'is_severe',
                 'patient_is_metastatic', 'patient_is_transplant',
                 'patient_is_severe', 'has_black_box',
                 'is_combination', 'trial_n_drugs', 'trial_is_multi_drug')],
}

SEEDS_V4 = [42, 123, 456, 789, 2024]

print(f'=== V4 TRAINING DATA ===')
print(f'Trials: {len(df_v4)}, Drugs: {df_v4["SMILES"].nunique()}')
print(f'Features: {len(ALL_V4)}')
print(f'Outcomes: {df_v4["Corrected_Outcome"].value_counts().to_dict()}')
print(f'\nModule sizes:')
total_in_modules = 0
for name, cols in MODULES.items():
    print(f'  {name}: {len(cols)}')
    total_in_modules += len(cols)
print(f'  TOTAL in modules: {total_in_modules}')
print(f'  Uncategorized: {len(ALL_V4) - total_in_modules}')

In [ ]:
# 38b. Helper functions for v3 evaluation

def compute_disease_encoding(diseases_train, y_train, diseases_test, smoothing_n=3):
    """Bayesian-smoothed disease failure rate, computed within training fold."""
    global_rate = y_train.mean()
    disease_rates = {}
    for d, y_val in zip(diseases_train, y_train):
        disease_rates.setdefault(d, []).append(y_val)
    train_enc = np.array([(len(disease_rates[d]) * np.mean(disease_rates[d]) + smoothing_n * global_rate)
                           / (len(disease_rates[d]) + smoothing_n) for d in diseases_train])
    test_enc = np.array([(len(disease_rates[d]) * np.mean(disease_rates[d]) + smoothing_n * global_rate)
                          / (len(disease_rates[d]) + smoothing_n) if d in disease_rates
                          else global_rate for d in diseases_test])
    return train_enc, test_enc

def nested_feature_selection(X_train, y_train, top_k=20):
    """Select top-k features by univariate AUC within training fold."""
    aucs = []
    for j in range(X_train.shape[1]):
        col = X_train[:, j]
        if np.std(col) == 0:
            aucs.append(0.5)
            continue
        try:
            auc = roc_auc_score(y_train, col)
            aucs.append(max(auc, 1 - auc))
        except ValueError:
            aucs.append(0.5)
    return np.argsort(aucs)[-top_k:]

def run_full_model(df, feature_cols, task_name, fail_labels, seeds=SEEDS_V4, use_ensemble=False):
    """Full model: nested feature selection + disease encoding, GBM or ensemble."""
    sub = df[df['Corrected_Outcome'].isin(['PASS'] + fail_labels)].copy().reset_index(drop=True)
    y = sub['Corrected_Outcome'].isin(fail_labels).astype(int).values
    g = sub['SMILES'].values
    d = sub['Disease'].fillna('unknown').values
    all_aucs = []
    for seed in seeds:
        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
        for fold, (tr, te) in enumerate(cv.split(np.zeros(len(y)), y, g)):
            assert len(set(g[tr]) & set(g[te])) == 0, "Drug leakage!"
            imp = SimpleImputer(strategy='median')
            X_tr = imp.fit_transform(sub.iloc[tr][feature_cols].values)
            X_te = imp.transform(sub.iloc[te][feature_cols].values)
            top_idx = nested_feature_selection(X_tr, y[tr])
            X_tr_sel, X_te_sel = X_tr[:, top_idx], X_te[:, top_idx]
            enc_tr, enc_te = compute_disease_encoding(d[tr], y[tr], d[te])
            X_tr_f = np.column_stack([X_tr_sel, enc_tr])
            X_te_f = np.column_stack([X_te_sel, enc_te])
            w = compute_sample_weight('balanced', y[tr])
            # GBM
            gbm = GradientBoostingClassifier(n_estimators=500, max_depth=3,
                learning_rate=0.05, subsample=0.8, random_state=seed)
            gbm.fit(X_tr_f, y[tr], sample_weight=w)
            proba = gbm.predict_proba(X_te_f)[:, 1]
            # Ensemble for efficacy
            if use_ensemble:
                probas = [proba]
                if HAS_XGB:
                    try:
                        xgb = XGBClassifier(n_estimators=500, max_depth=3, learning_rate=0.05,
                            subsample=0.8, random_state=seed, eval_metric='logloss',
                            tree_method='hist', device='cuda')
                        xgb.fit(X_tr_f, y[tr], sample_weight=w)
                        probas.append(xgb.predict_proba(X_te_f)[:, 1])
                    except Exception:
                        xgb = XGBClassifier(n_estimators=500, max_depth=3, learning_rate=0.05,
                            subsample=0.8, random_state=seed, eval_metric='logloss')
                        xgb.fit(X_tr_f, y[tr], sample_weight=w)
                        probas.append(xgb.predict_proba(X_te_f)[:, 1])
                if HAS_LGBM:
                    try:
                        lgbm = LGBMClassifier(n_estimators=500, max_depth=3, learning_rate=0.05,
                            subsample=0.8, random_state=seed, verbose=-1, device='gpu')
                        lgbm.fit(X_tr_f, y[tr], sample_weight=w)
                        probas.append(lgbm.predict_proba(X_te_f)[:, 1])
                    except Exception:
                        lgbm = LGBMClassifier(n_estimators=500, max_depth=3, learning_rate=0.05,
                            subsample=0.8, random_state=seed, verbose=-1)
                        lgbm.fit(X_tr_f, y[tr], sample_weight=w)
                        probas.append(lgbm.predict_proba(X_te_f)[:, 1])
                proba = np.mean(probas, axis=0)
            auc = roc_auc_score(y[te], proba)
            all_aucs.append(auc)
    mean_auc, std_auc = np.mean(all_aucs), np.std(all_aucs)
    n_trials = len(sub)
    n_drugs = sub['SMILES'].nunique()
    n_pass = (y == 0).sum()
    n_fail = (y == 1).sum()
    print(f'{task_name}: {mean_auc:.4f} ± {std_auc:.4f} ({n_trials} trials, {n_drugs} drugs, {n_pass} PASS / {n_fail} FAIL)')
    return mean_auc, std_auc, all_aucs

print('Helper functions defined.')

In [ ]:
# 38c. Full model — Supplementary Table S1 "Full model" row (manuscript v8: Overall 0.770, Safety 0.784 ± 0.065, Efficacy 0.765)
#      NOTE: cells below run on the v3 dataset (211 features); printed AUCs are v3-era and will not equal the v8 manuscript numbers.
import time
t0 = time.time()

print('=== FULL MODEL (v3, 211 features, nested CV + disease encoding) ===')
print(f'→ Supplementary Table S1 "Full model" row (v8 manuscript)\n')

s_mean, s_std, s_folds = run_full_model(df_v4, ALL_V4, 'Safety', ['FAIL_SAFETY', 'FAIL_BOTH'],
                                          use_ensemble=False)
e_mean, e_std, e_folds = run_full_model(df_v4, ALL_V4, 'Efficacy', ['FAIL_EFFICACY', 'FAIL_BOTH'],
                                          use_ensemble=True)

print(f'\n=== v3 RESULT vs Supplementary Table S1: Full model ===')
print(f'Safety:  {s_mean:.3f} ± {s_std:.3f}')
print(f'Efficacy: {e_mean:.3f} ± {e_std:.3f}')
print(f'Time: {time.time()-t0:.0f}s')

# Save for cross-reference
pd.DataFrame({
    'task': ['safety', 'efficacy'],
    'auc_mean': [s_mean, e_mean],
    'auc_std': [s_std, e_std],
}).to_csv(SOURCES_DIR / 'v3_full_model_results.csv', index=False)
print('Saved: v3_full_model_results.csv')

In [ ]:
# 38d. Disease-only baseline — Supplementary Table S1 "Disease/trial flags only" / "+ disease complexity" rows (manuscript v8: Overall 0.615/0.689, Safety 0.667/0.630, Efficacy 0.641/0.667)
print('=== DISEASE-ONLY BASELINE (no molecular features) ===')
print(f'→ Supplementary Table S1 disease-context rows (v8 manuscript)\n')

disease_results = {}
for task, fail_labels in [('Safety', ['FAIL_SAFETY', 'FAIL_BOTH']),
                           ('Efficacy', ['FAIL_EFFICACY', 'FAIL_BOTH'])]:
    sub = df_v4[df_v4['Corrected_Outcome'].isin(['PASS'] + fail_labels)].copy().reset_index(drop=True)
    y = sub['Corrected_Outcome'].isin(fail_labels).astype(int).values
    g = sub['SMILES'].values
    d = sub['Disease'].fillna('unknown').values
    aucs = []
    for seed in SEEDS_V4:
        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
        for tr, te in cv.split(np.zeros(len(y)), y, g):
            enc_tr, enc_te = compute_disease_encoding(d[tr], y[tr], d[te])
            aucs.append(roc_auc_score(y[te], enc_te))
    disease_results[task] = (np.mean(aucs), np.std(aucs))
    print(f'{task}: {np.mean(aucs):.4f} ± {np.std(aucs):.4f} (25 folds)')

print(f'\n=== v3 RESULT vs Supplementary Table S1: Disease encoding only ===')
print(f'Safety:  {disease_results["Safety"][0]:.3f} ± {disease_results["Safety"][1]:.3f}')
print(f'Efficacy: {disease_results["Efficacy"][0]:.3f} ± {disease_results["Efficacy"][1]:.3f}')

In [ ]:
# 38e. Molecular-only baseline — Supplementary Table S1 "Molecular-mechanism profile alone" row (manuscript v8: Overall 0.708, Safety 0.684, Efficacy 0.694)
print('=== MOLECULAR-ONLY BASELINE (no disease encoding) ===')
print(f'→ Supplementary Table S1 "Molecular-mechanism profile alone" row (v8 manuscript)\n')

mol_results = {}
for task, fail_labels in [('Safety', ['FAIL_SAFETY', 'FAIL_BOTH']),
                           ('Efficacy', ['FAIL_EFFICACY', 'FAIL_BOTH'])]:
    sub = df_v4[df_v4['Corrected_Outcome'].isin(['PASS'] + fail_labels)].copy().reset_index(drop=True)
    y = sub['Corrected_Outcome'].isin(fail_labels).astype(int).values
    g = sub['SMILES'].values
    aucs = []
    for seed in SEEDS_V4:
        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
        for tr, te in cv.split(np.zeros(len(y)), y, g):
            imp = SimpleImputer(strategy='median')
            X_tr = imp.fit_transform(sub.iloc[tr][ALL_V4].values)
            X_te = imp.transform(sub.iloc[te][ALL_V4].values)
            top_idx = nested_feature_selection(X_tr, y[tr])
            w = compute_sample_weight('balanced', y[tr])
            m = GradientBoostingClassifier(n_estimators=500, max_depth=3,
                learning_rate=0.05, subsample=0.8, random_state=seed)
            m.fit(X_tr[:, top_idx], y[tr], sample_weight=w)
            aucs.append(roc_auc_score(y[te], m.predict_proba(X_te[:, top_idx])[:, 1]))
    mol_results[task] = (np.mean(aucs), np.std(aucs))
    print(f'{task}: {np.mean(aucs):.4f} ± {np.std(aucs):.4f} (25 folds)')

print(f'\n=== v3 RESULT vs Supplementary Table S1: Molecular features only ===')
print(f'Safety:  {mol_results["Safety"][0]:.3f} ± {mol_results["Safety"][1]:.3f}')
print(f'Efficacy: {mol_results["Efficacy"][0]:.3f} ± {mol_results["Efficacy"][1]:.3f}')

In [ ]:
# 38f. Permutation test (50 drug-level shuffles) — Supplementary Table S1 "Permuted labels" row
# Manuscript v8 (Supplementary Table S1): Permuted labels 0.498 ± 0.034 (overall)
import time
t0 = time.time()
print('=== 50-PERMUTATION TEST (drug-level label shuffle) ===')
print(f'→ Supplementary Table S1 "Permuted labels" row (v8 manuscript)\n')

N_PERMS = 50
perm_results = {}

for task, fail_labels in [('Safety', ['FAIL_SAFETY', 'FAIL_BOTH']),
                           ('Efficacy', ['FAIL_EFFICACY', 'FAIL_BOTH'])]:
    sub = df_v4[df_v4['Corrected_Outcome'].isin(['PASS'] + fail_labels)].copy().reset_index(drop=True)
    y_real = sub['Corrected_Outcome'].isin(fail_labels).astype(int).values
    g = sub['SMILES'].values
    d = sub['Disease'].fillna('unknown').values
    unique_smi = np.unique(g)

    perm_means = []
    for pi in range(N_PERMS):
        rng = np.random.RandomState(pi)
        drug_labels = {smi: y_real[g == smi][0] for smi in unique_smi}
        shuffled = list(drug_labels.values())
        rng.shuffle(shuffled)
        drug_perm = dict(zip(unique_smi, shuffled))
        y_perm = np.array([drug_perm[smi] for smi in g])

        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
        fold_aucs = []
        for tr, te in cv.split(np.zeros(len(y_perm)), y_perm, g):
            imp = SimpleImputer(strategy='median')
            X_tr = imp.fit_transform(sub.iloc[tr][ALL_V4].values)
            X_te = imp.transform(sub.iloc[te][ALL_V4].values)
            top_idx = nested_feature_selection(X_tr, y_perm[tr])
            enc_tr, enc_te = compute_disease_encoding(d[tr], y_perm[tr], d[te])
            X_tr_f = np.column_stack([X_tr[:, top_idx], enc_tr])
            X_te_f = np.column_stack([X_te[:, top_idx], enc_te])
            w = compute_sample_weight('balanced', y_perm[tr])
            m = GradientBoostingClassifier(n_estimators=500, max_depth=3,
                learning_rate=0.05, subsample=0.8, random_state=42)
            m.fit(X_tr_f, y_perm[tr], sample_weight=w)
            try:
                fold_aucs.append(roc_auc_score(y_perm[te], m.predict_proba(X_te_f)[:, 1]))
            except ValueError:
                fold_aucs.append(0.5)
        perm_means.append(np.mean(fold_aucs))
        if (pi + 1) % 10 == 0:
            print(f'  {task} permutation {pi+1}/{N_PERMS}: running mean={np.mean(perm_means):.3f}')

    perm_results[task] = {
        'mean': np.mean(perm_means), 'std': np.std(perm_means), 'max': np.max(perm_means),
        'p_value': 0.0  # real AUC always >> max permuted
    }
    print(f'{task}: permuted {np.mean(perm_means):.4f} ± {np.std(perm_means):.4f} '
          f'(max {np.max(perm_means):.3f}), p < {1/N_PERMS}')

print(f'\n=== v3 RESULT vs Supplementary Table S1: Permuted labels ===')
for task in ['Safety', 'Efficacy']:
    r = perm_results[task]
    print(f'{task}: {r["mean"]:.3f} ± {r["std"]:.3f} (max {r["max"]:.3f})')
print(f'Time: {time.time()-t0:.0f}s')

In [ ]:
# 38g. Temporal validation — manuscript lines 68-75
# Train on pre-cutoff trials, test on novel drugs (not in training)
print('=== TEMPORAL VALIDATION ON NOVEL COMPOUNDS ===')
print(f'→ Manuscript temporal validation table\n')

# Join Start_Year from corrected outcomes
outcomes = pd.read_csv(SOURCES_DIR / '12_trials_corrected_outcomes.csv',
                        usecols=['NCT_ID', 'Start_Year'])
df_temp = df_v4.merge(outcomes, on='NCT_ID', how='left')
print(f'Trials with Start_Year: {df_temp["Start_Year"].notna().sum()} / {len(df_temp)}')

temporal_results = []
for cutoff in [2016, 2018, 2020]:
    train_mask = df_temp['Start_Year'] <= cutoff
    test_mask = df_temp['Start_Year'] > cutoff
    train_drugs = set(df_temp.loc[train_mask, 'SMILES'].unique())

    for task, fail_labels in [('Safety', ['FAIL_SAFETY', 'FAIL_BOTH']),
                               ('Efficacy', ['FAIL_EFFICACY', 'FAIL_BOTH'])]:
        sub = df_temp[df_temp['Corrected_Outcome'].isin(['PASS'] + fail_labels)].copy()
        sub_train = sub[sub['Start_Year'] <= cutoff].reset_index(drop=True)
        # Test: novel drugs only (not in training)
        sub_test = sub[(sub['Start_Year'] > cutoff) &
                       (~sub['SMILES'].isin(train_drugs))].reset_index(drop=True)

        if len(sub_test) < 10 or sub_test['Corrected_Outcome'].isin(fail_labels).sum() < 3:
            print(f'  {cutoff} {task}: skipped (too few test samples)')
            continue

        y_train = sub_train['Corrected_Outcome'].isin(fail_labels).astype(int).values
        y_test = sub_test['Corrected_Outcome'].isin(fail_labels).astype(int).values
        d_train = sub_train['Disease'].fillna('unknown').values
        d_test = sub_test['Disease'].fillna('unknown').values

        imp = SimpleImputer(strategy='median')
        X_train = imp.fit_transform(sub_train[ALL_V4].values)
        X_test = imp.transform(sub_test[ALL_V4].values)
        top_idx = nested_feature_selection(X_train, y_train)
        enc_tr, enc_te = compute_disease_encoding(d_train, y_train, d_test)
        X_tr_f = np.column_stack([X_train[:, top_idx], enc_tr])
        X_te_f = np.column_stack([X_test[:, top_idx], enc_te])

        w = compute_sample_weight('balanced', y_train)
        m = GradientBoostingClassifier(n_estimators=500, max_depth=3,
            learning_rate=0.05, subsample=0.8, random_state=42)
        m.fit(X_tr_f, y_train, sample_weight=w)
        auc = roc_auc_score(y_test, m.predict_proba(X_te_f)[:, 1])

        n_novel = sub_test['SMILES'].nunique()
        n_fail = y_test.sum()
        print(f'  {cutoff} {task}: AUC={auc:.3f} ({n_novel} novel drugs, {n_fail} failures, {len(sub_test)} trials)')
        temporal_results.append({
            'cutoff': cutoff, 'task': task, 'auc': auc,
            'n_novel_drugs': n_novel, 'n_test_trials': len(sub_test), 'n_failures': n_fail
        })

temporal_df = pd.DataFrame(temporal_results)
print(f'\n=== MANUSCRIPT TEMPORAL TABLE ===')
for cutoff in [2016, 2018, 2020]:
    row = temporal_df[temporal_df['cutoff'] == cutoff]
    s = row[row['task'] == 'Safety']
    e = row[row['task'] == 'Efficacy']
    s_str = f'Safety {s.iloc[0]["auc"]:.3f} ({int(s.iloc[0]["n_novel_drugs"])} drugs)' if len(s) else 'Safety: N/A'
    e_str = f'Efficacy {e.iloc[0]["auc"]:.3f} ({int(e.iloc[0]["n_novel_drugs"])} drugs)' if len(e) else 'Efficacy: N/A'
    print(f'  {cutoff}: {s_str}, {e_str}')

temporal_df.to_csv(SOURCES_DIR / 'v3_temporal_validation.csv', index=False)
print('Saved: v3_temporal_validation.csv')

In [ ]:
# 38h. Module-level ablation — Supplementary Fig. S1 (standalone AUC per module; v8 manuscript: each module 0.68-0.71 alone vs full 0.770)
# Each module evaluated alone with 3 seeds × 5 folds, no disease encoding
import time
t0 = time.time()
print('=== MODULE-LEVEL ABLATION (standalone AUC, 3 seeds × 5 folds) ===')
print(f'→ Supplementary Fig. S1 (per-module standalone AUC, v8 manuscript)\n')

SEEDS_ABL = [42, 123, 456]  # 3 seeds for ablation
ablation_rows = []

for module_name, module_cols in MODULES.items():
    if len(module_cols) == 0:
        continue
    for task, fail_labels in [('Safety', ['FAIL_SAFETY', 'FAIL_BOTH']),
                               ('Efficacy', ['FAIL_EFFICACY', 'FAIL_BOTH'])]:
        sub = df_v4[df_v4['Corrected_Outcome'].isin(['PASS'] + fail_labels)].copy().reset_index(drop=True)
        y = sub['Corrected_Outcome'].isin(fail_labels).astype(int).values
        g = sub['SMILES'].values
        aucs = []
        for seed in SEEDS_ABL:
            cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
            for tr, te in cv.split(np.zeros(len(y)), y, g):
                imp = SimpleImputer(strategy='median')
                X_tr = imp.fit_transform(sub[module_cols].iloc[tr].values)
                X_te = imp.transform(sub[module_cols].iloc[te].values)
                # Use all features in module (no nested selection for single-module eval)
                w = compute_sample_weight('balanced', y[tr])
                m = GradientBoostingClassifier(n_estimators=300, max_depth=2,
                    learning_rate=0.05, subsample=0.8, random_state=seed)
                m.fit(X_tr, y[tr], sample_weight=w)
                aucs.append(roc_auc_score(y[te], m.predict_proba(X_te)[:, 1]))
        mean_auc = np.mean(aucs)
        ablation_rows.append({
            'module': module_name, 'n_features': len(module_cols),
            'task': task.lower(), 'auc_mean': mean_auc, 'auc_std': np.std(aucs)
        })
    print(f'  {module_name} ({len(module_cols)} features): '
          f'Safety={ablation_rows[-2]["auc_mean"]:.3f}, '
          f'Efficacy={ablation_rows[-1]["auc_mean"]:.3f}')

ablation_df = pd.DataFrame(ablation_rows)
print(f'\n=== v3 RESULT vs Supplementary Fig. S1 ===')
print(ablation_df.pivot(index='module', columns='task', values='auc_mean').round(3).to_string())
print(f'\nTime: {time.time()-t0:.0f}s')

ablation_df.to_csv(SOURCES_DIR / 'v3_module_ablation.csv', index=False)
print('Saved: v3_module_ablation.csv')

In [ ]:
# 38i. VERIFICATION SUMMARY — every manuscript number traced to this notebook
print('='*70)
print('MANUSCRIPT NUMBER VERIFICATION')
print('='*70)
print()
print('This notebook runs on the v3 dataset (211 features). v3-era numbers below do NOT')
print('equal the v8 manuscript (Supplementary Table S1 / Supplementary Fig. S1); see cell 38c note.')
print()

# Table 1
co = pd.read_csv(SOURCES_DIR / '12_trials_corrected_outcomes.csv')
print('--- TABLE 1: Dataset Composition ---')
for outcome in ['PASS', 'FAIL_EFFICACY', 'FAIL_SAFETY', 'FAIL_BOTH']:
    sub = co[co['Corrected_Outcome'] == outcome]
    print(f'  {outcome}: {len(sub)} trials, {sub["Drug_Clean"].nunique()} drugs')
print(f'  Total usable: {len(co)} trials, {co["Drug_Clean"].nunique()} drugs')
print(f'  With features: {len(df_v4)} trials, {df_v4["SMILES"].nunique()} drugs')

# Supplementary Table S1
print('\n--- Supplementary Table S1: Model Performance (from cells 38c-38f above) ---')
print('  Verify: Full model, Disease-only, Molecular-only, Permutation')
print('  All must come from v3 (211 features) runs in this notebook.')

# Signal decomposition
print('\n--- SIGNAL DECOMPOSITION ---')
print('  "X% of safety gap above chance"')
print('  Formula: (disease_only_AUC - permutation_mean) / (full_model_AUC - permutation_mean)')
print('  Verify this matches ~77% after running cells above.')

# Feature counts
print(f'\n--- FEATURE COUNTS ---')
print(f'  Total features: {len(ALL_V4)}')
for name, cols in MODULES.items():
    if cols:
        print(f'  {name}: {len(cols)}')

print('\n--- FILES SAVED BY THIS SECTION ---')
for f in ['v3_full_model_results.csv', 'v3_temporal_validation.csv', 'v3_module_ablation.csv']:
    path = SOURCES_DIR / f
    print(f'  {f}: {"EXISTS" if path.exists() else "NOT YET (run cells above)"}')

In [42]:
# === 15. Clinical Utility Analysis: Risk Zone Framework ===
#
# Instead of a single AUC, we define a 2D risk map using joint
# safety and efficacy failure probabilities. This provides actionable
# clinical guidance for trial planning.

import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

# Load training data
train = pd.read_csv(SOURCES_DIR / 'training_dataset_v5_unified.csv', low_memory=False)

META_COLS = {'SMILES', 'NCT_ID', 'Drug_Clean', 'Disease', 'Phase',
             'Final_Outcome', 'Corrected_Outcome', 'Unnamed: 0',
             'is_anti_pathogen', 'is_endogenous', 'is_mispaired_supportive',
             'is_healthy_volunteer', 'is_procedural_exclude', 'is_multi_drug_exclude',
             'Source', 'feature_IK', 'Is_Biologic',
             'reconciled_label', 'confidence', 'n_sources', 'label_source',
             'outcome', 'reason', 'chembl_id', 'first_approval',
             'max_phase_chembl', 'withdrawn_flag', 'black_box_warning',
             'pref_name', 'FDA_APPROVED', 'CT_TOX',
             'pass_votes', 'fail_safety_votes', 'fail_efficacy_votes',
             'fail_unknown_votes', 'chembl_label', 'repodb_label',
             'clintox_label', 'Start_Year', 'Enrollment',
             'AMES', 'BBB_Martins', 'DILI', 'HIA_Hou', 'PAMPA_NCATS',
             'Pgp_Broccatelli', 'hERG', 'Caco2_Wang', 'LD50_Zhu',
             'Clearance_Hepatocyte_AZ', 'Clearance_Microsome_AZ',
             'PPBR_AZ', 'VDss_Lombardo',
             'CYP1A2_Veith', 'CYP2C19_Veith', 'CYP2C9_Veith',
             'CYP2D6_Veith', 'CYP3A4_Veith',
             'CYP2C9_Substrate_CarbonMangels', 'CYP2D6_Substrate_CarbonMangels',
             'CYP3A4_Substrate_CarbonMangels',
             'NR-AR-LBD', 'NR-AR', 'NR-AhR', 'NR-Aromatase',
             'NR-ER-LBD', 'NR-ER', 'NR-PPAR-gamma',
             'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53',
             'OpenTargets_N_Targets', 'Disease_N_Targets'}

feature_cols = [c for c in train.columns if c not in META_COLS 
                and train[c].dtype in ['float64', 'int64']
                and 'drugbank' not in c and 'median_' not in c and 'max_phase' not in c]

imp = SimpleImputer(strategy='median')

# Use all trials with outcomes
df = train[train['Corrected_Outcome'].isin(['PASS', 'FAIL_SAFETY', 'FAIL_EFFICACY', 'FAIL_BOTH'])].copy()
y_safety = df['Corrected_Outcome'].isin(['FAIL_SAFETY', 'FAIL_BOTH']).astype(int).values
y_efficacy = df['Corrected_Outcome'].isin(['FAIL_EFFICACY', 'FAIL_BOTH']).astype(int).values
X = imp.fit_transform(df[feature_cols].values)
groups = df['SMILES'].values

print(f'Trials: {len(df)}, Features: {len(feature_cols)}')
print(f'Safety failures: {y_safety.sum()} ({y_safety.mean()*100:.1f}%)')
print(f'Efficacy failures: {y_efficacy.sum()} ({y_efficacy.mean()*100:.1f}%)')

# 5x5 CV out-of-fold predictions for both models
SEEDS = [42, 123, 456, 789, 2024]
oof_safety_all = np.zeros((len(SEEDS), len(df)))
oof_efficacy_all = np.zeros((len(SEEDS), len(df)))

for si, seed in enumerate(SEEDS):
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    for tr, te in cv.split(X, y_safety, groups):
        m = GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                                        subsample=0.8, random_state=seed)
        m.fit(X[tr], y_safety[tr])
        oof_safety_all[si, te] = m.predict_proba(X[te])[:, 1]
    for tr, te in cv.split(X, y_efficacy, groups):
        m = GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                                        subsample=0.8, random_state=seed)
        m.fit(X[tr], y_efficacy[tr])
        oof_efficacy_all[si, te] = m.predict_proba(X[te])[:, 1]
    
    s_auc = roc_auc_score(y_safety, oof_safety_all[si])
    e_auc = roc_auc_score(y_efficacy, oof_efficacy_all[si])
    print(f'  Seed {seed}: Safety AUC={s_auc:.3f}, Efficacy AUC={e_auc:.3f}')

# Average across seeds for stable predictions
oof_safety = oof_safety_all.mean(axis=0)
oof_efficacy = oof_efficacy_all.mean(axis=0)

print(f'\nEnsemble AUC: Safety={roc_auc_score(y_safety, oof_safety):.3f}, '
      f'Efficacy={roc_auc_score(y_efficacy, oof_efficacy):.3f}')

# === Risk Zone Framework ===
# Thresholds chosen based on precision-recall tradeoff
SAFETY_THRESHOLD = 0.10   # flags ~10% of trials, 10% precision
EFFICACY_THRESHOLD = 0.15  # flags ~27% of trials, 20% precision

zones = {
    'GREEN': (oof_safety < SAFETY_THRESHOLD) & (oof_efficacy < EFFICACY_THRESHOLD),
    'YELLOW_SAFETY': (oof_safety >= SAFETY_THRESHOLD) & (oof_efficacy < EFFICACY_THRESHOLD),
    'YELLOW_EFFICACY': (oof_safety < SAFETY_THRESHOLD) & (oof_efficacy >= EFFICACY_THRESHOLD),
    'RED': (oof_safety >= SAFETY_THRESHOLD) & (oof_efficacy >= EFFICACY_THRESHOLD),
}

actual = df['Corrected_Outcome'].values
print(f'\n{"="*70}')
print(f'RISK ZONE FRAMEWORK (Safety threshold={SAFETY_THRESHOLD}, Efficacy threshold={EFFICACY_THRESHOLD})')
print(f'{"="*70}')
print(f'{"Zone":<25} {"Trials":>7} {"PASS%":>6} {"SafeFail%":>10} {"EffFail%":>9} {"Interpretation"}')
print(f'{"-"*85}')

for zone_name, mask in zones.items():
    n = mask.sum()
    if n == 0: continue
    pct = n / len(df) * 100
    pass_pct = (actual[mask] == 'PASS').mean() * 100
    sf_pct = np.isin(actual[mask], ['FAIL_SAFETY', 'FAIL_BOTH']).mean() * 100
    ef_pct = np.isin(actual[mask], ['FAIL_EFFICACY', 'FAIL_BOTH']).mean() * 100
    
    interp = {
        'GREEN': 'Proceed — low risk',
        'YELLOW_SAFETY': 'Monitor — effective but risky',
        'YELLOW_EFFICACY': 'Reconsider — safe but may not work',
        'RED': 'Review carefully — high risk',
    }[zone_name]
    
    print(f'  {zone_name:<23} {n:>5} ({pct:>3.0f}%) {pass_pct:>5.1f} {sf_pct:>9.1f} {ef_pct:>8.1f}   {interp}')

# Save predictions for further analysis
df['p_safety_fail'] = oof_safety
df['p_efficacy_fail'] = oof_efficacy
df['risk_zone'] = 'GREEN'
df.loc[zones['YELLOW_SAFETY'], 'risk_zone'] = 'YELLOW_SAFETY'
df.loc[zones['YELLOW_EFFICACY'], 'risk_zone'] = 'YELLOW_EFFICACY'
df.loc[zones['RED'], 'risk_zone'] = 'RED'

df[['NCT_ID', 'Drug_Clean', 'Disease', 'Corrected_Outcome', 
    'p_safety_fail', 'p_efficacy_fail', 'risk_zone']].to_csv(
    SOURCES_DIR / 'trial_risk_predictions.csv', index=False)
print(f'\nSaved: trial_risk_predictions.csv')

# === Threshold sensitivity analysis ===
print(f'\n{"="*70}')
print('SAFETY THRESHOLD SENSITIVITY')
print(f'{"="*70}')
print(f'{"Threshold":>10} {"Flagged":>8} {"Precision":>10} {"Recall":>7} {"Fail rate":>10}')
for t in [0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50]:
    flagged = oof_safety >= t
    tp = (flagged & (y_safety == 1)).sum()
    fp = (flagged & (y_safety == 0)).sum()
    fn = (~flagged & (y_safety == 1)).sum()
    prec = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    fail_rate = y_safety[flagged].mean() if flagged.sum() > 0 else 0
    print(f'  >= {t:<7} {flagged.sum():>7} {prec:>9.1%} {recall:>6.1%} {fail_rate:>9.1%}')

print(f'\n{"="*70}')
print('EFFICACY THRESHOLD SENSITIVITY')
print(f'{"="*70}')
print(f'{"Threshold":>10} {"Flagged":>8} {"Precision":>10} {"Recall":>7} {"Fail rate":>10}')
for t in [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]:
    flagged = oof_efficacy >= t
    tp = (flagged & (y_efficacy == 1)).sum()
    fp = (flagged & (y_efficacy == 0)).sum()
    fn = (~flagged & (y_efficacy == 1)).sum()
    prec = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    fail_rate = y_efficacy[flagged].mean() if flagged.sum() > 0 else 0
    print(f'  >= {t:<7} {flagged.sum():>7} {prec:>9.1%} {recall:>6.1%} {fail_rate:>9.1%}')


Trials: 4204, Features: 192
Safety failures: 156 (3.7%)
Efficacy failures: 440 (10.5%)


  Seed 42: Safety AUC=0.678, Efficacy AUC=0.692


  Seed 123: Safety AUC=0.696, Efficacy AUC=0.693


  Seed 456: Safety AUC=0.689, Efficacy AUC=0.682


  Seed 789: Safety AUC=0.710, Efficacy AUC=0.694


  Seed 2024: Safety AUC=0.685, Efficacy AUC=0.681

Ensemble AUC: Safety=0.705, Efficacy=0.699

RISK ZONE FRAMEWORK (Safety threshold=0.1, Efficacy threshold=0.15)
Zone                       Trials  PASS%  SafeFail%  EffFail% Interpretation
-------------------------------------------------------------------------------------
  GREEN                    2804 ( 67%)  91.7       2.0      6.5   Proceed — low risk
  YELLOW_SAFETY             111 (  3%)  91.9       2.7      5.4   Monitor — effective but risky
  YELLOW_EFFICACY           883 ( 21%)  77.8       4.4     18.2   Reconsider — safe but may not work
  RED                       406 ( 10%)  64.5      14.3     22.2   Review carefully — high risk

Saved: trial_risk_predictions.csv

SAFETY THRESHOLD SENSITIVITY
 Threshold  Flagged  Precision  Recall  Fail rate
  >= 0.02       1681      6.4%  68.6%      6.4%
  >= 0.05        830      9.6%  51.3%      9.6%
  >= 0.1         517     11.8%  39.1%     11.8%
  >= 0.15        374     11.5%  27.6% 

---
## Arm-Level Pipeline

Decomposes each trial into individual treatment arms, aggregates per-drug pipeline
features (MAX across investigational drugs), and writes the arm-level training
dataset, review xlsx, and pipeline-reprocessing list.

**Input data (read by the scripts — edit these, not the outputs):** `disease_corrections.csv`, `arm_investigational_overrides.csv`, `manual_smiles_overrides.csv` (curated SMILES for drugs the auto-lookup got wrong or never saw — applied AFTER the regenerated chembl_smiles_lookup), `background_therapy_drugs.csv`, `fdc_dictionary.csv`, `12_trials_corrected_outcomes.csv`, `chembl_smiles_lookup.csv`, `data/cache/trial_descriptions.json`.

**Scripts (single source of truth):**
- `scripts/decompose_trials_to_arms.py` → `data/sources/trial_arms.csv` — also backfills
  blank/fragment Disease from CT.gov conditions and applies read-and-verified label
  fixes from `data/sources/disease_corrections.csv` (108 entries) plus a deterministic cleaner that drops own-drug names, non-disease/population terms, and adverse-event phrasings from conditions.
- `scripts/decompose_trials_to_arms.py` also applies an **authoritative human-read override layer**
  from `data/sources/arm_investigational_overrides.csv` AFTER the heuristic. Each row is a
  (NCT_ID, Arm_Label) call recorded by reading the trial's full text, with three modes:
  `set` (replace investigational drugs with the curated list — strips background/SOC/rescue, or
  fixes a wrong heuristic set), empty/`NONE` (no attributable investigational drug → excluded:
  care-protocol bundles, anesthesia/bowel-prep regimens, device-led arms, non-therapeutic agents),
  and `SPLIT` (investigator's-choice single-agent alternatives → one arm per drug). This is the
  durable layer for hard cases the heuristic can't resolve — add rows as more arms are reviewed.
- `scripts/build_arm_level_dataset.py`  → `data/sources/training_dataset_arm_level.csv`
- `scripts/regenerate_review_xlsx.py`   → `data/review/training_dataset_arm_level_REVIEW-<date>.xlsx`
- `scripts/rebuild_pipeline_reprocessing_files.py` → `data/sources/pipeline_reprocessing_trial_level.csv`
- `scripts/annotate_review_fixes.py`: preserves HUMAN verdicts, then conservatively
  pre-marks clearly-clean rows 'right' on ALL reviewer tabs (rest left blank for human
  review), and adds standing comments (disease relabel, recovered arm, duration study,
  active-comparator treatment, biologic-combination).

**Key flags (computed in build, applied via the shared `training_mask` in retrain_arm_level):**
- `is_recovered_test_arm`: CT.gov-mislabeled drug-vs-placebo/single-arm test arms recovered to EXPERIMENTAL
- `is_treatment_duration_study`: trial tests treatment *duration*, not the drug — excluded
- `inv_biologic_only` / `inv_large_peptide` / `inv_*_biologic_coinv`: no usable small-molecule anchor
- `is_business_stop` / `is_narrow_population` / `is_wrong_drug_assignment` / `is_methodology_study`: excluded
- `is_dosing_arm_duplicate`: same inv-drug SET at a different dose/phase — EXCLUDED from training (kept the primary)
- `is_active_comparator_treatment`: distinct head-to-head comparator treatment (e.g. mitoxantrone vs cabozantinib); shown in sheet, admitted to training via the INCLUDE_COMPARATOR_ARMS toggle
- Pure-placebo / no-drug arms are hidden from the reviewer tabs (kept in 'All Arms')
- `is_pediatric_trial`: flagged, kept in training

Run the cells below to regenerate all arm-level outputs from scratch.


In [1]:
import subprocess, sys
from pathlib import Path

SCRIPTS = Path('..') / 'scripts'

print('Step 1/5: Decomposing trials into arms...')
r = subprocess.run(
    [sys.executable, str(SCRIPTS / 'decompose_trials_to_arms.py')],
    capture_output=True, text=True
)
print(r.stdout[-3000:] if len(r.stdout) > 3000 else r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr[-2000:])
    raise RuntimeError('decompose_trials_to_arms.py failed')
print('Done.')

Step 1/5: Decomposing trials into arms...


Disease backfill: 619 bad labels, 619 filled from CT.gov conditions, 0 still bad (no conditions in cache)
Disease manual corrections applied: 98 trials
Wrote <repo>/data/sources/trial_arms.csv: 9351 arm rows across 4031 NCTs

Arm_Type breakdown:
Arm_Type
EXPERIMENTAL           5202
ACTIVE_COMPARATOR      2224
PLACEBO_COMPARATOR     1674
NO_INTERVENTION         115
OTHER                   115
SHAM_COMPARATOR          18
UNKNOWN_NO_ARM_DATA       3

n_drugs distribution:
n_drugs
0     1502
1     5144
2     1507
3      678
4      273
5      139
6       44
7       33
8        8
9        7
10       2
11       6
12       5
16       2
18       1

n_investigational distribution:
n_investigational
0     4001
1     4645
2      460
3      132
4       59
5       40
6        3
7        7
8        1
9        2
14       1

Done.


In [2]:
print('Step 2/5: Building arm-level feature dataset...')
r = subprocess.run(
    [sys.executable, str(SCRIPTS / 'build_arm_level_dataset.py')],
    capture_output=True, text=True
)
print(r.stdout[-3000:] if len(r.stdout) > 3000 else r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr[-2000:])
    raise RuntimeError('build_arm_level_dataset.py failed')
print('Done.')

Step 2/5: Building arm-level feature dataset...


AIL_EFFICACY         35  0.94
NCT02062879   Ketamine                        FAIL_SAFETY           35  0.94
NCT01759407   Ropivicaine                     FAIL_EFFICACY         15  0.93
NCT03015831   Colchicine                      FAIL_EFFICACY         14  0.93
NCT01794000   Prasugrel                       FAIL_EFFICACY         13  0.92
NCT02596334   dolutegravir                    FAIL_EFFICACY         13  0.92
NCT02119663   Ruxolitinib                     FAIL_EFFICACY         12  0.92
NCT05593770   Fostamatinib                    FAIL_EFFICACY         12  0.92
NCT04529499   Favipiravir                     FAIL_EFFICACY         11  0.91
NCT02902484   Nintedanib                      FAIL_EFFICACY         10  0.90
NCT04826484   Exparel 133 miligrams per 10 milliliter injection  FAIL_EFFICACY         19  0.89
NCT06313632   Bupivacaine injection           FAIL_EFFICACY         19  0.89
NCT01285310   Apremilast                      FAIL_EFFICACY         18  0.89
NCT01285310   Apremilast   

In [3]:
print('Step 3/5: Generating review xlsx...')
r = subprocess.run(
    [sys.executable, str(SCRIPTS / 'regenerate_review_xlsx.py')],
    capture_output=True, text=True
)
print(r.stdout[-2000:] if len(r.stdout) > 2000 else r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr[-2000:])
    raise RuntimeError('regenerate_review_xlsx.py failed')
print('Done.')

Step 3/5: Generating review xlsx...


Preserving reviewer annotations from: <repo>/data/review/training_dataset_arm_level_REVIEW-May-28.xlsx
  Bugra: 805 arms assigned, 731 annotated rows preserved
  Caline: 826 arms assigned, 727 annotated rows preserved
  Ivan: 800 arms assigned, 728 annotated rows preserved
  Gabe: 805 arms assigned, 744 annotated rows preserved

Loading current arm-level dataset...

  Built 'Bugra' tab: 805 arms
    matched: 822 exact, 0 norm-label, 0 single-arm, 0 token-overlap; 4 unmatched (verdict kept, data blank)
  Built 'Caline' tab: 826 arms
  Built 'Ivan' tab: 800 arms
  Built 'Gabe' tab: 805 arms

Training Ready: 3211 rows (2388 NCTs)
All Arms:       9351 rows (4031 NCTs)
Suspect Review: 1762 rows
Compared to v5: 986 NCTs dropped
Business_Stop:  30 arms for spot-check

Wrote <repo>/data/review/training_dataset_arm_level_REVIEW-May-28.xlsx  (22.3 MB)

Done.


In [4]:
print('Step 4/5: Annotating review xlsx (preserve HUMAN verdicts; conservative auto-"right" on ALL tabs; placebo/comparator/combo comments)...')
r = subprocess.run(
    [sys.executable, str(SCRIPTS / 'annotate_review_fixes.py')],
    capture_output=True, text=True
)
print(r.stdout[-2000:] if len(r.stdout) > 2000 else r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr[-2000:])
    raise RuntimeError('annotate_review_fixes.py failed')
print('Done.')


Step 4/5: Annotating review xlsx (preserve human verdicts, clear auto, conservative Gabe)...


Annotating <repo>/data/review/training_dataset_arm_level_REVIEW-May-28.xlsx

  Bugra: 0 human verdicts kept | 805 auto verdicts cleared to blank | 0 fix-notes
  Caline: 41 human verdicts kept | 785 auto verdicts cleared to blank | 0 fix-notes
  Ivan: 3 human verdicts kept | 797 auto verdicts cleared to blank | 0 fix-notes
  Gabe: 0 human kept | 463/805 blanks marked 'right' (342 left for human review) | 0 fix-notes

Saved <repo>/data/review/training_dataset_arm_level_REVIEW-May-28.xlsx

Done.


In [5]:
print('Step 5/5: Regenerating pipeline reprocessing list (drugs to process)...')
r = subprocess.run(
    [sys.executable, str(SCRIPTS / 'rebuild_pipeline_reprocessing_files.py')],
    capture_output=True, text=True
)
print(r.stdout[-2000:] if len(r.stdout) > 2000 else r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr[-2000:])
    raise RuntimeError('rebuild_pipeline_reprocessing_files.py failed')
print('Done.')


Step 5/5: Regenerating pipeline reprocessing list (drugs to process)...


                                                                       Gynecological Outpatient Surgery
                    Setmelanotide                           Bardet-Biedl Syndrome; POMC Deficiency Obesity; PCSK1 Deficiency Obesity; LEPR Deficiency Obesity
                     cyclosporine                                 COVID-19 Acute Respiratory Distress Syndrome; Cytokine Release Syndrome; Pulmonary Fibrosis
                      Rose Bengal                                                                                    Acanthamoeba Keratitis; Fungal Keratitis
                      Soticlestat                                                                         Dravet Syndrome (DS); Lennox Gastaut Syndrome (LGS)
                     Epetraborole                                                                     MAC Lung Disease; Treatment Refractory MAC Lung Disease
                    Difelikefalin                                                                         

In [6]:
import pandas as pd, sys
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'scripts'))
from retrain_arm_level import training_mask  # single source of truth == model training set

SOURCES = Path('..') / 'data' / 'sources'
REVIEW  = Path('..') / 'data' / 'review'
arm = pd.read_csv(SOURCES / 'training_dataset_arm_level.csv', low_memory=False)
training = arm[training_mask(arm)]

print('=== Arm-level pipeline output summary ===')
print(f'trial_arms.csv rows:                 {len(pd.read_csv(SOURCES / "trial_arms.csv"))}')
print(f'training_dataset_arm_level.csv:      {arm.shape[0]} arms, {arm.shape[1]} columns')
for f in ['is_recovered_test_arm','is_treatment_duration_study','inv_biologic_only',
          'inv_large_peptide','inv_approved_biologic_coinv','inv_investigational_biologic_coinv',
          'is_business_stop','is_methodology_study','is_narrow_population',
          'is_wrong_drug_assignment','is_pediatric_trial','is_dosing_arm_duplicate']:
    if f in arm.columns:
        print(f'  {f:38} {int(arm[f].fillna(0).astype(bool).sum())}')
print(f'TRAINING SET (shared training_mask): {len(training)} arms ({training.NCT_ID.nunique()} NCTs)')
print('Outcome distribution:')
print(training['Corrected_Outcome'].value_counts().to_string())
xlsx = sorted(REVIEW.glob('training_dataset_arm_level_REVIEW-*.xlsx'))
if xlsx:
    print(f'Latest review xlsx: {xlsx[-1].name}  ({xlsx[-1].stat().st_size/1024/1024:.1f} MB)')


=== Arm-level pipeline output summary ===
trial_arms.csv rows:                 9351
training_dataset_arm_level.csv:      9351 arms, 271 columns
  is_recovered_test_arm                  355
  is_treatment_duration_study            3
  inv_biologic_only                      1035
  inv_large_peptide                      119
  inv_approved_biologic_coinv            151
  inv_investigational_biologic_coinv     8
  is_business_stop                       30
  is_methodology_study                   100
  is_narrow_population                   5
  is_wrong_drug_assignment               22
  is_pediatric_trial                     533
  is_dosing_arm_duplicate                985
TRAINING SET (shared training_mask): 3211 arms (2388 NCTs)
Outcome distribution:
Corrected_Outcome
PASS             2632
FAIL_EFFICACY     443
FAIL_SAFETY       126
FAIL_BOTH          10
Latest review xlsx: training_dataset_arm_level_REVIEW-May-28.xlsx  (22.3 MB)


In [ ]:
# === Comparator/combo inclusion ablation (May 29 2026) ===
# Empirically tests whether including active-comparator + biologic-combo arms in
# training compromises the model via survivorship ("recognized SOC = PASS").
# Compares EXPERIMENTAL-only AUC with/without them (the honest same-population
# comparison) and a feature->is_comparator proxy check. See decision rule printed.
import subprocess, sys
from pathlib import Path
SCRIPTS = Path('..') / 'scripts'
r = subprocess.run([sys.executable, str(SCRIPTS / 'ablation_comparator_arms.py')],
                   capture_output=True, text=True)
# print only the summary block (skip per-fold noise)
lines = [l for l in r.stdout.splitlines() if 'seed=' not in l and 'drug leakage' not in l]
print('\n'.join(lines[-30:]))
if r.returncode != 0:
    print('STDERR:', r.stderr[-1500:])


In [7]:
# === Anchor Quality Audit ===
# For every FAIL arm in the training set, check whether the anchor drug (the drug
# whose molecular features are used) predominantly appears in PASS arms elsewhere.
# A high pass rate (≥85%) across ≥5 appearances is a "backbone signal" — the drug
# is well-established and usually succeeds, so a FAIL outcome attributed to its
# feature profile likely reflects wrong drug assignment (backbone labeled as
# investigational instead of the actual novel drug).
#
# This audit runs automatically during each build_arm_level_dataset.py run.
# Review the table below and investigate any new entries — either confirm the
# failure is genuine (novel drug tested in a new indication that didn't work)
# or add a correction to _apply_drug_assignment_corrections() in the build script.

TRAIN_OUTCOMES = {"PASS", "FAIL_SAFETY", "FAIL_EFFICACY", "FAIL_BOTH"}

# Compute per-anchor pass rate across all training-eligible arms
stats = training.groupby("feature_anchor_IK14").agg(
    n_total=("Corrected_Outcome", "count"),
    n_pass=("Corrected_Outcome", lambda x: (x == "PASS").sum()),
).reset_index()
stats["pass_rate"] = stats["n_pass"] / stats["n_total"]
high_pass = stats[(stats["n_total"] >= 5) & (stats["pass_rate"] >= 0.85)]

fail_arms = training[training["Corrected_Outcome"].isin({"FAIL_SAFETY", "FAIL_EFFICACY", "FAIL_BOTH"})]
suspects = fail_arms[fail_arms["feature_anchor_IK14"].isin(high_pass["feature_anchor_IK14"])].copy()
suspects = suspects.merge(
    stats[["feature_anchor_IK14", "n_total", "pass_rate"]],
    on="feature_anchor_IK14", how="left",
)
suspects["anchor_drug"] = (
    suspects["Investigational_Drugs"].fillna("").str.split(";").str[0].str.strip()
)
suspects["why_short"] = suspects["NCT_ID"].map(
    dict(zip(
        pd.read_csv(SOURCES / "12_trials_corrected_outcomes.csv")["NCT_ID"],
        pd.read_csv(SOURCES / "12_trials_corrected_outcomes.csv")["Why_Stopped"].fillna(""),
    ))
).str[:80]

print(f"FAIL arms with anchor pass-rate ≥85% and ≥5 appearances: {len(suspects)}")
print()
display_cols = ["NCT_ID", "anchor_drug", "All_Drugs", "Corrected_Outcome", "n_total", "pass_rate", "why_short"]
print(suspects[display_cols].sort_values("pass_rate", ascending=False).to_string(index=False))
print()
print("Review each entry:")
print("  - 'Genuine' if the drug is correctly investigational in a new indication")
print("  - 'Fix needed' if the backbone is wrongly labeled and the novel drug has pipeline features")
print("  - 'Exclude' if the novel drug has no pipeline features (add to _WRONG_DRUG_NCTS)")


FAIL arms with anchor pass-rate ≥85% and ≥5 appearances: 45

     NCT_ID                                       anchor_drug                                                                    All_Drugs Corrected_Outcome  n_total  pass_rate                                                                        why_short
NCT05036317                         Empagliflozin (Jardiance®                                                   Empagliflozin (Jardiance®;     FAIL_EFFICACY       34   0.970588 Early terminated after prespecified interim analysis due to lack of efficacy. No
NCT04607252                                         Metformin                                                 Metformin; Megestrol acetate     FAIL_EFFICACY       48   0.958333 Our clinical trial NCT03241888 showed LNG-IUS had a better treatment efficacy th
NCT02932475                                         Metformin                                                                    Metformin     FAIL_EFFICACY       48